In [0]:
# CommerceLake - Incremental Bronze Ingestion
# Cell 1: Pipeline Configuration

CATALOG = "commercelake"

BRONZE_SCHEMA = "bronze"

BASE_PATH = "/Volumes/commercelake/landing/source_data"

PIPELINE_NAME = "commercelake_bronze_incremental_ingestion"

BATCH_PREFIX = "batch_"

SOURCE_CONFIG = {
    "customers": {
        "format": "csv",
        "extension": ".csv",
        "target_table": "raw_customers",
        "primary_key": "customer_id"
    },
    "products": {
        "format": "csv",
        "extension": ".csv",
        "target_table": "raw_products",
        "primary_key": "product_id"
    },
    "orders": {
        "format": "json",
        "extension": ".json",
        "target_table": "raw_orders",
        "primary_key": "order_id"
    },
    "order_items": {
        "format": "parquet",
        "extension": ".parquet",
        "target_table": "raw_order_items",
        "primary_key": "order_item_id"
    },
    "payments": {
        "format": "json",
        "extension": ".json",
        "target_table": "raw_payments",
        "primary_key": "payment_id"
    }
}

# Construct dynamic source and target paths

for entity, config in SOURCE_CONFIG.items():

    config["source_directory"] = (
        f"{BASE_PATH}/{entity}"
    )

    config["target_path"] = (
        f"{CATALOG}.{BRONZE_SCHEMA}."
        f"{config['target_table']}"
    )


# Display configuration

print("=" * 65)
print("COMMERCELAKE INCREMENTAL BRONZE CONFIGURATION")
print("=" * 65)

print(f"Pipeline Name: {PIPELINE_NAME}")
print(f"Catalog: {CATALOG}")
print(f"Target Schema: {BRONZE_SCHEMA}")
print(f"Base Path: {BASE_PATH}")
print(f"Batch Prefix: {BATCH_PREFIX}")
print(f"Configured Entities: {len(SOURCE_CONFIG)}")

for entity, config in SOURCE_CONFIG.items():

    print(f"\nEntity: {entity}")
    print(f"Format: {config['format']}")
    print(f"Primary Key: {config['primary_key']}")
    print(f"Source Directory: {config['source_directory']}")
    print(f"Target Table: {config['target_path']}")

print("\nPipeline Configuration: SUCCESS")

COMMERCELAKE INCREMENTAL BRONZE CONFIGURATION
Pipeline Name: commercelake_bronze_incremental_ingestion
Catalog: commercelake
Target Schema: bronze
Base Path: /Volumes/commercelake/landing/source_data
Batch Prefix: batch_
Configured Entities: 5

Entity: customers
Format: csv
Primary Key: customer_id
Source Directory: /Volumes/commercelake/landing/source_data/customers
Target Table: commercelake.bronze.raw_customers

Entity: products
Format: csv
Primary Key: product_id
Source Directory: /Volumes/commercelake/landing/source_data/products
Target Table: commercelake.bronze.raw_products

Entity: orders
Format: json
Primary Key: order_id
Source Directory: /Volumes/commercelake/landing/source_data/orders
Target Table: commercelake.bronze.raw_orders

Entity: order_items
Format: parquet
Primary Key: order_item_id
Source Directory: /Volumes/commercelake/landing/source_data/order_items
Target Table: commercelake.bronze.raw_order_items

Entity: payments
Format: json
Primary Key: payment_id
Source D

In [0]:
# Cell 2: Dynamic Source File Discovery

import re


def normalize_file_path(file_path: str) -> str:
    """
    Normalize Databricks file paths to match
    existing Bronze source file metadata.
    """

    if file_path.startswith("dbfs:/"):
        return file_path[len("dbfs:"):]

    return file_path


def discover_source_files(
    entity: str,
    config: dict
) -> list:
    """
    Discover source files across all valid batch
    directories for a configured entity.
    """

    discovered_files = []

    source_directory = config["source_directory"]

    directory_entries = dbutils.fs.ls(
        source_directory
    )

    batch_pattern = (
        rf"{re.escape(BATCH_PREFIX)}\d+"
    )

    for entry in directory_entries:

        batch_id = entry.name.rstrip("/")

        # Process only valid batch directories
        if not entry.isDir():
            continue

        if not re.fullmatch(batch_pattern, batch_id):
            continue

        batch_files = dbutils.fs.ls(entry.path)

        for file in batch_files:

            # Ignore nested directories
            if file.isDir():
                continue

            # Ignore hidden and temporary files
            if file.name.startswith((".", "_")):
                continue

            # Process only expected file formats
            if not file.name.lower().endswith(
                config["extension"]
            ):
                continue

            # Reject empty source files
            if file.size <= 0:
                raise ValueError(
                    f"Empty source file: {file.path}"
                )

            discovered_files.append({
                "entity": entity,
                "batch_id": batch_id,
                "file_name": file.name,
                "file_path": normalize_file_path(
                    file.path
                ),
                "file_format": config["format"],
                "file_size_bytes": file.size,
                "target_table": config["target_path"]
            })

    # Ensure deterministic processing order
    discovered_files.sort(
        key=lambda file: (
            int(file["batch_id"][len(BATCH_PREFIX):]),
            file["file_path"]
        )
    )

    return discovered_files


print("=" * 65)
print("DYNAMIC FILE DISCOVERY FUNCTIONS")
print("=" * 65)

print("normalize_file_path(): READY")
print("discover_source_files(): READY")

print("\nFunction Initialization: SUCCESS")

DYNAMIC FILE DISCOVERY FUNCTIONS
normalize_file_path(): READY
discover_source_files(): READY

Function Initialization: SUCCESS


In [0]:
# Cell 3: Execute Dynamic Source File Discovery

from collections import Counter

discovered_source_files = []

print("=" * 65)
print("INCREMENTAL SOURCE FILE DISCOVERY")
print("=" * 65)

for entity, config in SOURCE_CONFIG.items():

    entity_files = discover_source_files(
        entity=entity,
        config=config
    )

    discovered_source_files.extend(entity_files)

    print(f"\nEntity: {entity}")
    print(f"Discovered Files: {len(entity_files)}")

    for file in entity_files:

        print(f"  Batch: {file['batch_id']}")
        print(f"  File: {file['file_name']}")
        print(f"  Size: {file['file_size_bytes']:,} bytes")
        print(f"  Path: {file['file_path']}")
        print()


# Ensure at least one source file was discovered

if not discovered_source_files:
    raise RuntimeError(
        "No source files discovered."
    )


# Create discovery DataFrame

discovery_df = spark.createDataFrame(
    discovered_source_files
)

display(
    discovery_df.orderBy("entity", "batch_id", "file_path")
)


# Calculate discovery metrics

batch_counts = Counter(
    file["batch_id"]
    for file in discovered_source_files
)

entity_counts = Counter(
    file["entity"]
    for file in discovered_source_files
)

total_file_size = sum(
    file["file_size_bytes"]
    for file in discovered_source_files
)


# Display discovery summary

print("\n" + "=" * 65)
print("SOURCE FILE DISCOVERY SUMMARY")
print("=" * 65)

print(f"Configured Entities: {len(SOURCE_CONFIG)}")
print(f"Discovered Entities: {len(entity_counts)}")
print(f"Discovered Batches: {len(batch_counts)}")
print(f"Total Discovered Files: {len(discovered_source_files)}")
print(f"Total Source File Size: {total_file_size:,} bytes")

print("\nFiles by Batch:")

for batch_id, count in sorted(batch_counts.items()):
    print(f"  {batch_id}: {count} files")

print("\nFiles by Entity:")

for entity, count in sorted(entity_counts.items()):
    print(f"  {entity}: {count} files")

print("\nSource File Discovery: SUCCESS")

INCREMENTAL SOURCE FILE DISCOVERY

Entity: customers
Discovered Files: 2
  Batch: batch_001
  File: customers.csv
  Size: 171,044 bytes
  Path: /Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv

  Batch: batch_002
  File: customers.csv
  Size: 34,352 bytes
  Path: /Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv


Entity: products
Discovered Files: 2
  Batch: batch_001
  File: products.csv
  Size: 27,458 bytes
  Path: /Volumes/commercelake/landing/source_data/products/batch_001/products.csv

  Batch: batch_002
  File: products.csv
  Size: 7,535 bytes
  Path: /Volumes/commercelake/landing/source_data/products/batch_002/products.csv


Entity: orders
Discovered Files: 2
  Batch: batch_001
  File: orders.json
  Size: 1,685,858 bytes
  Path: /Volumes/commercelake/landing/source_data/orders/batch_001/orders.json

  Batch: batch_002
  File: orders.json
  Size: 359,292 bytes
  Path: /Volumes/commercelake/landing/source_data/orders/batch_002/

batch_id,entity,file_format,file_name,file_path,file_size_bytes,target_table
batch_001,customers,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,171044,commercelake.bronze.raw_customers
batch_002,customers,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,34352,commercelake.bronze.raw_customers
batch_001,order_items,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,391937,commercelake.bronze.raw_order_items
batch_002,order_items,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,84679,commercelake.bronze.raw_order_items
batch_001,orders,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,1685858,commercelake.bronze.raw_orders
batch_002,orders,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,359292,commercelake.bronze.raw_orders
batch_001,payments,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,2030395,commercelake.bronze.raw_payments
batch_002,payments,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,405932,commercelake.bronze.raw_payments
batch_001,products,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,27458,commercelake.bronze.raw_products
batch_002,products,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,7535,commercelake.bronze.raw_products



SOURCE FILE DISCOVERY SUMMARY
Configured Entities: 5
Discovered Entities: 5
Discovered Batches: 2
Total Discovered Files: 10
Total Source File Size: 5,198,482 bytes

Files by Batch:
  batch_001: 5 files
  batch_002: 5 files

Files by Entity:
  customers: 2 files
  order_items: 2 files
  orders: 2 files
  payments: 2 files
  products: 2 files

Source File Discovery: SUCCESS


In [0]:
# Cell 4: Validate Discovered Source Files

discovery_validation_results = []


def record_discovery_validation(
    check_name: str,
    violation_count: int
):

    status = (
        "PASS" if violation_count == 0
        else "FAIL"
    )

    discovery_validation_results.append({
        "check_name": check_name,
        "violation_count": violation_count,
        "status": status
    })

    print(
        f"[{status}] {check_name}: "
        f"{violation_count} violations"
    )


print("=" * 65)
print("SOURCE DISCOVERY VALIDATION")
print("=" * 65)


# 1. Validate unique source file paths

file_paths = [
    file["file_path"]
    for file in discovered_source_files
]

record_discovery_validation(
    "Duplicate source file paths",
    len(file_paths) - len(set(file_paths))
)


# 2. Validate configured entity coverage

discovered_entities = {
    file["entity"]
    for file in discovered_source_files
}

record_discovery_validation(
    "Missing configured entities",
    len(set(SOURCE_CONFIG) - discovered_entities)
)


# 3. Validate entity names

record_discovery_validation(
    "Unknown source entities",
    sum(
        file["entity"] not in SOURCE_CONFIG
        for file in discovered_source_files
    )
)


# 4. Validate normalized paths

record_discovery_validation(
    "Non-normalized source paths",
    sum(
        file["file_path"].startswith("dbfs:")
        for file in discovered_source_files
    )
)


# 5. Validate batch naming convention

record_discovery_validation(
    "Invalid batch identifiers",
    sum(
        not re.fullmatch(
            rf"{re.escape(BATCH_PREFIX)}\d+",
            file["batch_id"]
        )
        for file in discovered_source_files
    )
)


# 6. Validate source directory structure

record_discovery_validation(
    "Invalid source directory paths",
    sum(
        file["file_path"] != (
            f"{BASE_PATH}/{file['entity']}/"
            f"{file['batch_id']}/{file['file_name']}"
        )
        for file in discovered_source_files
    )
)


# 7. Validate file extensions

record_discovery_validation(
    "Invalid file extensions",
    sum(
        file["entity"] not in SOURCE_CONFIG
        or not file["file_name"].lower().endswith(
            SOURCE_CONFIG[file["entity"]]["extension"]
        )
        for file in discovered_source_files
    )
)


# 8. Validate source file formats

record_discovery_validation(
    "Incorrect source file formats",
    sum(
        file["entity"] not in SOURCE_CONFIG
        or file["file_format"] !=
        SOURCE_CONFIG[file["entity"]]["format"]
        for file in discovered_source_files
    )
)


# 9. Validate non-empty files

record_discovery_validation(
    "Empty source files",
    sum(
        file["file_size_bytes"] <= 0
        for file in discovered_source_files
    )
)


# 10. Validate Bronze target table mapping

record_discovery_validation(
    "Incorrect target table mappings",
    sum(
        file["entity"] not in SOURCE_CONFIG
        or file["target_table"] !=
        SOURCE_CONFIG[file["entity"]]["target_path"]
        for file in discovered_source_files
    )
)


# 11. Validate batch entity coverage

batch_entities = {}

for file in discovered_source_files:

    batch_entities.setdefault(
        file["batch_id"], set()
    ).add(file["entity"])

missing_batch_entities = sum(
    len(set(SOURCE_CONFIG) - entities)
    for entities in batch_entities.values()
)

record_discovery_validation(
    "Missing entities across batches",
    missing_batch_entities
)


# 12. Validate target table existence

missing_target_tables = sum(
    not spark.catalog.tableExists(
        config["target_path"]
    )
    for config in SOURCE_CONFIG.values()
)

record_discovery_validation(
    "Missing Bronze target tables",
    missing_target_tables
)


# Display validation results

display(
    spark.createDataFrame(
        discovery_validation_results
    )
)


# Final validation summary

total_checks = len(discovery_validation_results)

passed_checks = sum(
    result["status"] == "PASS"
    for result in discovery_validation_results
)

failed_checks = total_checks - passed_checks

total_violations = sum(
    result["violation_count"]
    for result in discovery_validation_results
)

print("\n" + "=" * 65)
print("SOURCE DISCOVERY VALIDATION SUMMARY")
print("=" * 65)

print(f"Discovered Files: {len(discovered_source_files)}")
print(f"Total Validation Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")
print(f"Total Violations: {total_violations}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0:
    raise RuntimeError(
        "Source discovery validation failed."
    )

print("\nSource Discovery Validation: SUCCESS")
print("Discovered files are ready for ingestion planning.")

SOURCE DISCOVERY VALIDATION
[PASS] Duplicate source file paths: 0 violations
[PASS] Missing configured entities: 0 violations
[PASS] Unknown source entities: 0 violations
[PASS] Non-normalized source paths: 0 violations
[PASS] Invalid batch identifiers: 0 violations
[PASS] Invalid source directory paths: 0 violations
[PASS] Invalid file extensions: 0 violations
[PASS] Incorrect source file formats: 0 violations
[PASS] Empty source files: 0 violations
[PASS] Incorrect target table mappings: 0 violations
[PASS] Missing entities across batches: 0 violations
[PASS] Missing Bronze target tables: 0 violations


check_name,status,violation_count
Duplicate source file paths,PASS,0
Missing configured entities,PASS,0
Unknown source entities,PASS,0
Non-normalized source paths,PASS,0
Invalid batch identifiers,PASS,0
Invalid source directory paths,PASS,0
Invalid file extensions,PASS,0
Incorrect source file formats,PASS,0
Empty source files,PASS,0
Incorrect target table mappings,PASS,0



SOURCE DISCOVERY VALIDATION SUMMARY
Discovered Files: 10
Total Validation Checks: 12
Passed Checks: 12
Failed Checks: 0
Total Violations: 0
Validation Success Rate: 100.00%

Source Discovery Validation: SUCCESS
Discovered files are ready for ingestion planning.


In [0]:
# Cell 5: Retrieve Previously Ingested Files

from pyspark.sql import functions as F


def get_ingested_file_paths(
    target_table: str
) -> set:
    """
    Retrieve distinct source file paths
    from an existing Bronze Delta table.
    """

    if not spark.catalog.tableExists(target_table):
        raise RuntimeError(
            f"Target table does not exist: {target_table}"
        )

    bronze_df = spark.table(target_table)

    if "_source_file" not in bronze_df.columns:
        raise RuntimeError(
            f"Missing _source_file metadata: {target_table}"
        )

    # Reject invalid source metadata

    invalid_metadata_count = bronze_df.filter(
        F.col("_source_file").isNull()
        | (F.trim(F.col("_source_file")) == "")
    ).count()

    if invalid_metadata_count > 0:
        raise RuntimeError(
            f"Invalid source metadata in {target_table}: "
            f"{invalid_metadata_count} records"
        )

    # Retrieve distinct source paths

    source_files = (
        bronze_df
        .select("_source_file")
        .distinct()
        .collect()
    )

    return {
        normalize_file_path(row["_source_file"])
        for row in source_files
    }


# Retrieve previously ingested files
# for every configured entity

ingested_files_by_entity = {}

ingested_file_registry = []

print("=" * 65)
print("PREVIOUSLY INGESTED FILE DISCOVERY")
print("=" * 65)

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    ingested_paths = get_ingested_file_paths(
        target_table
    )

    ingested_files_by_entity[entity] = ingested_paths

    print(f"\nEntity: {entity}")
    print(f"Target Table: {target_table}")
    print(f"Previously Ingested Files: {len(ingested_paths)}")

    for file_path in sorted(ingested_paths):

        print(f"  {file_path}")

        ingested_file_registry.append({
            "entity": entity,
            "target_table": target_table,
            "file_path": file_path,
            "status": "ALREADY_INGESTED"
        })


# Display previously ingested file registry

if ingested_file_registry:

    display(
        spark.createDataFrame(
            ingested_file_registry
        ).orderBy("entity", "file_path")
    )


# Generate summary

total_ingested_files = sum(
    len(paths)
    for paths in ingested_files_by_entity.values()
)

print("\n" + "=" * 65)
print("PREVIOUSLY INGESTED FILE SUMMARY")
print("=" * 65)

print(f"Configured Entities: {len(SOURCE_CONFIG)}")
print(f"Checked Bronze Tables: {len(ingested_files_by_entity)}")
print(f"Previously Ingested Files: {total_ingested_files}")

print("\nIngested Files by Entity:")

for entity, paths in ingested_files_by_entity.items():

    print(f"  {entity}: {len(paths)} files")

print("\nIngested File Discovery: SUCCESS")

PREVIOUSLY INGESTED FILE DISCOVERY

Entity: customers
Target Table: commercelake.bronze.raw_customers
Previously Ingested Files: 1
  /Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv

Entity: products
Target Table: commercelake.bronze.raw_products
Previously Ingested Files: 1
  /Volumes/commercelake/landing/source_data/products/batch_001/products.csv

Entity: orders
Target Table: commercelake.bronze.raw_orders
Previously Ingested Files: 1
  /Volumes/commercelake/landing/source_data/orders/batch_001/orders.json

Entity: order_items
Target Table: commercelake.bronze.raw_order_items
Previously Ingested Files: 1
  /Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet

Entity: payments
Target Table: commercelake.bronze.raw_payments
Previously Ingested Files: 1
  /Volumes/commercelake/landing/source_data/payments/batch_001/payments.json


entity,file_path,status,target_table
customers,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,ALREADY_INGESTED,commercelake.bronze.raw_customers
order_items,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,ALREADY_INGESTED,commercelake.bronze.raw_order_items
orders,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,ALREADY_INGESTED,commercelake.bronze.raw_orders
payments,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,ALREADY_INGESTED,commercelake.bronze.raw_payments
products,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,ALREADY_INGESTED,commercelake.bronze.raw_products



PREVIOUSLY INGESTED FILE SUMMARY
Configured Entities: 5
Checked Bronze Tables: 5
Previously Ingested Files: 5

Ingested Files by Entity:
  customers: 1 files
  products: 1 files
  orders: 1 files
  order_items: 1 files
  payments: 1 files

Ingested File Discovery: SUCCESS


In [0]:
# Cell 6: Classify Source Files

def classify_source_files(
    discovered_files: list,
    ingested_files: dict
) -> list:
    """
    Classify discovered source files as new
    or previously ingested using Bronze metadata.
    """

    classified_files = []

    for file in discovered_files:

        entity = file["entity"]

        file_path = normalize_file_path(
            file["file_path"]
        )

        if entity not in ingested_files:
            raise RuntimeError(
                f"Missing ingestion registry: {entity}"
            )

        already_ingested = (
            file_path in ingested_files[entity]
        )

        classified_file = {
            **file,
            "already_ingested": already_ingested,
            "ingestion_status": (
                "ALREADY_INGESTED"
                if already_ingested
                else "NEW_FILE"
            ),
            "planned_action": (
                "SKIP"
                if already_ingested
                else "INGEST"
            )
        }

        classified_files.append(classified_file)

    return classified_files


# Execute file classification

classified_source_files = classify_source_files(
    discovered_files=discovered_source_files,
    ingested_files=ingested_files_by_entity
)


# Separate new and previously ingested files

new_source_files = [
    file for file in classified_source_files
    if not file["already_ingested"]
]

skipped_source_files = [
    file for file in classified_source_files
    if file["already_ingested"]
]


# Display classification results

print("=" * 65)
print("SOURCE FILE CLASSIFICATION")
print("=" * 65)

for file in classified_source_files:

    print(
        f"[{file['planned_action']}] "
        f"{file['entity']} | "
        f"{file['batch_id']} | "
        f"{file['file_name']} | "
        f"{file['ingestion_status']}"
    )


display(
    spark.createDataFrame(
        classified_source_files
    ).orderBy("entity", "batch_id", "file_path")
)


# Calculate classification metrics

total_discovered = len(classified_source_files)

total_new = len(new_source_files)

total_skipped = len(skipped_source_files)

new_file_size = sum(
    file["file_size_bytes"]
    for file in new_source_files
)

skipped_file_size = sum(
    file["file_size_bytes"]
    for file in skipped_source_files
)


# Display classification summary

print("\n" + "=" * 65)
print("FILE CLASSIFICATION SUMMARY")
print("=" * 65)

print(f"Total Discovered Files: {total_discovered}")
print(f"New Files: {total_new}")
print(f"Previously Ingested Files: {total_skipped}")
print(f"Files to Process: {total_new}")
print(f"Files to Skip: {total_skipped}")
print(f"New Source Data Size: {new_file_size:,} bytes")
print(f"Skipped Data Size: {skipped_file_size:,} bytes")

print(
    f"File Skip Rate: "
    f"{total_skipped / total_discovered * 100:.2f}%"
)

print("\nFile Classification: SUCCESS")

SOURCE FILE CLASSIFICATION
[SKIP] customers | batch_001 | customers.csv | ALREADY_INGESTED
[INGEST] customers | batch_002 | customers.csv | NEW_FILE
[SKIP] products | batch_001 | products.csv | ALREADY_INGESTED
[INGEST] products | batch_002 | products.csv | NEW_FILE
[SKIP] orders | batch_001 | orders.json | ALREADY_INGESTED
[INGEST] orders | batch_002 | orders.json | NEW_FILE
[SKIP] order_items | batch_001 | order_items.parquet | ALREADY_INGESTED
[INGEST] order_items | batch_002 | order_items.parquet | NEW_FILE
[SKIP] payments | batch_001 | payments.json | ALREADY_INGESTED
[INGEST] payments | batch_002 | payments.json | NEW_FILE


already_ingested,batch_id,entity,file_format,file_name,file_path,file_size_bytes,ingestion_status,planned_action,target_table
true,batch_001,customers,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,171044,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_customers
false,batch_002,customers,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,34352,NEW_FILE,INGEST,commercelake.bronze.raw_customers
true,batch_001,order_items,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,391937,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_order_items
false,batch_002,order_items,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,84679,NEW_FILE,INGEST,commercelake.bronze.raw_order_items
true,batch_001,orders,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,1685858,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_orders
false,batch_002,orders,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,359292,NEW_FILE,INGEST,commercelake.bronze.raw_orders
true,batch_001,payments,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,2030395,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_payments
false,batch_002,payments,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,405932,NEW_FILE,INGEST,commercelake.bronze.raw_payments
true,batch_001,products,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,27458,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_products
false,batch_002,products,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,7535,NEW_FILE,INGEST,commercelake.bronze.raw_products



FILE CLASSIFICATION SUMMARY
Total Discovered Files: 10
New Files: 5
Previously Ingested Files: 5
Files to Process: 5
Files to Skip: 5
New Source Data Size: 891,790 bytes
Skipped Data Size: 4,306,692 bytes
File Skip Rate: 50.00%

File Classification: SUCCESS


In [0]:
# Cell 7: Generate Incremental Ingestion Plan

import uuid

from datetime import datetime, timezone


# Initialize pipeline execution context

PIPELINE_RUN_ID = str(uuid.uuid4())

PIPELINE_START_TIMESTAMP = datetime.now(timezone.utc)

PIPELINE_STATUS = "PLANNING"


# Define entity processing order

ENTITY_PROCESSING_ORDER = {
    "customers": 1,
    "products": 2,
    "orders": 3,
    "order_items": 4,
    "payments": 5
}


def generate_ingestion_plan(
    new_files: list,
    pipeline_run_id: str
) -> list:
    """
    Generate a deterministic execution plan
    containing only unprocessed source files.
    """

    sorted_files = sorted(
        new_files,
        key=lambda file: (
            int(file["batch_id"][len(BATCH_PREFIX):]),
            ENTITY_PROCESSING_ORDER[file["entity"]],
            file["file_path"]
        )
    )

    ingestion_plan = []

    for sequence, file in enumerate(
        sorted_files, start=1
    ):

        ingestion_plan.append({
            **file,
            "execution_sequence": sequence,
            "pipeline_run_id": pipeline_run_id,
            "execution_status": "PENDING"
        })

    return ingestion_plan


# Generate execution plan

ingestion_plan = generate_ingestion_plan(
    new_files=new_source_files,
    pipeline_run_id=PIPELINE_RUN_ID
)


# Display execution context

print("=" * 65)
print("INCREMENTAL INGESTION EXECUTION PLAN")
print("=" * 65)

print(f"Pipeline Name: {PIPELINE_NAME}")
print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(
    f"Execution Started: "
    f"{PIPELINE_START_TIMESTAMP.isoformat()}"
)

print(f"Pipeline Status: {PIPELINE_STATUS}")
print(f"Planned Files: {len(ingestion_plan)}")


# Display planned execution sequence

for file in ingestion_plan:

    print(
        f"\nSequence: {file['execution_sequence']}"
    )

    print(f"Entity: {file['entity']}")
    print(f"Batch: {file['batch_id']}")
    print(f"Source: {file['file_path']}")
    print(f"Target: {file['target_table']}")
    print(f"Format: {file['file_format']}")
    print(f"Status: {file['execution_status']}")


# Display execution plan DataFrame

if ingestion_plan:

    display(
        spark.createDataFrame(
            ingestion_plan
        ).orderBy("execution_sequence")
    )

else:

    print("\nNo new files require ingestion.")


# Calculate execution plan metrics

planned_batches = {
    file["batch_id"]
    for file in ingestion_plan
}

planned_entities = {
    file["entity"]
    for file in ingestion_plan
}

planned_bytes = sum(
    file["file_size_bytes"]
    for file in ingestion_plan
)


# Final execution plan summary

print("\n" + "=" * 65)
print("INGESTION PLAN SUMMARY")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Planned Batches: {len(planned_batches)}")
print(f"Planned Entities: {len(planned_entities)}")
print(f"Files to Ingest: {len(ingestion_plan)}")
print(f"Files to Skip: {len(skipped_source_files)}")
print(f"Planned Data Size: {planned_bytes:,} bytes")

print(
    f"Batches to Process: "
    f"{sorted(planned_batches)}"
)

print("\nIngestion Plan Generation: SUCCESS")

INCREMENTAL INGESTION EXECUTION PLAN
Pipeline Name: commercelake_bronze_incremental_ingestion
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Execution Started: 2026-11-26T13:00:10.379140+00:00
Pipeline Status: PLANNING
Planned Files: 5

Sequence: 1
Entity: customers
Batch: batch_002
Source: /Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv
Target: commercelake.bronze.raw_customers
Format: csv
Status: PENDING

Sequence: 2
Entity: products
Batch: batch_002
Source: /Volumes/commercelake/landing/source_data/products/batch_002/products.csv
Target: commercelake.bronze.raw_products
Format: csv
Status: PENDING

Sequence: 3
Entity: orders
Batch: batch_002
Source: /Volumes/commercelake/landing/source_data/orders/batch_002/orders.json
Target: commercelake.bronze.raw_orders
Format: json
Status: PENDING

Sequence: 4
Entity: order_items
Batch: batch_002
Source: /Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet
Target: commercelake.br

already_ingested,batch_id,entity,execution_sequence,execution_status,file_format,file_name,file_path,file_size_bytes,ingestion_status,pipeline_run_id,planned_action,target_table
false,batch_002,customers,1,PENDING,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,34352,NEW_FILE,5cd36830-4695-40d1-ab22-17d8db1df9f9,INGEST,commercelake.bronze.raw_customers
false,batch_002,products,2,PENDING,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,7535,NEW_FILE,5cd36830-4695-40d1-ab22-17d8db1df9f9,INGEST,commercelake.bronze.raw_products
false,batch_002,orders,3,PENDING,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,359292,NEW_FILE,5cd36830-4695-40d1-ab22-17d8db1df9f9,INGEST,commercelake.bronze.raw_orders
false,batch_002,order_items,4,PENDING,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,84679,NEW_FILE,5cd36830-4695-40d1-ab22-17d8db1df9f9,INGEST,commercelake.bronze.raw_order_items
false,batch_002,payments,5,PENDING,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,405932,NEW_FILE,5cd36830-4695-40d1-ab22-17d8db1df9f9,INGEST,commercelake.bronze.raw_payments



INGESTION PLAN SUMMARY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Planned Batches: 1
Planned Entities: 5
Files to Ingest: 5
Files to Skip: 5
Planned Data Size: 891,790 bytes
Batches to Process: ['batch_002']

Ingestion Plan Generation: SUCCESS


In [0]:
# Cell 8: Validate Incremental Ingestion Plan

plan_validation_results = []


def record_plan_validation(
    check_name: str,
    violation_count: int
):

    status = (
        "PASS" if violation_count == 0
        else "FAIL"
    )

    plan_validation_results.append({
        "check_name": check_name,
        "violation_count": violation_count,
        "status": status
    })

    print(
        f"[{status}] {check_name}: "
        f"{violation_count} violations"
    )


print("=" * 65)
print("INGESTION PLAN VALIDATION")
print("=" * 65)


# 1. Validate unique planned source paths

planned_paths = [
    file["file_path"]
    for file in ingestion_plan
]

record_plan_validation(
    "Duplicate planned file paths",
    len(planned_paths) - len(set(planned_paths))
)


# 2. Ensure previously ingested files are excluded

record_plan_validation(
    "Previously ingested files in execution plan",
    sum(
        file["file_path"]
        in ingested_files_by_entity[file["entity"]]
        for file in ingestion_plan
    )
)


# 3. Validate planned file count

record_plan_validation(
    "Incorrect planned file count",
    abs(len(ingestion_plan) - len(new_source_files))
)


# 4. Validate complete new-file coverage

expected_new_paths = {
    file["file_path"]
    for file in new_source_files
}

record_plan_validation(
    "Missing or unexpected planned files",
    len(set(planned_paths) ^ expected_new_paths)
)


# 5. Validate execution sequence

actual_sequence = [
    file["execution_sequence"]
    for file in ingestion_plan
]

expected_sequence = list(
    range(1, len(ingestion_plan) + 1)
)

record_plan_validation(
    "Invalid execution sequence",
    int(actual_sequence != expected_sequence)
)


# 6. Validate deterministic processing order

expected_order = sorted(
    ingestion_plan,
    key=lambda file: (
        int(file["batch_id"][len(BATCH_PREFIX):]),
        ENTITY_PROCESSING_ORDER[file["entity"]],
        file["file_path"]
    )
)

record_plan_validation(
    "Incorrect entity processing order",
    int(ingestion_plan != expected_order)
)


# 7. Validate pipeline run ID

record_plan_validation(
    "Incorrect pipeline run identifiers",
    sum(
        file["pipeline_run_id"] != PIPELINE_RUN_ID
        for file in ingestion_plan
    )
)


# 8. Validate execution status

record_plan_validation(
    "Invalid execution statuses",
    sum(
        file["execution_status"] != "PENDING"
        for file in ingestion_plan
    )
)


# 9. Validate planned actions

record_plan_validation(
    "Invalid planned ingestion actions",
    sum(
        file["planned_action"] != "INGEST"
        or file["already_ingested"]
        or file["ingestion_status"] != "NEW_FILE"
        for file in ingestion_plan
    )
)


# 10. Validate target table mappings

record_plan_validation(
    "Incorrect target table mappings",
    sum(
        file["target_table"] !=
        SOURCE_CONFIG[file["entity"]]["target_path"]
        for file in ingestion_plan
    )
)


# 11. Validate planned source data size

expected_bytes = sum(
    file["file_size_bytes"]
    for file in new_source_files
)

actual_bytes = sum(
    file["file_size_bytes"]
    for file in ingestion_plan
)

record_plan_validation(
    "Planned source size mismatch",
    int(expected_bytes != actual_bytes)
)


# 12. Validate complete file classification

discovered_paths = {
    file["file_path"]
    for file in discovered_source_files
}

skipped_paths = {
    file["file_path"]
    for file in skipped_source_files
}

classified_paths = set(planned_paths) | skipped_paths

classification_violations = (
    len(discovered_paths ^ classified_paths)
    + len(set(planned_paths) & skipped_paths)
)

record_plan_validation(
    "Incomplete or overlapping file classification",
    classification_violations
)


# Display validation results

display(
    spark.createDataFrame(
        plan_validation_results
    )
)


# Generate final validation summary

total_checks = len(plan_validation_results)

passed_checks = sum(
    result["status"] == "PASS"
    for result in plan_validation_results
)

failed_checks = total_checks - passed_checks

total_violations = sum(
    result["violation_count"]
    for result in plan_validation_results
)

print("\n" + "=" * 65)
print("INGESTION PLAN VALIDATION SUMMARY")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Planned Files: {len(ingestion_plan)}")
print(f"Skipped Files: {len(skipped_source_files)}")
print(f"Total Validation Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")
print(f"Total Violations: {total_violations}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0:

    PIPELINE_STATUS = "FAILED"

    raise RuntimeError(
        "Ingestion plan validation failed. "
        "Do not execute ingestion."
    )


PIPELINE_STATUS = "READY"

print(f"\nPipeline Status: {PIPELINE_STATUS}")
print("Ingestion Plan Validation: SUCCESS")
print("Pipeline is ready for incremental ingestion.")

INGESTION PLAN VALIDATION
[PASS] Duplicate planned file paths: 0 violations
[PASS] Previously ingested files in execution plan: 0 violations
[PASS] Incorrect planned file count: 0 violations
[PASS] Missing or unexpected planned files: 0 violations
[PASS] Invalid execution sequence: 0 violations
[PASS] Incorrect entity processing order: 0 violations
[PASS] Incorrect pipeline run identifiers: 0 violations
[PASS] Invalid execution statuses: 0 violations
[PASS] Invalid planned ingestion actions: 0 violations
[PASS] Incorrect target table mappings: 0 violations
[PASS] Planned source size mismatch: 0 violations
[PASS] Incomplete or overlapping file classification: 0 violations


check_name,status,violation_count
Duplicate planned file paths,PASS,0
Previously ingested files in execution plan,PASS,0
Incorrect planned file count,PASS,0
Missing or unexpected planned files,PASS,0
Invalid execution sequence,PASS,0
Incorrect entity processing order,PASS,0
Incorrect pipeline run identifiers,PASS,0
Invalid execution statuses,PASS,0
Invalid planned ingestion actions,PASS,0
Incorrect target table mappings,PASS,0



INGESTION PLAN VALIDATION SUMMARY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Planned Files: 5
Skipped Files: 5
Total Validation Checks: 12
Passed Checks: 12
Failed Checks: 0
Total Violations: 0
Validation Success Rate: 100.00%

Pipeline Status: READY
Ingestion Plan Validation: SUCCESS
Pipeline is ready for incremental ingestion.


In [0]:
# Cell 9: Reusable Incremental Source Reader

from pyspark.sql import DataFrame


def read_source_data(
    file_path: str,
    file_format: str
) -> DataFrame:
    """
    Read a source file into a Spark DataFrame
    while preserving Bronze source data types.

    Supported formats:
        - CSV
        - JSON Lines
        - Parquet
    """

    file_path = normalize_file_path(file_path)

    file_format = file_format.strip().lower()

    if not file_path.startswith(f"{BASE_PATH}/"):
        raise ValueError(
            f"Invalid source file path: {file_path}"
        )

    # Read CSV source

    if file_format == "csv":

        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .option("mode", "FAILFAST")
            .load(file_path)
        )

    # Read JSON Lines source

    elif file_format == "json":

        df = (
            spark.read
            .format("json")
            .option("multiLine", "false")
            .option("mode", "FAILFAST")
            .load(file_path)
        )

    # Read Parquet source

    elif file_format == "parquet":

        df = (
            spark.read
            .format("parquet")
            .load(file_path)
        )

    else:

        raise ValueError(
            f"Unsupported source format: {file_format}"
        )

    if not df.columns:

        raise RuntimeError(
            f"No columns found in source: {file_path}"
        )

    return df


print("=" * 65)
print("REUSABLE INCREMENTAL SOURCE READER")
print("=" * 65)

print("Supported Formats: CSV, JSON, Parquet")
print("CSV Schema Inference: DISABLED")
print("JSON MultiLine: DISABLED")
print("CSV Parsing Mode: FAILFAST")
print("JSON Parsing Mode: FAILFAST")
print("Path Normalization: ENABLED")
print("Landing Path Validation: ENABLED")

print("\nread_source_data(): READY")

print("\nSource Reader Initialization: SUCCESS")

REUSABLE INCREMENTAL SOURCE READER
Supported Formats: CSV, JSON, Parquet
CSV Schema Inference: DISABLED
JSON MultiLine: DISABLED
CSV Parsing Mode: FAILFAST
JSON Parsing Mode: FAILFAST
Path Normalization: ENABLED
Landing Path Validation: ENABLED

read_source_data(): READY

Source Reader Initialization: SUCCESS


In [0]:
# Cell 10: Schema Compatibility Validation

METADATA_COLUMNS = [
    "_ingestion_timestamp",
    "_source_file",
    "_batch_id",
    "_pipeline_run_id",
    "_record_hash"
]


def validate_schema_compatibility(
    source_df: DataFrame,
    target_table: str
) -> dict:
    """
    Compare incoming source schema against the
    existing Bronze business-column schema.

    Reject missing columns, unexpected columns,
    duplicate column names and incompatible types.
    """

    if not spark.catalog.tableExists(target_table):
        raise RuntimeError(
            f"Target table does not exist: {target_table}"
        )

    target_df = spark.table(target_table)

    # Validate required Bronze metadata columns

    missing_metadata = [
        column for column in METADATA_COLUMNS
        if column not in target_df.columns
    ]

    if missing_metadata:
        raise RuntimeError(
            f"Missing Bronze metadata columns: "
            f"{missing_metadata}"
        )

    # Reject duplicate source column names

    if len(source_df.columns) != len(set(source_df.columns)):
        raise ValueError(
            "Duplicate column names in source schema."
        )

    # Extract incoming source schema

    source_schema = {
        field.name: field.dataType.simpleString()
        for field in source_df.schema.fields
    }

    # Extract target business schema
    # Exclude ingestion metadata columns

    target_schema = {
        field.name: field.dataType.simpleString()
        for field in target_df.schema.fields
        if field.name not in METADATA_COLUMNS
    }

    # Identify missing columns

    missing_columns = sorted(
        set(target_schema) - set(source_schema)
    )

    # Identify unexpected columns

    unexpected_columns = sorted(
        set(source_schema) - set(target_schema)
    )

    # Identify incompatible data types

    common_columns = (
        set(source_schema) & set(target_schema)
    )

    type_mismatches = []

    for column in sorted(common_columns):

        source_type = source_schema[column]

        target_type = target_schema[column]

        if source_type != target_type:

            type_mismatches.append({
                "column": column,
                "source_type": source_type,
                "target_type": target_type
            })

    # Determine compatibility

    schema_compatible = (
        not missing_columns
        and not unexpected_columns
        and not type_mismatches
    )

    validation_result = {
        "target_table": target_table,
        "source_column_count": len(source_schema),
        "target_business_column_count": len(target_schema),
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "type_mismatches": type_mismatches,
        "schema_compatible": schema_compatible,
        "status": (
            "PASS" if schema_compatible else "FAIL"
        )
    }

    if not schema_compatible:

        raise ValueError(
            f"Schema compatibility failed for {target_table}. "
            f"Missing columns: {missing_columns}. "
            f"Unexpected columns: {unexpected_columns}. "
            f"Type mismatches: {type_mismatches}."
        )

    return validation_result


print("=" * 65)
print("SCHEMA COMPATIBILITY VALIDATION")
print("=" * 65)

print(f"Metadata Columns: {len(METADATA_COLUMNS)}")

for column in METADATA_COLUMNS:
    print(f"  {column}")

print("\nMissing Column Detection: ENABLED")
print("Unexpected Column Detection: ENABLED")
print("Duplicate Column Detection: ENABLED")
print("Data Type Compatibility: ENABLED")
print("Target Table Validation: ENABLED")
print("Automatic Schema Evolution: DISABLED")

print("\nvalidate_schema_compatibility(): READY")

print("\nSchema Validator Initialization: SUCCESS")

SCHEMA COMPATIBILITY VALIDATION
Metadata Columns: 5
  _ingestion_timestamp
  _source_file
  _batch_id
  _pipeline_run_id
  _record_hash

Missing Column Detection: ENABLED
Unexpected Column Detection: ENABLED
Duplicate Column Detection: ENABLED
Data Type Compatibility: ENABLED
Target Table Validation: ENABLED
Automatic Schema Evolution: DISABLED

validate_schema_compatibility(): READY

Schema Validator Initialization: SUCCESS


In [0]:
# Cell 11: Reusable Metadata Enrichment

from pyspark.sql import functions as F
from datetime import datetime, timezone


def add_ingestion_metadata(
    df: DataFrame,
    source_path: str,
    batch_id: str,
    pipeline_run_id: str,
    ingestion_timestamp: datetime
) -> DataFrame:
    """
    Enrich incoming source records with
    standardized Bronze ingestion metadata.
    """

    source_path = normalize_file_path(source_path)

    # Prevent metadata column collisions

    conflicting_columns = sorted(
        set(df.columns) & set(METADATA_COLUMNS)
    )

    if conflicting_columns:
        raise ValueError(
            f"Source contains reserved metadata columns: "
            f"{conflicting_columns}"
        )

    # Sort source columns for deterministic hashing

    source_columns = sorted(df.columns)

    # Generate SHA-256 hash using business columns only

    record_hash = F.sha2(
        F.to_json(
            F.struct(
                *[
                    F.col(column)
                    for column in source_columns
                ]
            ),
            options={
                "ignoreNullFields": "false"
            }
        ),
        256
    )

    # Add standardized ingestion metadata

    enriched_df = (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.lit(ingestion_timestamp).cast("timestamp")
        )
        .withColumn(
            "_source_file",
            F.lit(source_path)
        )
        .withColumn(
            "_batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "_pipeline_run_id",
            F.lit(pipeline_run_id)
        )
        .withColumn(
            "_record_hash",
            record_hash
        )
    )

    return enriched_df


print("=" * 65)
print("REUSABLE METADATA ENRICHMENT")
print("=" * 65)

print(f"Metadata Columns: {len(METADATA_COLUMNS)}")

for column in METADATA_COLUMNS:
    print(f"  {column}")

print("\nHash Algorithm: SHA-256")
print("Hash Length: 64 hexadecimal characters")
print("Deterministic Column Ordering: ENABLED")
print("Null Field Preservation: ENABLED")
print("Metadata Collision Detection: ENABLED")
print("Source Path Normalization: ENABLED")
print("Business Column Hashing: ENABLED")

print("\nadd_ingestion_metadata(): READY")

print("\nMetadata Enrichment Initialization: SUCCESS")

REUSABLE METADATA ENRICHMENT
Metadata Columns: 5
  _ingestion_timestamp
  _source_file
  _batch_id
  _pipeline_run_id
  _record_hash

Hash Algorithm: SHA-256
Hash Length: 64 hexadecimal characters
Deterministic Column Ordering: ENABLED
Null Field Preservation: ENABLED
Metadata Collision Detection: ENABLED
Source Path Normalization: ENABLED
Business Column Hashing: ENABLED

add_ingestion_metadata(): READY

Metadata Enrichment Initialization: SUCCESS


In [0]:
# Cell 12: Duplicate Ingestion Guard

def check_file_already_ingested(
    target_table: str,
    source_path: str
) -> bool:
    """
    Check the current Bronze table to determine
    whether a source file has already been ingested.
    """

    source_path = normalize_file_path(source_path)

    if not spark.catalog.tableExists(target_table):
        raise RuntimeError(
            f"Target table does not exist: {target_table}"
        )

    bronze_df = spark.table(target_table)

    if "_source_file" not in bronze_df.columns:
        raise RuntimeError(
            f"Missing _source_file metadata: {target_table}"
        )

    # Normalize existing Bronze source paths

    normalized_source_column = F.regexp_replace(
        F.col("_source_file"),
        r"^dbfs:",
        ""
    )

    # Check whether the source file already exists

    matching_records = (
        bronze_df
        .filter(
            normalized_source_column == source_path
        )
        .limit(1)
        .count()
    )

    return matching_records > 0


def validate_ingestion_guard(
    target_table: str,
    source_path: str
) -> dict:
    """
    Return a file-level ingestion decision based
    on the current state of the Bronze table.
    """

    source_path = normalize_file_path(source_path)

    already_ingested = check_file_already_ingested(
        target_table=target_table,
        source_path=source_path
    )

    return {
        "target_table": target_table,
        "source_path": source_path,
        "already_ingested": already_ingested,
        "ingestion_allowed": not already_ingested,
        "action": (
            "SKIP" if already_ingested else "INGEST"
        ),
        "reason": (
            "Source file already exists in Bronze"
            if already_ingested
            else "Source file not found in Bronze"
        )
    }


print("=" * 65)
print("DUPLICATE INGESTION GUARD")
print("=" * 65)

print("File-Level Duplicate Detection: ENABLED")
print("Current Bronze State Verification: ENABLED")
print("Source Path Normalization: ENABLED")
print("Target Table Validation: ENABLED")
print("Metadata Column Validation: ENABLED")
print("Previously Ingested File Protection: ENABLED")

print("\ncheck_file_already_ingested(): READY")
print("validate_ingestion_guard(): READY")

print("\nDuplicate Ingestion Guard Initialization: SUCCESS")

DUPLICATE INGESTION GUARD
File-Level Duplicate Detection: ENABLED
Current Bronze State Verification: ENABLED
Source Path Normalization: ENABLED
Target Table Validation: ENABLED
Metadata Column Validation: ENABLED
Previously Ingested File Protection: ENABLED

check_file_already_ingested(): READY
validate_ingestion_guard(): READY

Duplicate Ingestion Guard Initialization: SUCCESS


In [0]:
# Cell 13: Reusable Incremental Delta Writer

from time import perf_counter


def write_incremental_delta(
    df: DataFrame,
    target_table: str,
    source_path: str,
    batch_id: str,
    pipeline_run_id: str
) -> dict:
    """
    Safely append incremental records to an existing
    Bronze Delta table and verify the write operation.
    """

    start_time = perf_counter()

    source_path = normalize_file_path(source_path)

    # 1. Validate target table existence

    if not spark.catalog.tableExists(target_table):
        raise RuntimeError(
            f"Target table does not exist: {target_table}"
        )

    target_df = spark.table(target_table)

    # 2. Validate complete enriched schema

    source_schema = {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }

    target_schema = {
        field.name: field.dataType.simpleString()
        for field in target_df.schema.fields
    }

    if source_schema != target_schema:
        raise ValueError(
            f"Enriched schema mismatch for {target_table}"
        )

    # Align DataFrame with target column ordering

    df = df.select(*target_df.columns)

    # 3. Count incoming records

    source_count = df.count()

    if source_count == 0:
        raise ValueError(
            f"Source contains no records: {source_path}"
        )

    # 4. Validate ingestion metadata

    invalid_metadata = (
        df.filter(
            F.col("_source_file").isNull()
            | (F.col("_source_file") != source_path)
            | F.col("_batch_id").isNull()
            | (F.col("_batch_id") != batch_id)
            | F.col("_pipeline_run_id").isNull()
            | (F.col("_pipeline_run_id") != pipeline_run_id)
            | F.col("_ingestion_timestamp").isNull()
            | F.col("_record_hash").isNull()
            | (~F.col("_record_hash").rlike("^[a-f0-9]{64}$"))
        )
        .limit(1)
        .count()
    )

    if invalid_metadata > 0:
        raise ValueError(
            f"Invalid ingestion metadata: {source_path}"
        )

    # 5. Perform final duplicate ingestion check

    guard_result = validate_ingestion_guard(
        target_table=target_table,
        source_path=source_path
    )

    if not guard_result["ingestion_allowed"]:

        return {
            "target_table": target_table,
            "source_path": source_path,
            "batch_id": batch_id,
            "pipeline_run_id": pipeline_run_id,
            "source_records": source_count,
            "inserted_records": 0,
            "duration_seconds": round(
                perf_counter() - start_time, 4
            ),
            "status": "SKIPPED",
            "message": guard_result["reason"]
        }

    # 6. Append records to Delta table

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "false")
        .saveAsTable(target_table)
    )

    # 7. Verify inserted records

    inserted_count = (
        spark.table(target_table)
        .filter(
            (F.col("_source_file") == source_path)
            & (F.col("_batch_id") == batch_id)
            & (F.col("_pipeline_run_id") == pipeline_run_id)
        )
        .count()
    )

    if inserted_count != source_count:
        raise RuntimeError(
            f"Post-write verification failed for "
            f"{target_table}. "
            f"Expected: {source_count}, "
            f"Actual: {inserted_count}. "
            f"Inspect the committed Delta data "
            f"before retrying."
        )

    # 8. Generate execution metrics

    duration_seconds = round(
        perf_counter() - start_time,
        4
    )

    return {
        "target_table": target_table,
        "source_path": source_path,
        "batch_id": batch_id,
        "pipeline_run_id": pipeline_run_id,
        "source_records": source_count,
        "inserted_records": inserted_count,
        "duration_seconds": duration_seconds,
        "status": "SUCCESS",
        "message": "Incremental Delta append verified"
    }


print("=" * 65)
print("REUSABLE INCREMENTAL DELTA WRITER")
print("=" * 65)

print("Target Table Validation: ENABLED")
print("Enriched Schema Validation: ENABLED")
print("Column Order Alignment: ENABLED")
print("Empty Dataset Protection: ENABLED")
print("Ingestion Metadata Validation: ENABLED")
print("Pre-Write Duplicate Guard: ENABLED")
print("Write Format: DELTA")
print("Write Mode: APPEND")
print("Schema Merging: DISABLED")
print("Post-Write Record Verification: ENABLED")
print("Execution Metrics Collection: ENABLED")

print("\nwrite_incremental_delta(): READY")

print("\nDelta Writer Initialization: SUCCESS")

REUSABLE INCREMENTAL DELTA WRITER
Target Table Validation: ENABLED
Enriched Schema Validation: ENABLED
Column Order Alignment: ENABLED
Empty Dataset Protection: ENABLED
Ingestion Metadata Validation: ENABLED
Pre-Write Duplicate Guard: ENABLED
Write Format: DELTA
Write Mode: APPEND
Schema Merging: DISABLED
Post-Write Record Verification: ENABLED
Execution Metrics Collection: ENABLED

write_incremental_delta(): READY

Delta Writer Initialization: SUCCESS


In [0]:
# Cell 14: Reusable File-Level Ingestion Function

def ingest_source_file(file_info: dict) -> dict:
    """
    Process an individual source file through the
    complete incremental Bronze ingestion workflow.
    """

    start_time = perf_counter()

    entity = file_info["entity"]
    batch_id = file_info["batch_id"]
    source_path = normalize_file_path(
        file_info["file_path"]
    )
    file_format = file_info["file_format"]
    target_table = file_info["target_table"]
    pipeline_run_id = file_info["pipeline_run_id"]
    execution_sequence = file_info["execution_sequence"]

    print("\n" + "=" * 65)
    print(f"PROCESSING FILE {execution_sequence}")
    print("=" * 65)

    print(f"Entity: {entity}")
    print(f"Batch: {batch_id}")
    print(f"Source: {source_path}")
    print(f"Target: {target_table}")
    print(f"Format: {file_format}")

    source_count = None
    write_attempted = False

    try:

        # 1. Validate configured target mapping

        if target_table != SOURCE_CONFIG[entity]["target_path"]:
            raise ValueError(
                f"Incorrect target mapping for {entity}"
            )

        if file_format != SOURCE_CONFIG[entity]["format"]:
            raise ValueError(
                f"Incorrect source format for {entity}"
            )

        # 2. Perform early duplicate ingestion check

        guard_result = validate_ingestion_guard(
            target_table=target_table,
            source_path=source_path
        )

        if not guard_result["ingestion_allowed"]:

            print("[SKIPPED] File already ingested.")

            return {
                "entity": entity,
                "execution_sequence": execution_sequence,
                "batch_id": batch_id,
                "source_path": source_path,
                "target_table": target_table,
                "pipeline_run_id": pipeline_run_id,
                "source_records": None,
                "inserted_records": 0,
                "duration_seconds": round(
                    perf_counter() - start_time, 4
                ),
                "status": "SKIPPED",
                "message": guard_result["reason"]
            }

        print("[PASS] Duplicate ingestion guard")

        # 3. Read source data

        source_df = read_source_data(
            file_path=source_path,
            file_format=file_format
        )

        source_count = source_df.count()

        if source_count == 0:
            raise ValueError(
                f"Empty source dataset: {source_path}"
            )

        print(
            f"[PASS] Source records: "
            f"{source_count:,}"
        )

        # 4. Validate source schema

        schema_result = validate_schema_compatibility(
            source_df=source_df,
            target_table=target_table
        )

        print(
            f"[PASS] Schema compatibility: "
            f"{schema_result['source_column_count']} columns"
        )

        # 5. Enrich source records with metadata

        ingestion_timestamp = datetime.now(timezone.utc)

        enriched_df = add_ingestion_metadata(
            df=source_df,
            source_path=source_path,
            batch_id=batch_id,
            pipeline_run_id=pipeline_run_id,
            ingestion_timestamp=ingestion_timestamp
        )

        print("[PASS] Metadata enrichment")

        # 6. Execute incremental Delta write

        write_attempted = True

        write_result = write_incremental_delta(
            df=enriched_df,
            target_table=target_table,
            source_path=source_path,
            batch_id=batch_id,
            pipeline_run_id=pipeline_run_id
        )

        # 7. Generate file-level execution result

        result = {
            **write_result,
            "entity": entity,
            "execution_sequence": execution_sequence,
            "duration_seconds": round(
                perf_counter() - start_time, 4
            )
        }

        print(f"Inserted Records: {result['inserted_records']:,}")
        print(f"Duration: {result['duration_seconds']} seconds")
        print(f"File Ingestion Status: {result['status']}")

        return result

    except Exception as error:

        duration_seconds = round(
            perf_counter() - start_time,
            4
        )

        print(f"[FAILED] {entity}")
        print(f"Error Type: {type(error).__name__}")
        print(f"Error Message: {str(error)}")

        return {
            "entity": entity,
            "execution_sequence": execution_sequence,
            "batch_id": batch_id,
            "source_path": source_path,
            "target_table": target_table,
            "pipeline_run_id": pipeline_run_id,
            "source_records": source_count,
            "inserted_records": (
                None if write_attempted else 0
            ),
            "duration_seconds": duration_seconds,
            "status": "FAILED",
            "message": (
                f"{type(error).__name__}: {str(error)}"
            )
        }


print("=" * 65)
print("REUSABLE FILE-LEVEL INGESTION FUNCTION")
print("=" * 65)

print("Target Mapping Validation: ENABLED")
print("Source Format Validation: ENABLED")
print("Early Duplicate Detection: ENABLED")
print("Reusable Source Reader: INTEGRATED")
print("Schema Compatibility Validator: INTEGRATED")
print("Metadata Enrichment: INTEGRATED")
print("Incremental Delta Writer: INTEGRATED")
print("File-Level Exception Handling: ENABLED")
print("Execution Metrics Collection: ENABLED")

print("\ningest_source_file(): READY")

print("\nFile-Level Ingestion Initialization: SUCCESS")

REUSABLE FILE-LEVEL INGESTION FUNCTION
Target Mapping Validation: ENABLED
Source Format Validation: ENABLED
Early Duplicate Detection: ENABLED
Reusable Source Reader: INTEGRATED
Schema Compatibility Validator: INTEGRATED
Metadata Enrichment: INTEGRATED
Incremental Delta Writer: INTEGRATED
File-Level Exception Handling: ENABLED
Execution Metrics Collection: ENABLED

ingest_source_file(): READY

File-Level Ingestion Initialization: SUCCESS


In [0]:
# Cell 15: Incremental Pipeline Execution Function

def execute_incremental_pipeline(
    ingestion_plan: list,
    pipeline_run_id: str
) -> dict:
    """
    Execute the complete incremental Bronze ingestion
    plan in deterministic sequence.

    Stop processing if any file fails.
    """

    execution_started = datetime.now(timezone.utc)
    start_time = perf_counter()

    execution_results = []

    # 1. Validate pipeline run identifiers

    if any(
        file["pipeline_run_id"] != pipeline_run_id
        for file in ingestion_plan
    ):
        raise ValueError(
            "Pipeline run ID mismatch in ingestion plan."
        )

    # 2. Sort files by execution sequence

    ordered_plan = sorted(
        ingestion_plan,
        key=lambda file: file["execution_sequence"]
    )

    actual_sequence = [
        file["execution_sequence"]
        for file in ordered_plan
    ]

    expected_sequence = list(
        range(1, len(ordered_plan) + 1)
    )

    if actual_sequence != expected_sequence:
        raise ValueError(
            "Invalid pipeline execution sequence."
        )

    total_planned_files = len(ordered_plan)

    print("=" * 65)
    print("INCREMENTAL BRONZE PIPELINE EXECUTION")
    print("=" * 65)

    print(f"Pipeline Name: {PIPELINE_NAME}")
    print(f"Pipeline Run ID: {pipeline_run_id}")
    print(f"Execution Started: {execution_started.isoformat()}")
    print(f"Planned Files: {total_planned_files}")
    print("Pipeline Status: RUNNING")

    # 3. Execute files sequentially

    for file_info in ordered_plan:

        result = ingest_source_file(file_info)

        execution_results.append(result)

        # Stop processing on failure

        if result["status"] == "FAILED":

            print("\n[ERROR] File ingestion failed.")
            print("Stopping remaining pipeline execution.")

            break

    # 4. Calculate execution metrics

    successful_files = sum(
        result["status"] == "SUCCESS"
        for result in execution_results
    )

    skipped_files = sum(
        result["status"] == "SKIPPED"
        for result in execution_results
    )

    failed_files = sum(
        result["status"] == "FAILED"
        for result in execution_results
    )

    processed_files = len(execution_results)

    unprocessed_files = (
        total_planned_files - processed_files
    )

    total_source_records = sum(
        result["source_records"]
        for result in execution_results
        if result["source_records"] is not None
    )

    total_inserted_records = sum(
        result["inserted_records"]
        for result in execution_results
        if result["inserted_records"] is not None
    )

    uncertain_write_files = sum(
        result["inserted_records"] is None
        for result in execution_results
    )

    # 5. Determine final pipeline status

    if failed_files > 0:
        pipeline_status = "FAILED"

    elif total_planned_files == 0:
        pipeline_status = "NO_NEW_FILES"

    else:
        pipeline_status = "SUCCESS"

    # 6. Calculate execution duration

    execution_completed = datetime.now(timezone.utc)

    duration_seconds = round(
        perf_counter() - start_time,
        4
    )

    # 7. Generate consolidated pipeline summary

    pipeline_summary = {
        "pipeline_name": PIPELINE_NAME,
        "pipeline_run_id": pipeline_run_id,
        "execution_started": execution_started.isoformat(),
        "execution_completed": execution_completed.isoformat(),
        "planned_files": total_planned_files,
        "processed_files": processed_files,
        "successful_files": successful_files,
        "skipped_files": skipped_files,
        "failed_files": failed_files,
        "unprocessed_files": unprocessed_files,
        "uncertain_write_files": uncertain_write_files,
        "total_source_records": total_source_records,
        "total_inserted_records": total_inserted_records,
        "duration_seconds": duration_seconds,
        "pipeline_status": pipeline_status
    }

    # 8. Print execution summary

    print("\n" + "=" * 65)
    print("PIPELINE EXECUTION SUMMARY")
    print("=" * 65)

    print(f"Pipeline Run ID: {pipeline_run_id}")
    print(f"Planned Files: {total_planned_files}")
    print(f"Processed Files: {processed_files}")
    print(f"Successful Files: {successful_files}")
    print(f"Skipped Files: {skipped_files}")
    print(f"Failed Files: {failed_files}")
    print(f"Unprocessed Files: {unprocessed_files}")
    print(f"Uncertain Write Files: {uncertain_write_files}")
    print(f"Known Source Records: {total_source_records:,}")
    print(f"Verified Inserted Records: {total_inserted_records:,}")
    print(f"Execution Duration: {duration_seconds} seconds")
    print(f"Pipeline Status: {pipeline_status}")

    if uncertain_write_files > 0:
        print(
            "WARNING: One or more write outcomes "
            "require Delta table inspection."
        )

    return {
        "pipeline_summary": pipeline_summary,
        "file_results": execution_results
    }


print("=" * 65)
print("INCREMENTAL PIPELINE EXECUTION ENGINE")
print("=" * 65)

print("Sequential File Processing: ENABLED")
print("Execution Sequence Validation: ENABLED")
print("Pipeline Run ID Validation: ENABLED")
print("Fail-Fast Execution: ENABLED")
print("File-Level Result Collection: ENABLED")
print("Consolidated Metrics: ENABLED")
print("Empty Ingestion Plan Support: ENABLED")
print("Unprocessed File Tracking: ENABLED")
print("Uncertain Write Tracking: ENABLED")
print("Execution Duration Tracking: ENABLED")
print("Pipeline Status Management: ENABLED")

print("\nexecute_incremental_pipeline(): READY")

print("\nPipeline Execution Engine Initialization: SUCCESS")

INCREMENTAL PIPELINE EXECUTION ENGINE
Sequential File Processing: ENABLED
Execution Sequence Validation: ENABLED
Pipeline Run ID Validation: ENABLED
Fail-Fast Execution: ENABLED
File-Level Result Collection: ENABLED
Consolidated Metrics: ENABLED
Empty Ingestion Plan Support: ENABLED
Unprocessed File Tracking: ENABLED
Uncertain Write Tracking: ENABLED
Execution Duration Tracking: ENABLED
Pipeline Status Management: ENABLED

execute_incremental_pipeline(): READY

Pipeline Execution Engine Initialization: SUCCESS


In [0]:
# Cell 16: Read-Only Pre-Ingestion Testing

EXPECTED_INCREMENTAL_COUNTS = {
    "customers": 200,
    "products": 50,
    "orders": 1000,
    "order_items": 1969,
    "payments": 1000
}

EXPECTED_HISTORICAL_COUNTS = {
    "customers": 1000,
    "products": 200,
    "orders": 5000,
    "order_items": 10055,
    "payments": 5000
}

pre_ingestion_results = []


def record_pre_ingestion_test(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    pre_ingestion_results.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("=" * 65)
print("READ-ONLY PRE-INGESTION TESTING")
print("=" * 65)

print("Execution Mode: READ ONLY")
print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Planned Files: {len(ingestion_plan)}")


# Capture Bronze state before testing

bronze_state_before = {}

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    bronze_state_before[entity] = {
        "records": spark.table(target_table).count(),
        "version": (
            spark.sql(
                f"DESCRIBE HISTORY {target_table}"
            )
            .agg(F.max("version").alias("version"))
            .first()["version"]
        )
    }


# Test each incremental source file

for file_info in ingestion_plan:

    entity = file_info["entity"]
    source_path = file_info["file_path"]
    target_table = file_info["target_table"]
    batch_id = file_info["batch_id"]

    print("\n" + "-" * 65)
    print(f"TESTING ENTITY: {entity.upper()}")
    print("-" * 65)

    try:

        # 1. Test source reader

        source_df = read_source_data(
            file_path=source_path,
            file_format=file_info["file_format"]
        )

        source_count = source_df.count()

        record_pre_ingestion_test(
            entity,
            "Source file readable",
            source_count > 0
        )

        # 2. Validate source record count

        record_pre_ingestion_test(
            entity,
            "Expected incremental record count",
            source_count ==
            EXPECTED_INCREMENTAL_COUNTS[entity]
        )

        print(f"Source Records: {source_count:,}")

        # 3. Test schema compatibility

        schema_result = validate_schema_compatibility(
            source_df=source_df,
            target_table=target_table
        )

        record_pre_ingestion_test(
            entity,
            "Source schema compatibility",
            schema_result["schema_compatible"]
        )

        # 4. Test metadata enrichment

        test_timestamp = datetime.now(timezone.utc)

        enriched_df = add_ingestion_metadata(
            df=source_df,
            source_path=source_path,
            batch_id=batch_id,
            pipeline_run_id=PIPELINE_RUN_ID,
            ingestion_timestamp=test_timestamp
        )

        record_pre_ingestion_test(
            entity,
            "All metadata columns present",
            set(METADATA_COLUMNS).issubset(
                enriched_df.columns
            )
        )

        # 5. Verify enrichment preserves records

        record_pre_ingestion_test(
            entity,
            "Record count preserved after enrichment",
            enriched_df.count() == source_count
        )

        # 6. Verify enriched schema matches target

        enriched_schema = {
            field.name: field.dataType.simpleString()
            for field in enriched_df.schema.fields
        }

        target_schema = {
            field.name: field.dataType.simpleString()
            for field in spark.table(
                target_table
            ).schema.fields
        }

        record_pre_ingestion_test(
            entity,
            "Complete enriched schema compatibility",
            enriched_schema == target_schema
        )

        # 7. Validate metadata values

        invalid_metadata = enriched_df.filter(
            F.col("_source_file").isNull()
            | (F.col("_source_file") != source_path)
            | F.col("_batch_id").isNull()
            | (F.col("_batch_id") != batch_id)
            | F.col("_pipeline_run_id").isNull()
            | (
                F.col("_pipeline_run_id")
                != PIPELINE_RUN_ID
            )
            | F.col("_ingestion_timestamp").isNull()
            | (
                F.col("_ingestion_timestamp")
                != F.lit(test_timestamp).cast("timestamp")
            )
        ).count()

        record_pre_ingestion_test(
            entity,
            "Ingestion metadata values valid",
            invalid_metadata == 0
        )

        # 8. Validate SHA-256 hash format

        invalid_hashes = enriched_df.filter(
            F.col("_record_hash").isNull()
            | (
                ~F.col("_record_hash")
                .rlike("^[a-f0-9]{64}$")
            )
        ).count()

        record_pre_ingestion_test(
            entity,
            "SHA-256 record hash format",
            invalid_hashes == 0
        )

        # 9. Verify record hash consistency

        source_columns = sorted(source_df.columns)

        expected_hash = F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(column)
                        for column in source_columns
                    ]
                ),
                options={
                    "ignoreNullFields": "false"
                }
            ),
            256
        )

        hash_mismatches = enriched_df.filter(
            ~F.col("_record_hash").eqNullSafe(
                expected_hash
            )
        ).count()

        record_pre_ingestion_test(
            entity,
            "Record hash consistency",
            hash_mismatches == 0
        )

        # 10. Verify new file is allowed

        new_file_guard = validate_ingestion_guard(
            target_table=target_table,
            source_path=source_path
        )

        record_pre_ingestion_test(
            entity,
            "New incremental file allowed",
            new_file_guard["ingestion_allowed"]
        )

        # 11. Verify historical files are blocked

        historical_paths = [
            file["file_path"]
            for file in skipped_source_files
            if file["entity"] == entity
            and file["batch_id"] == "batch_001"
        ]

        historical_files_blocked = (
            bool(historical_paths)
            and all(
                not validate_ingestion_guard(
                    target_table=target_table,
                    source_path=path
                )["ingestion_allowed"]
                for path in historical_paths
            )
        )

        record_pre_ingestion_test(
            entity,
            "Historical file duplicate protection",
            historical_files_blocked
        )

        # 12. Validate existing Bronze record count

        record_pre_ingestion_test(
            entity,
            "Historical Bronze record count",
            bronze_state_before[entity]["records"]
            == EXPECTED_HISTORICAL_COUNTS[entity]
        )

    except Exception as error:

        record_pre_ingestion_test(
            entity,
            f"Execution exception: {type(error).__name__}",
            False
        )

        print(f"Error: {str(error)}")


# Verify Bronze tables remain unchanged

print("\n" + "=" * 65)
print("BRONZE STATE VERIFICATION")
print("=" * 65)

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    current_count = spark.table(target_table).count()

    current_version = (
        spark.sql(
            f"DESCRIBE HISTORY {target_table}"
        )
        .agg(F.max("version").alias("version"))
        .first()["version"]
    )

    record_pre_ingestion_test(
        entity,
        "Bronze record count unchanged",
        current_count ==
        bronze_state_before[entity]["records"]
    )

    record_pre_ingestion_test(
        entity,
        "Delta table version unchanged",
        current_version ==
        bronze_state_before[entity]["version"]
    )


# Validate complete batch coverage

record_pre_ingestion_test(
    "pipeline",
    "Complete batch_002 execution plan",
    len(ingestion_plan) == 5
    and {
        file["entity"]
        for file in ingestion_plan
    } == set(EXPECTED_INCREMENTAL_COUNTS)
    and all(
        file["batch_id"] == "batch_002"
        for file in ingestion_plan
    )
)


# Display detailed validation results

display(
    spark.createDataFrame(
        pre_ingestion_results
    )
)


# Generate validation summary

total_tests = len(pre_ingestion_results)

passed_tests = sum(
    result["status"] == "PASS"
    for result in pre_ingestion_results
)

failed_tests = total_tests - passed_tests

expected_tests = (
    len(EXPECTED_INCREMENTAL_COUNTS) * 14 + 1
)

print("\n" + "=" * 65)
print("PRE-INGESTION TEST SUMMARY")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print("Execution Mode: READ ONLY")
print(f"Expected Tests: {expected_tests}")
print(f"Executed Tests: {total_tests}")
print(f"Passed Tests: {passed_tests}")
print(f"Failed Tests: {failed_tests}")

print(
    f"Test Success Rate: "
    f"{passed_tests / total_tests * 100:.2f}%"
)

if failed_tests > 0 or total_tests != expected_tests:

    PIPELINE_STATUS = "FAILED"

    raise RuntimeError(
        "Pre-ingestion testing failed or incomplete. "
        "Do not execute incremental ingestion."
    )


PIPELINE_STATUS = "READY"

print("\nBronze Tables Modified: NO")
print(f"Pipeline Status: {PIPELINE_STATUS}")
print("Pre-Ingestion Testing: SUCCESS")
print("Pipeline is ready for actual Delta ingestion.")

READ-ONLY PRE-INGESTION TESTING
Execution Mode: READ ONLY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Planned Files: 5

-----------------------------------------------------------------
TESTING ENTITY: CUSTOMERS
-----------------------------------------------------------------
[PASS] Source file readable
[PASS] Expected incremental record count
Source Records: 200
[PASS] Source schema compatibility
[PASS] All metadata columns present
[PASS] Record count preserved after enrichment
[PASS] Complete enriched schema compatibility
[PASS] Ingestion metadata values valid
[PASS] SHA-256 record hash format
[PASS] Record hash consistency
[PASS] New incremental file allowed
[PASS] Historical file duplicate protection
[PASS] Historical Bronze record count

-----------------------------------------------------------------
TESTING ENTITY: PRODUCTS
-----------------------------------------------------------------
[PASS] Source file readable
[PASS] Expected incremental record count
Source Rec

check_name,entity,status
Source file readable,customers,PASS
Expected incremental record count,customers,PASS
Source schema compatibility,customers,PASS
All metadata columns present,customers,PASS
Record count preserved after enrichment,customers,PASS
Complete enriched schema compatibility,customers,PASS
Ingestion metadata values valid,customers,PASS
SHA-256 record hash format,customers,PASS
Record hash consistency,customers,PASS
New incremental file allowed,customers,PASS



PRE-INGESTION TEST SUMMARY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Execution Mode: READ ONLY
Expected Tests: 71
Executed Tests: 71
Passed Tests: 71
Failed Tests: 0
Test Success Rate: 100.00%

Bronze Tables Modified: NO
Pipeline Status: READY
Pre-Ingestion Testing: SUCCESS
Pipeline is ready for actual Delta ingestion.


In [0]:
# Cell 17: Execute Incremental Bronze Ingestion

print("=" * 65)
print("INCREMENTAL INGESTION EXECUTION AUTHORIZATION")
print("=" * 65)


# 1. Verify pipeline readiness

if PIPELINE_STATUS != "READY":
    raise RuntimeError(
        f"Pipeline is not ready. "
        f"Current status: {PIPELINE_STATUS}"
    )


# 2. Verify pre-ingestion test results

if (
    len(pre_ingestion_results) != 71
    or any(
        result["status"] != "PASS"
        for result in pre_ingestion_results
    )
):
    raise RuntimeError(
        "Pre-ingestion validation is incomplete or failed."
    )


# 3. Verify execution plan

if len(ingestion_plan) != 5:
    raise RuntimeError(
        "Unexpected number of planned files."
    )

if any(
    file["batch_id"] != "batch_002"
    or file["pipeline_run_id"] != PIPELINE_RUN_ID
    for file in ingestion_plan
):
    raise RuntimeError(
        "Invalid batch or pipeline run identifier."
    )


# 4. Confirm Bronze state has not changed

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    current_count = spark.table(target_table).count()

    current_version = (
        spark.sql(
            f"DESCRIBE HISTORY {target_table}"
        )
        .agg(F.max("version").alias("version"))
        .first()["version"]
    )

    if (
        current_count != bronze_state_before[entity]["records"]
        or current_version != bronze_state_before[entity]["version"]
    ):
        raise RuntimeError(
            f"Bronze state changed for {target_table}. "
            f"Inspect table before proceeding."
        )

    print(f"[PASS] Bronze state verified: {entity}")


# 5. Print execution authorization

EXPECTED_NEW_RECORDS = sum(
    EXPECTED_INCREMENTAL_COUNTS.values()
)

print("\nPre-Ingestion Tests: 71/71 PASSED")
print("Target Batch: batch_002")
print(f"Files Authorized: {len(ingestion_plan)}")
print(f"Expected New Records: {EXPECTED_NEW_RECORDS:,}")
print("Write Mode: APPEND")
print("Execution Authorization: APPROVED")


# 6. Execute actual incremental pipeline

PIPELINE_STATUS = "RUNNING"

try:

    incremental_execution_result = (
        execute_incremental_pipeline(
            ingestion_plan=ingestion_plan,
            pipeline_run_id=PIPELINE_RUN_ID
        )
    )

except Exception:

    PIPELINE_STATUS = "FAILED"

    raise


# 7. Extract execution results

incremental_pipeline_summary = (
    incremental_execution_result["pipeline_summary"]
)

incremental_file_results = (
    incremental_execution_result["file_results"]
)

PIPELINE_STATUS = (
    incremental_pipeline_summary["pipeline_status"]
)


# 8. Display final execution confirmation

print("\n" + "=" * 65)
print("INCREMENTAL INGESTION CONFIRMATION")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline Status: {PIPELINE_STATUS}")

print(
    f"Expected New Records: "
    f"{EXPECTED_NEW_RECORDS:,}"
)

print(
    f"Verified Inserted Records: "
    f"{incremental_pipeline_summary['total_inserted_records']:,}"
)


# 9. Verify overall execution success

if (
    PIPELINE_STATUS != "SUCCESS"
    or incremental_pipeline_summary["successful_files"] != 5
    or incremental_pipeline_summary["failed_files"] != 0
    or incremental_pipeline_summary["uncertain_write_files"] != 0
    or incremental_pipeline_summary["total_inserted_records"]
       != EXPECTED_NEW_RECORDS
):

    PIPELINE_STATUS = "FAILED"

    raise RuntimeError(
        "Incremental ingestion did not meet "
        "expected execution results. "
        "Inspect committed Delta data before retrying."
    )


print("\nIncremental Batch Ingestion: SUCCESS")
print("All five incremental files successfully ingested.")

INCREMENTAL INGESTION EXECUTION AUTHORIZATION
[PASS] Bronze state verified: customers
[PASS] Bronze state verified: products
[PASS] Bronze state verified: orders
[PASS] Bronze state verified: order_items
[PASS] Bronze state verified: payments

Pre-Ingestion Tests: 71/71 PASSED
Target Batch: batch_002
Files Authorized: 5
Expected New Records: 4,219
Write Mode: APPEND
Execution Authorization: APPROVED
INCREMENTAL BRONZE PIPELINE EXECUTION
Pipeline Name: commercelake_bronze_incremental_ingestion
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Execution Started: 2026-11-26T13:29:26.386958+00:00
Planned Files: 5
Pipeline Status: RUNNING

PROCESSING FILE 1
Entity: customers
Batch: batch_002
Source: /Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv
Target: commercelake.bronze.raw_customers
Format: csv
[PASS] Duplicate ingestion guard
[PASS] Source records: 200
[PASS] Schema compatibility: 13 columns
[PASS] Metadata enrichment
Inserted Records: 200
Duration: 8.1

In [0]:
# Cell 18: Validate File-Level Execution Results

file_execution_validation = []


def record_execution_validation(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    file_execution_validation.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("=" * 65)
print("FILE-LEVEL EXECUTION VALIDATION")
print("=" * 65)


# 1. Validate complete execution result coverage

planned_paths = [
    file["file_path"]
    for file in ingestion_plan
]

executed_paths = [
    result["source_path"]
    for result in incremental_file_results
]

record_execution_validation(
    "pipeline",
    "Complete execution result coverage",
    sorted(planned_paths) == sorted(executed_paths)
)

record_execution_validation(
    "pipeline",
    "Unique executed source files",
    len(executed_paths) == len(set(executed_paths))
)


# Build execution result lookup

results_by_path = {
    result["source_path"]: result
    for result in incremental_file_results
}


# 2. Validate each planned file

for file_info in ingestion_plan:

    entity = file_info["entity"]
    source_path = file_info["file_path"]

    print("\n" + "-" * 65)
    print(f"VALIDATING ENTITY: {entity.upper()}")
    print("-" * 65)

    result = results_by_path.get(source_path)

    if result is None:

        record_execution_validation(
            entity,
            "Execution result exists",
            False
        )

        continue

    expected_records = EXPECTED_INCREMENTAL_COUNTS[entity]

    # Check 1: Execution result exists

    record_execution_validation(
        entity,
        "Execution result exists",
        True
    )

    # Check 2: Successful ingestion

    record_execution_validation(
        entity,
        "File ingestion successful",
        result["status"] == "SUCCESS"
    )

    # Check 3: Expected source records

    record_execution_validation(
        entity,
        "Source record count matches",
        result["source_records"] == expected_records
    )

    # Check 4: Expected inserted records

    record_execution_validation(
        entity,
        "Inserted record count matches",
        result["inserted_records"] == expected_records
    )

    # Check 5: Source-to-target reconciliation

    record_execution_validation(
        entity,
        "Source-to-target record reconciliation",
        result["source_records"] is not None
        and result["source_records"]
        == result["inserted_records"]
    )

    # Check 6: Target table mapping

    record_execution_validation(
        entity,
        "Correct target table",
        result["target_table"]
        == file_info["target_table"]
    )

    # Check 7: Batch identifier

    record_execution_validation(
        entity,
        "Correct batch identifier",
        result["batch_id"] == file_info["batch_id"]
    )

    # Check 8: Pipeline run identifier

    record_execution_validation(
        entity,
        "Correct pipeline run identifier",
        result["pipeline_run_id"] == PIPELINE_RUN_ID
    )

    # Check 9: Execution sequence

    record_execution_validation(
        entity,
        "Correct execution sequence",
        result["execution_sequence"]
        == file_info["execution_sequence"]
    )

    # Check 10: Execution duration

    record_execution_validation(
        entity,
        "Valid execution duration",
        result["duration_seconds"] > 0
    )

    print(f"Expected Records: {expected_records:,}")
    print(f"Source Records: {result['source_records']}")
    print(f"Inserted Records: {result['inserted_records']}")
    print(f"Duration: {result['duration_seconds']} seconds")


# 3. Validate consolidated execution metrics

print("\n" + "=" * 65)
print("CONSOLIDATED EXECUTION VALIDATION")
print("=" * 65)

summary = incremental_pipeline_summary

record_execution_validation(
    "pipeline",
    "All planned files successfully processed",
    summary["planned_files"] == len(ingestion_plan)
    and summary["processed_files"] == len(ingestion_plan)
    and summary["successful_files"] == len(ingestion_plan)
)

record_execution_validation(
    "pipeline",
    "No failed, skipped or unprocessed files",
    summary["failed_files"] == 0
    and summary["skipped_files"] == 0
    and summary["unprocessed_files"] == 0
)

record_execution_validation(
    "pipeline",
    "No uncertain write outcomes",
    summary["uncertain_write_files"] == 0
)

record_execution_validation(
    "pipeline",
    "Total source records match",
    summary["total_source_records"]
    == EXPECTED_NEW_RECORDS
)

record_execution_validation(
    "pipeline",
    "Total inserted records match",
    summary["total_inserted_records"]
    == EXPECTED_NEW_RECORDS
)

record_execution_validation(
    "pipeline",
    "Pipeline execution successful",
    summary["pipeline_status"] == "SUCCESS"
)


# 4. Display file-level execution metrics

print("\n" + "=" * 65)
print("FILE-LEVEL EXECUTION METRICS")
print("=" * 65)

display(
    spark.createDataFrame(
        incremental_file_results
    ).orderBy("execution_sequence")
)


# 5. Display validation results

display(
    spark.createDataFrame(
        file_execution_validation
    )
)


# 6. Generate final validation summary

total_checks = len(file_execution_validation)

passed_checks = sum(
    result["status"] == "PASS"
    for result in file_execution_validation
)

failed_checks = total_checks - passed_checks

expected_checks = len(ingestion_plan) * 10 + 8

throughput = (
    summary["total_inserted_records"]
    / summary["duration_seconds"]
    if summary["duration_seconds"] > 0
    else 0
)

print("\n" + "=" * 65)
print("FILE-LEVEL VALIDATION SUMMARY")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Files Validated: {len(incremental_file_results)}")
print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Verified Inserted Records: "
    f"{summary['total_inserted_records']:,}"
)

print(
    f"Execution Duration: "
    f"{summary['duration_seconds']} seconds"
)

print(f"Pipeline Throughput: {throughput:.2f} records/sec")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0 or total_checks != expected_checks:

    raise RuntimeError(
        "File-level execution validation failed. "
        "Inspect execution results and committed data."
    )

print("\nFile-Level Execution Validation: SUCCESS")

FILE-LEVEL EXECUTION VALIDATION
[PASS] Complete execution result coverage
[PASS] Unique executed source files

-----------------------------------------------------------------
VALIDATING ENTITY: CUSTOMERS
-----------------------------------------------------------------
[PASS] Execution result exists
[PASS] File ingestion successful
[PASS] Source record count matches
[PASS] Inserted record count matches
[PASS] Source-to-target record reconciliation
[PASS] Correct target table
[PASS] Correct batch identifier
[PASS] Correct pipeline run identifier
[PASS] Correct execution sequence
[PASS] Valid execution duration
Expected Records: 200
Source Records: 200
Inserted Records: 200
Duration: 8.1908 seconds

-----------------------------------------------------------------
VALIDATING ENTITY: PRODUCTS
-----------------------------------------------------------------
[PASS] Execution result exists
[PASS] File ingestion successful
[PASS] Source record count matches
[PASS] Inserted record count mat

batch_id,duration_seconds,entity,execution_sequence,inserted_records,message,pipeline_run_id,source_path,source_records,status,target_table
batch_002,8.1908,customers,1,200,Incremental Delta append verified,5cd36830-4695-40d1-ab22-17d8db1df9f9,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,200,SUCCESS,commercelake.bronze.raw_customers
batch_002,6.5008,products,2,50,Incremental Delta append verified,5cd36830-4695-40d1-ab22-17d8db1df9f9,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,50,SUCCESS,commercelake.bronze.raw_products
batch_002,5.7418,orders,3,1000,Incremental Delta append verified,5cd36830-4695-40d1-ab22-17d8db1df9f9,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,1000,SUCCESS,commercelake.bronze.raw_orders
batch_002,6.0477,order_items,4,1969,Incremental Delta append verified,5cd36830-4695-40d1-ab22-17d8db1df9f9,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,1969,SUCCESS,commercelake.bronze.raw_order_items
batch_002,5.8139,payments,5,1000,Incremental Delta append verified,5cd36830-4695-40d1-ab22-17d8db1df9f9,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,1000,SUCCESS,commercelake.bronze.raw_payments


check_name,entity,status
Complete execution result coverage,pipeline,PASS
Unique executed source files,pipeline,PASS
Execution result exists,customers,PASS
File ingestion successful,customers,PASS
Source record count matches,customers,PASS
Inserted record count matches,customers,PASS
Source-to-target record reconciliation,customers,PASS
Correct target table,customers,PASS
Correct batch identifier,customers,PASS
Correct pipeline run identifier,customers,PASS



FILE-LEVEL VALIDATION SUMMARY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Files Validated: 5
Expected Checks: 58
Executed Checks: 58
Passed Checks: 58
Failed Checks: 0
Verified Inserted Records: 4,219
Execution Duration: 32.2961 seconds
Pipeline Throughput: 130.63 records/sec
Validation Success Rate: 100.00%

File-Level Execution Validation: SUCCESS


In [0]:
# Cell 19: Verify Incremental Delta Transaction History

delta_history_validation = []
delta_transaction_results = []


def record_delta_validation(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    delta_history_validation.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("=" * 65)
print("INCREMENTAL DELTA TRANSACTION VALIDATION")
print("=" * 65)


# 1. Inspect transaction history for each Bronze table

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    expected_records = EXPECTED_INCREMENTAL_COUNTS[entity]

    previous_version = bronze_state_before[entity]["version"]

    expected_version = previous_version + 1

    print("\n" + "-" * 65)
    print(f"VALIDATING TABLE: {target_table}")
    print("-" * 65)

    history_df = spark.sql(
        f"DESCRIBE HISTORY {target_table}"
    )

    latest_transaction = (
        history_df
        .orderBy(F.col("version").desc())
        .first()
        .asDict()
    )

    operation_metrics = (
        latest_transaction.get("operationMetrics") or {}
    )

    operation_parameters = (
        latest_transaction.get("operationParameters") or {}
    )

    actual_version = latest_transaction["version"]

    operation = latest_transaction["operation"]

    write_mode = operation_parameters.get(
        "mode", "UNKNOWN"
    )

    output_rows = int(
        operation_metrics.get("numOutputRows", -1)
    )

    output_files = int(
        operation_metrics.get("numFiles", -1)
    )

    output_bytes = int(
        operation_metrics.get("numOutputBytes", -1)
    )


    # 2. Validate transaction properties

    record_delta_validation(
        entity,
        "Expected Delta version",
        actual_version == expected_version
    )

    record_delta_validation(
        entity,
        "Correct transaction operation",
        operation == "WRITE"
    )

    record_delta_validation(
        entity,
        "Append write mode",
        write_mode.lower() == "append"
    )

    record_delta_validation(
        entity,
        "Transaction output row count",
        output_rows == expected_records
    )

    record_delta_validation(
        entity,
        "Delta data files created",
        output_files > 0
    )

    record_delta_validation(
        entity,
        "Valid transaction output size",
        output_bytes > 0
    )


    # 3. Verify previous version still exists

    previous_transaction_count = (
        history_df
        .filter(
            F.col("version") == previous_version
        )
        .count()
    )

    record_delta_validation(
        entity,
        "Historical transaction preserved",
        previous_transaction_count == 1
    )


    # 4. Verify exactly one new transaction

    new_transaction_count = (
        history_df
        .filter(
            F.col("version") > previous_version
        )
        .count()
    )

    record_delta_validation(
        entity,
        "Exactly one incremental transaction",
        new_transaction_count == 1
    )


    # 5. Store transaction metrics

    delta_transaction_results.append({
        "entity": entity,
        "target_table": target_table,
        "previous_version": previous_version,
        "expected_version": expected_version,
        "actual_version": actual_version,
        "operation": operation,
        "write_mode": write_mode,
        "expected_records": expected_records,
        "output_rows": output_rows,
        "output_files": output_files,
        "output_bytes": output_bytes,
        "commit_timestamp": str(
            latest_transaction["timestamp"]
        )
    })

    print(f"Previous Version: {previous_version}")
    print(f"Current Version: {actual_version}")
    print(f"Operation: {operation}")
    print(f"Write Mode: {write_mode}")
    print(f"Expected Records: {expected_records:,}")
    print(f"Transaction Output Rows: {output_rows:,}")
    print(f"New Delta Files: {output_files}")
    print(f"Output Bytes: {output_bytes:,}")
    print(
        f"Commit Timestamp: "
        f"{latest_transaction['timestamp']}"
    )


# 6. Validate consolidated transaction metrics

print("\n" + "=" * 65)
print("CONSOLIDATED DELTA TRANSACTION VALIDATION")
print("=" * 65)

total_output_rows = sum(
    result["output_rows"]
    for result in delta_transaction_results
)

total_output_files = sum(
    result["output_files"]
    for result in delta_transaction_results
)

total_output_bytes = sum(
    result["output_bytes"]
    for result in delta_transaction_results
)

record_delta_validation(
    "pipeline",
    "All Bronze transactions inspected",
    len(delta_transaction_results)
    == len(SOURCE_CONFIG)
)

record_delta_validation(
    "pipeline",
    "Total transaction output rows match",
    total_output_rows == EXPECTED_NEW_RECORDS
)

record_delta_validation(
    "pipeline",
    "Transaction and execution reconciliation",
    total_output_rows
    == incremental_pipeline_summary[
        "total_inserted_records"
    ]
)


# 7. Display transaction metrics

print("\n" + "=" * 65)
print("DELTA TRANSACTION METRICS")
print("=" * 65)

display(
    spark.createDataFrame(
        delta_transaction_results
    )
)


# 8. Display validation results

display(
    spark.createDataFrame(
        delta_history_validation
    )
)


# 9. Generate final validation summary

total_checks = len(delta_history_validation)

passed_checks = sum(
    result["status"] == "PASS"
    for result in delta_history_validation
)

failed_checks = total_checks - passed_checks

expected_checks = len(SOURCE_CONFIG) * 8 + 3

print("\n" + "=" * 65)
print("DELTA TRANSACTION VALIDATION SUMMARY")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Tables Inspected: {len(delta_transaction_results)}")
print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")
print(f"Total Transaction Output Rows: {total_output_rows:,}")
print(f"Total New Delta Files: {total_output_files}")
print(f"Total Output Bytes: {total_output_bytes:,}")

print(
    f"Total Output Size: "
    f"{total_output_bytes / 1024:.2f} KiB"
)

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0 or total_checks != expected_checks:

    raise RuntimeError(
        "Delta transaction validation failed. "
        "Inspect transaction history before proceeding."
    )

print("\nDelta Transaction History Validation: SUCCESS")
print("All incremental append transactions verified.")

INCREMENTAL DELTA TRANSACTION VALIDATION

-----------------------------------------------------------------
VALIDATING TABLE: commercelake.bronze.raw_customers
-----------------------------------------------------------------
[PASS] Expected Delta version
[PASS] Correct transaction operation
[PASS] Append write mode
[PASS] Transaction output row count
[PASS] Delta data files created
[PASS] Valid transaction output size
[PASS] Historical transaction preserved
[PASS] Exactly one incremental transaction
Previous Version: 1
Current Version: 2
Operation: WRITE
Write Mode: Append
Expected Records: 200
Transaction Output Rows: 200
New Delta Files: 1
Output Bytes: 22,105
Commit Timestamp: 2026-11-26 13:29:33.120000

-----------------------------------------------------------------
VALIDATING TABLE: commercelake.bronze.raw_products
-----------------------------------------------------------------
[PASS] Expected Delta version
[PASS] Correct transaction operation
[PASS] Append write mode
[PASS] 

actual_version,commit_timestamp,entity,expected_records,expected_version,operation,output_bytes,output_files,output_rows,previous_version,target_table,write_mode
2,2026-11-26 13:29:33.120000,customers,200,2,WRITE,22105,1,200,1,commercelake.bronze.raw_customers,Append
2,2026-11-26 13:29:40.225000,products,50,2,WRITE,9335,1,50,1,commercelake.bronze.raw_products,Append
2,2026-11-26 13:29:45.940000,orders,1000,2,WRITE,65278,1,1000,1,commercelake.bronze.raw_orders,Append
2,2026-11-26 13:29:51.897000,order_items,1969,2,WRITE,124853,1,1969,1,commercelake.bronze.raw_order_items,Append
2,2026-11-26 13:29:57.797000,payments,1000,2,WRITE,69526,1,1000,1,commercelake.bronze.raw_payments,Append


check_name,entity,status
Expected Delta version,customers,PASS
Correct transaction operation,customers,PASS
Append write mode,customers,PASS
Transaction output row count,customers,PASS
Delta data files created,customers,PASS
Valid transaction output size,customers,PASS
Historical transaction preserved,customers,PASS
Exactly one incremental transaction,customers,PASS
Expected Delta version,products,PASS
Correct transaction operation,products,PASS



DELTA TRANSACTION VALIDATION SUMMARY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Tables Inspected: 5
Expected Checks: 43
Executed Checks: 43
Passed Checks: 43
Failed Checks: 0
Total Transaction Output Rows: 4,219
Total New Delta Files: 5
Total Output Bytes: 291,097
Total Output Size: 284.27 KiB
Validation Success Rate: 100.00%

Delta Transaction History Validation: SUCCESS
All incremental append transactions verified.


In [0]:
# Cell 20: Validate Persisted Incremental Metadata

metadata_validation_results = []
incremental_metadata_summary = []


def record_metadata_validation(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    metadata_validation_results.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("=" * 65)
print("INCREMENTAL INGESTION METADATA VALIDATION")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print("Target Batch: batch_002")
print("Execution Mode: READ ONLY")


# 1. Validate persisted records for each entity

for file_info in ingestion_plan:

    entity = file_info["entity"]
    target_table = file_info["target_table"]
    source_path = file_info["file_path"]

    expected_records = EXPECTED_INCREMENTAL_COUNTS[entity]

    print("\n" + "-" * 65)
    print(f"VALIDATING ENTITY: {entity.upper()}")
    print("-" * 65)

    bronze_df = spark.table(target_table)

    incremental_df = bronze_df.filter(
        F.col("_batch_id") == "batch_002"
    )

    historical_df = bronze_df.filter(
        F.col("_batch_id") == "batch_001"
    )

    incremental_count = incremental_df.count()

    historical_count = historical_df.count()

    # Check 1: Incremental record count

    record_metadata_validation(
        entity,
        "Expected incremental record count",
        incremental_count == expected_records
    )

    # Check 2: Metadata columns present

    record_metadata_validation(
        entity,
        "All metadata columns present",
        set(METADATA_COLUMNS).issubset(
            bronze_df.columns
        )
    )

    # Check 3: No null metadata

    null_condition = F.lit(False)

    for column in METADATA_COLUMNS:

        null_condition = (
            null_condition
            | F.col(column).isNull()
        )

    null_metadata_count = (
        incremental_df
        .filter(null_condition)
        .count()
    )

    record_metadata_validation(
        entity,
        "No null ingestion metadata",
        null_metadata_count == 0
    )

    # Check 4: Correct source file path

    invalid_source_paths = incremental_df.filter(
        ~F.col("_source_file").eqNullSafe(
            F.lit(source_path)
        )
    ).count()

    record_metadata_validation(
        entity,
        "Correct source file metadata",
        invalid_source_paths == 0
    )

    # Check 5: Correct batch identifier

    source_file_df = bronze_df.filter(
        F.col("_source_file") == source_path
    )

    invalid_batch_ids = source_file_df.filter(
        ~F.col("_batch_id").eqNullSafe(
            F.lit("batch_002")
        )
    ).count()

    record_metadata_validation(
        entity,
        "Correct batch identifier",
        invalid_batch_ids == 0
    )

    # Check 6: Correct pipeline run identifier

    invalid_run_ids = incremental_df.filter(
        ~F.col("_pipeline_run_id").eqNullSafe(
            F.lit(PIPELINE_RUN_ID)
        )
    ).count()

    record_metadata_validation(
        entity,
        "Correct pipeline run identifier",
        invalid_run_ids == 0
    )

    # Check 7: Valid ingestion timestamp

    execution_started = datetime.fromisoformat(
        incremental_pipeline_summary[
            "execution_started"
        ]
    )

    execution_completed = datetime.fromisoformat(
        incremental_pipeline_summary[
            "execution_completed"
        ]
    )

    invalid_timestamps = incremental_df.filter(
        F.col("_ingestion_timestamp").isNull()
        | (
            F.col("_ingestion_timestamp")
            < F.lit(execution_started).cast("timestamp")
        )
        | (
            F.col("_ingestion_timestamp")
            > F.lit(execution_completed).cast("timestamp")
        )
    ).count()

    record_metadata_validation(
        entity,
        "Ingestion timestamp within execution window",
        invalid_timestamps == 0
    )

    # Check 8: One ingestion timestamp per source file

    distinct_timestamps = (
        incremental_df
        .select("_ingestion_timestamp")
        .distinct()
        .count()
    )

    record_metadata_validation(
        entity,
        "Consistent file-level ingestion timestamp",
        distinct_timestamps == 1
    )

    # Check 9: Valid SHA-256 hash format

    invalid_hashes = incremental_df.filter(
        F.col("_record_hash").isNull()
        | (
            ~F.col("_record_hash")
            .rlike("^[a-f0-9]{64}$")
        )
    ).count()

    record_metadata_validation(
        entity,
        "Valid SHA-256 record hash format",
        invalid_hashes == 0
    )

    # Check 10: Recompute persisted record hashes

    business_columns = sorted([
        column
        for column in bronze_df.columns
        if column not in METADATA_COLUMNS
    ])

    expected_hash = F.sha2(
        F.to_json(
            F.struct(
                *[
                    F.col(column)
                    for column in business_columns
                ]
            ),
            options={
                "ignoreNullFields": "false"
            }
        ),
        256
    )

    hash_mismatches = incremental_df.filter(
        ~F.col("_record_hash").eqNullSafe(
            expected_hash
        )
    ).count()

    record_metadata_validation(
        entity,
        "Persisted record hash consistency",
        hash_mismatches == 0
    )

    # Check 11: Historical record count preserved

    record_metadata_validation(
        entity,
        "Historical record count preserved",
        historical_count
        == EXPECTED_HISTORICAL_COUNTS[entity]
    )

    # Check 12: Historical records unchanged
    # Compare current historical rows with Delta version 1

    previous_version = bronze_state_before[
        entity
    ]["version"]

    previous_df = (
        spark.read
        .format("delta")
        .option("versionAsOf", previous_version)
        .table(target_table)
    )

    historical_df = historical_df.select(
        *previous_df.columns
    )

    removed_or_modified = (
        previous_df
        .exceptAll(historical_df)
        .count()
    )

    added_or_modified = (
        historical_df
        .exceptAll(previous_df)
        .count()
    )

    record_metadata_validation(
        entity,
        "Historical records unchanged using Time Travel",
        removed_or_modified == 0
        and added_or_modified == 0
    )

    # Check 13: No unexpected batch identifiers

    unexpected_batch_records = bronze_df.filter(
        F.col("_batch_id").isNull()
        | (
            ~F.col("_batch_id").isin(
                "batch_001",
                "batch_002"
            )
        )
    ).count()

    record_metadata_validation(
        entity,
        "No unexpected batch identifiers",
        unexpected_batch_records == 0
    )

    # Check 14: Complete source-file reconciliation

    source_df = read_source_data(
        file_path=source_path,
        file_format=file_info["file_format"]
    ).select(*business_columns)

    persisted_business_df = incremental_df.select(
        *business_columns
    )

    missing_source_records = (
        source_df
        .exceptAll(persisted_business_df)
        .count()
    )

    unexpected_persisted_records = (
        persisted_business_df
        .exceptAll(source_df)
        .count()
    )

    record_metadata_validation(
        entity,
        "Exact source-to-Bronze record reconciliation",
        missing_source_records == 0
        and unexpected_persisted_records == 0
    )

    # Store metadata validation metrics

    incremental_metadata_summary.append({
        "entity": entity,
        "target_table": target_table,
        "batch_id": "batch_002",
        "historical_records": historical_count,
        "incremental_records": incremental_count,
        "null_metadata_records": null_metadata_count,
        "invalid_source_paths": invalid_source_paths,
        "invalid_batch_ids": invalid_batch_ids,
        "invalid_run_ids": invalid_run_ids,
        "invalid_timestamps": invalid_timestamps,
        "invalid_hashes": invalid_hashes,
        "hash_mismatches": hash_mismatches,
        "historical_record_differences": (
            removed_or_modified + added_or_modified
        ),
        "source_record_differences": (
            missing_source_records
            + unexpected_persisted_records
        ),
        "unexpected_batch_records": (
            unexpected_batch_records
        )
    })

    print(f"Historical Records: {historical_count:,}")
    print(f"Incremental Records: {incremental_count:,}")
    print(f"Null Metadata Records: {null_metadata_count}")
    print(f"Hash Mismatches: {hash_mismatches}")

    print(
        f"Historical Record Differences: "
        f"{removed_or_modified + added_or_modified}"
    )

    print(
        f"Source Record Differences: "
        f"{missing_source_records + unexpected_persisted_records}"
    )


# 2. Validate consolidated metadata results

print("\n" + "=" * 65)
print("CONSOLIDATED METADATA VALIDATION")
print("=" * 65)

total_incremental_records = sum(
    result["incremental_records"]
    for result in incremental_metadata_summary
)

total_historical_records = sum(
    result["historical_records"]
    for result in incremental_metadata_summary
)

total_hash_mismatches = sum(
    result["hash_mismatches"]
    for result in incremental_metadata_summary
)

total_historical_differences = sum(
    result["historical_record_differences"]
    for result in incremental_metadata_summary
)

total_source_differences = sum(
    result["source_record_differences"]
    for result in incremental_metadata_summary
)

record_metadata_validation(
    "pipeline",
    "Complete entity validation coverage",
    len(incremental_metadata_summary) == 5
)

record_metadata_validation(
    "pipeline",
    "Total incremental records reconciled",
    total_incremental_records == EXPECTED_NEW_RECORDS
)

record_metadata_validation(
    "pipeline",
    "Total historical records preserved",
    total_historical_records
    == sum(EXPECTED_HISTORICAL_COUNTS.values())
)

record_metadata_validation(
    "pipeline",
    "Zero persisted record hash mismatches",
    total_hash_mismatches == 0
)

record_metadata_validation(
    "pipeline",
    "Zero historical record differences",
    total_historical_differences == 0
)

record_metadata_validation(
    "pipeline",
    "Zero source-to-Bronze record differences",
    total_source_differences == 0
)


# 3. Display metadata validation metrics

display(
    spark.createDataFrame(
        incremental_metadata_summary
    )
)

display(
    spark.createDataFrame(
        metadata_validation_results
    )
)


# 4. Generate final validation summary

total_checks = len(metadata_validation_results)

passed_checks = sum(
    result["status"] == "PASS"
    for result in metadata_validation_results
)

failed_checks = total_checks - passed_checks

expected_checks = len(SOURCE_CONFIG) * 14 + 6

print("\n" + "=" * 65)
print("INCREMENTAL METADATA VALIDATION SUMMARY")
print("=" * 65)

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Entities Validated: {len(incremental_metadata_summary)}")
print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Historical Records Preserved: "
    f"{total_historical_records:,}"
)

print(
    f"Incremental Records Validated: "
    f"{total_incremental_records:,}"
)

print(f"Hash Mismatches: {total_hash_mismatches}")
print(f"Historical Record Differences: {total_historical_differences}")
print(f"Source Record Differences: {total_source_differences}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0 or total_checks != expected_checks:

    raise RuntimeError(
        "Incremental metadata validation failed. "
        "Inspect persisted Bronze records."
    )

print("\nIncremental Metadata Validation: SUCCESS")
print("Historical and incremental records verified.")

INCREMENTAL INGESTION METADATA VALIDATION
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Target Batch: batch_002
Execution Mode: READ ONLY

-----------------------------------------------------------------
VALIDATING ENTITY: CUSTOMERS
-----------------------------------------------------------------
[PASS] Expected incremental record count
[PASS] All metadata columns present
[PASS] No null ingestion metadata
[PASS] Correct source file metadata
[PASS] Correct batch identifier
[PASS] Correct pipeline run identifier
[PASS] Ingestion timestamp within execution window
[PASS] Consistent file-level ingestion timestamp
[PASS] Valid SHA-256 record hash format
[PASS] Persisted record hash consistency
[PASS] Historical record count preserved
[PASS] Historical records unchanged using Time Travel
[PASS] No unexpected batch identifiers
[PASS] Exact source-to-Bronze record reconciliation
Historical Records: 1,000
Incremental Records: 200
Null Metadata Records: 0
Hash Mismatches: 0
Historical R

batch_id,entity,hash_mismatches,historical_record_differences,historical_records,incremental_records,invalid_batch_ids,invalid_hashes,invalid_run_ids,invalid_source_paths,invalid_timestamps,null_metadata_records,source_record_differences,target_table,unexpected_batch_records
batch_002,customers,0,0,1000,200,0,0,0,0,0,0,0,commercelake.bronze.raw_customers,0
batch_002,products,0,0,200,50,0,0,0,0,0,0,0,commercelake.bronze.raw_products,0
batch_002,orders,0,0,5000,1000,0,0,0,0,0,0,0,commercelake.bronze.raw_orders,0
batch_002,order_items,0,0,10055,1969,0,0,0,0,0,0,0,commercelake.bronze.raw_order_items,0
batch_002,payments,0,0,5000,1000,0,0,0,0,0,0,0,commercelake.bronze.raw_payments,0


check_name,entity,status
Expected incremental record count,customers,PASS
All metadata columns present,customers,PASS
No null ingestion metadata,customers,PASS
Correct source file metadata,customers,PASS
Correct batch identifier,customers,PASS
Correct pipeline run identifier,customers,PASS
Ingestion timestamp within execution window,customers,PASS
Consistent file-level ingestion timestamp,customers,PASS
Valid SHA-256 record hash format,customers,PASS
Persisted record hash consistency,customers,PASS



INCREMENTAL METADATA VALIDATION SUMMARY
Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Entities Validated: 5
Expected Checks: 76
Executed Checks: 76
Passed Checks: 76
Failed Checks: 0
Historical Records Preserved: 21,255
Incremental Records Validated: 4,219
Hash Mismatches: 0
Historical Record Differences: 0
Source Record Differences: 0
Validation Success Rate: 100.00%

Incremental Metadata Validation: SUCCESS
Historical and incremental records verified.


In [0]:
# Cell 21: Capture Pre-Rerun Bronze State

EXPECTED_CUMULATIVE_COUNTS = {
    "customers": 1200,
    "products": 250,
    "orders": 6000,
    "order_items": 12024,
    "payments": 6000
}

idempotency_baseline = {}


def capture_bronze_state(
    target_table: str
) -> dict:
    """
    Capture the current state of a Bronze Delta table
    for idempotency verification.
    """

    bronze_df = spark.table(target_table)

    # 1. Total records

    total_records = bronze_df.count()

    # 2. Records per batch

    batch_counts = {
        row["_batch_id"]: row["count"]
        for row in (
            bronze_df
            .groupBy("_batch_id")
            .count()
            .collect()
        )
    }

    # 3. Distinct source files

    distinct_source_files = (
        bronze_df
        .select("_source_file")
        .distinct()
        .count()
    )

    # 4. Distinct pipeline run identifiers

    distinct_pipeline_runs = (
        bronze_df
        .select("_pipeline_run_id")
        .distinct()
        .count()
    )

    # 5. Current Delta version

    latest_version = (
        spark.sql(
            f"DESCRIBE HISTORY {target_table}"
        )
        .agg(
            F.max("version").alias("version")
        )
        .first()["version"]
    )

    # 6. Physical Delta table metrics

    table_details = (
        spark.sql(
            f"DESCRIBE DETAIL {target_table}"
        )
        .first()
        .asDict()
    )

    # 7. Return complete table state

    return {
        "target_table": target_table,
        "total_records": total_records,
        "batch_counts": batch_counts,
        "distinct_source_files": distinct_source_files,
        "distinct_pipeline_runs": distinct_pipeline_runs,
        "delta_version": latest_version,
        "num_files": table_details["numFiles"],
        "size_bytes": table_details["sizeInBytes"]
    }


print("=" * 65)
print("IDEMPOTENCY BASELINE CAPTURE")
print("=" * 65)

print("Execution Mode: READ ONLY")
print("Baseline Type: PRE-RERUN")

IDEMPOTENCY_BASELINE_TIMESTAMP = (
    datetime.now(timezone.utc)
)

print(
    f"Capture Timestamp: "
    f"{IDEMPOTENCY_BASELINE_TIMESTAMP.isoformat()}"
)


# Capture baseline for all Bronze tables

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    state = capture_bronze_state(
        target_table=target_table
    )

    # Validate expected cumulative record count

    if (
        state["total_records"]
        != EXPECTED_CUMULATIVE_COUNTS[entity]
    ):
        raise RuntimeError(
            f"Unexpected baseline record count "
            f"for {entity}"
        )

    # Validate expected batch distribution

    expected_batch_counts = {
        "batch_001": EXPECTED_HISTORICAL_COUNTS[entity],
        "batch_002": EXPECTED_INCREMENTAL_COUNTS[entity]
    }

    if state["batch_counts"] != expected_batch_counts:

        raise RuntimeError(
            f"Unexpected batch distribution "
            f"for {entity}"
        )

    # Validate source file coverage

    if state["distinct_source_files"] != 2:

        raise RuntimeError(
            f"Unexpected source file count "
            f"for {entity}"
        )

    # Validate pipeline run coverage

    if state["distinct_pipeline_runs"] != 2:

        raise RuntimeError(
            f"Unexpected pipeline run count "
            f"for {entity}"
        )

    # Validate Delta version

    if state["delta_version"] != 2:

        raise RuntimeError(
            f"Unexpected Delta version "
            f"for {entity}"
        )

    idempotency_baseline[entity] = state

    print("\n" + "-" * 65)
    print(f"ENTITY: {entity.upper()}")
    print("-" * 65)

    print(f"Target Table: {target_table}")
    print(f"Total Records: {state['total_records']:,}")
    print(f"Batch Distribution: {state['batch_counts']}")

    print(
        f"Distinct Source Files: "
        f"{state['distinct_source_files']}"
    )

    print(
        f"Distinct Pipeline Runs: "
        f"{state['distinct_pipeline_runs']}"
    )

    print(f"Delta Version: {state['delta_version']}")
    print(f"Physical Data Files: {state['num_files']}")
    print(f"Storage Bytes: {state['size_bytes']:,}")

    print("Baseline Status: PASS")


# Prepare baseline display

baseline_display = []

for entity, state in idempotency_baseline.items():

    baseline_display.append({
        "entity": entity,
        "target_table": state["target_table"],
        "historical_records": (
            state["batch_counts"]["batch_001"]
        ),
        "incremental_records": (
            state["batch_counts"]["batch_002"]
        ),
        "total_records": state["total_records"],
        "distinct_source_files": (
            state["distinct_source_files"]
        ),
        "distinct_pipeline_runs": (
            state["distinct_pipeline_runs"]
        ),
        "delta_version": state["delta_version"],
        "num_files": state["num_files"],
        "size_bytes": state["size_bytes"]
    })

display(
    spark.createDataFrame(
        baseline_display
    )
)


# Consolidated baseline metrics

baseline_total_records = sum(
    state["total_records"]
    for state in idempotency_baseline.values()
)

baseline_total_source_files = sum(
    state["distinct_source_files"]
    for state in idempotency_baseline.values()
)

baseline_total_delta_files = sum(
    state["num_files"]
    for state in idempotency_baseline.values()
)

baseline_total_storage_bytes = sum(
    state["size_bytes"]
    for state in idempotency_baseline.values()
)


# Final baseline validation

if (
    len(idempotency_baseline) != 5
    or baseline_total_records != 25474
    or baseline_total_source_files != 10
):
    raise RuntimeError(
        "Idempotency baseline validation failed."
    )


print("\n" + "=" * 65)
print("IDEMPOTENCY BASELINE SUMMARY")
print("=" * 65)

print(f"Tables Captured: {len(idempotency_baseline)}")
print(f"Total Bronze Records: {baseline_total_records:,}")

print(
    f"Total Ingested Source Files: "
    f"{baseline_total_source_files}"
)

print(
    f"Total Physical Delta Files: "
    f"{baseline_total_delta_files}"
)

print(
    f"Total Storage Bytes: "
    f"{baseline_total_storage_bytes:,}"
)

print(
    f"Total Storage Size: "
    f"{baseline_total_storage_bytes / 1024**2:.2f} MiB"
)

print("Delta Versions: All tables at version 2")
print("Baseline Validation: SUCCESS")

print("\nIdempotency Baseline Capture: SUCCESS")
print("Pipeline is ready for repeated execution testing.")

IDEMPOTENCY BASELINE CAPTURE
Execution Mode: READ ONLY
Baseline Type: PRE-RERUN
Capture Timestamp: 2026-11-26T13:41:58.750462+00:00

-----------------------------------------------------------------
ENTITY: CUSTOMERS
-----------------------------------------------------------------
Target Table: commercelake.bronze.raw_customers
Total Records: 1,200
Batch Distribution: {'batch_001': 1000, 'batch_002': 200}
Distinct Source Files: 2
Distinct Pipeline Runs: 2
Delta Version: 2
Physical Data Files: 2
Storage Bytes: 101,167
Baseline Status: PASS

-----------------------------------------------------------------
ENTITY: PRODUCTS
-----------------------------------------------------------------
Target Table: commercelake.bronze.raw_products
Total Records: 250
Batch Distribution: {'batch_001': 200, 'batch_002': 50}
Distinct Source Files: 2
Distinct Pipeline Runs: 2
Delta Version: 2
Physical Data Files: 2
Storage Bytes: 28,360
Baseline Status: PASS

----------------------------------------------

delta_version,distinct_pipeline_runs,distinct_source_files,entity,historical_records,incremental_records,num_files,size_bytes,target_table,total_records
2,2,2,customers,1000,200,2,101167,commercelake.bronze.raw_customers,1200
2,2,2,products,200,50,2,28360,commercelake.bronze.raw_products,250
2,2,2,orders,5000,1000,2,366231,commercelake.bronze.raw_orders,6000
2,2,2,order_items,10055,1969,2,696716,commercelake.bronze.raw_order_items,12024
2,2,2,payments,5000,1000,2,414890,commercelake.bronze.raw_payments,6000



IDEMPOTENCY BASELINE SUMMARY
Tables Captured: 5
Total Bronze Records: 25,474
Total Ingested Source Files: 10
Total Physical Delta Files: 10
Total Storage Bytes: 1,607,364
Total Storage Size: 1.53 MiB
Delta Versions: All tables at version 2
Baseline Validation: SUCCESS

Idempotency Baseline Capture: SUCCESS
Pipeline is ready for repeated execution testing.


In [0]:
# Cell 22: Refresh Discovery and Ingestion Registry

print("=" * 65)
print("REPEATED EXECUTION FILE DISCOVERY")
print("=" * 65)

print("Execution Mode: READ ONLY")


# 1. Rediscover source files

rerun_discovered_files = []

for entity, config in SOURCE_CONFIG.items():

    discovered_files = discover_source_files(
        entity=entity,
        config=config
    )

    rerun_discovered_files.extend(
        discovered_files
    )

    print(
        f"[PASS] {entity}: "
        f"{len(discovered_files)} source files discovered"
    )


# 2. Refresh ingestion registry from Bronze tables

print("\n" + "=" * 65)
print("REFRESHING INGESTION REGISTRY")
print("=" * 65)

rerun_ingested_files_by_entity = {}

rerun_ingested_file_registry = []

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    ingested_paths = get_ingested_file_paths(
        target_table=target_table
    )

    rerun_ingested_files_by_entity[entity] = (
        ingested_paths
    )

    for file_path in sorted(ingested_paths):

        rerun_ingested_file_registry.append({
            "entity": entity,
            "target_table": target_table,
            "file_path": file_path,
            "status": "ALREADY_INGESTED"
        })

    print(
        f"[PASS] {entity}: "
        f"{len(ingested_paths)} ingested files registered"
    )


# 3. Reclassify discovered files

rerun_classified_files = classify_source_files(
    rerun_discovered_files,
    rerun_ingested_files_by_entity
)

rerun_new_source_files = [
    file
    for file in rerun_classified_files
    if file["planned_action"] == "INGEST"
]

rerun_skipped_source_files = [
    file
    for file in rerun_classified_files
    if file["planned_action"] == "SKIP"
]


# 4. Validate refreshed discovery and registry

rerun_registry_validation = []


def record_registry_validation(
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    rerun_registry_validation.append({
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("\n" + "=" * 65)
print("REFRESHED REGISTRY VALIDATION")
print("=" * 65)


# Prepare file path sets

rerun_discovered_paths = {
    file["file_path"]
    for file in rerun_discovered_files
}

rerun_registered_paths = {
    file["file_path"]
    for file in rerun_ingested_file_registry
}

original_discovered_paths = {
    file["file_path"]
    for file in discovered_source_files
}

rerun_skipped_paths = {
    file["file_path"]
    for file in rerun_skipped_source_files
}


# Check 1: Expected discovered files

record_registry_validation(
    "Expected source file count",
    len(rerun_discovered_files) == 10
)


# Check 2: Unique discovered files

record_registry_validation(
    "Unique discovered source paths",
    len(rerun_discovered_paths)
    == len(rerun_discovered_files)
)


# Check 3: Source inventory unchanged

record_registry_validation(
    "Source file inventory unchanged",
    rerun_discovered_paths
    == original_discovered_paths
)


# Check 4: Expected registry size

record_registry_validation(
    "Expected ingested registry size",
    len(rerun_ingested_file_registry) == 10
)


# Check 5: Unique registered files

record_registry_validation(
    "Unique registered source paths",
    len(rerun_registered_paths)
    == len(rerun_ingested_file_registry)
)


# Check 6: Complete source registration

record_registry_validation(
    "All discovered files registered in Bronze",
    rerun_discovered_paths
    == rerun_registered_paths
)


# Check 7: Two registered files per entity

record_registry_validation(
    "Complete entity-level registry coverage",
    set(rerun_ingested_files_by_entity)
    == set(SOURCE_CONFIG)
    and all(
        len(paths) == 2
        for paths in rerun_ingested_files_by_entity.values()
    )
)


# Check 8: Both batches discovered

rerun_batch_counts = Counter(
    file["batch_id"]
    for file in rerun_discovered_files
)

record_registry_validation(
    "Both source batches discovered",
    dict(rerun_batch_counts) == {
        "batch_001": 5,
        "batch_002": 5
    }
)


# Check 9: No new source files

record_registry_validation(
    "Zero new source files",
    len(rerun_new_source_files) == 0
)


# Check 10: All files skipped

record_registry_validation(
    "All discovered files marked for skipping",
    len(rerun_skipped_source_files) == 10
    and rerun_skipped_paths == rerun_discovered_paths
)


# Check 11: Correct ingestion classification

record_registry_validation(
    "All files classified as already ingested",
    len(rerun_classified_files) == 10
    and all(
        file["already_ingested"]
        and file["ingestion_status"] == "ALREADY_INGESTED"
        and file["planned_action"] == "SKIP"
        for file in rerun_classified_files
    )
)


# Check 12: Original ingestion results preserved

record_registry_validation(
    "Original execution results preserved",
    incremental_pipeline_summary["pipeline_run_id"]
    == PIPELINE_RUN_ID
    and incremental_pipeline_summary["pipeline_status"]
    == "SUCCESS"
    and incremental_pipeline_summary["total_inserted_records"]
    == 4219
    and len(incremental_file_results) == 5
)


# 5. Display refreshed ingestion registry

print("\n" + "=" * 65)
print("REFRESHED INGESTION REGISTRY")
print("=" * 65)

display(
    spark.createDataFrame(
        rerun_ingested_file_registry
    ).orderBy("entity", "file_path")
)


# 6. Display refreshed file classification

print("\n" + "=" * 65)
print("REPEATED EXECUTION FILE CLASSIFICATION")
print("=" * 65)

display(
    spark.createDataFrame(
        rerun_classified_files
    ).orderBy("batch_id", "entity")
)


# 7. Display validation results

display(
    spark.createDataFrame(
        rerun_registry_validation
    )
)


# 8. Calculate consolidated metrics

rerun_total_discovered = len(
    rerun_discovered_files
)

rerun_total_registered = len(
    rerun_ingested_file_registry
)

rerun_total_new = len(
    rerun_new_source_files
)

rerun_total_skipped = len(
    rerun_skipped_source_files
)

rerun_skipped_bytes = sum(
    file["file_size_bytes"]
    for file in rerun_skipped_source_files
)

rerun_skip_rate = (
    rerun_total_skipped
    / rerun_total_discovered * 100
    if rerun_total_discovered > 0
    else 0
)

total_checks = len(rerun_registry_validation)

passed_checks = sum(
    result["status"] == "PASS"
    for result in rerun_registry_validation
)

failed_checks = total_checks - passed_checks


# 9. Final validation summary

print("\n" + "=" * 65)
print("REFRESHED REGISTRY VALIDATION SUMMARY")
print("=" * 65)

print(f"Discovered Source Files: {rerun_total_discovered}")
print(f"Registered Source Files: {rerun_total_registered}")
print(f"Discovered Batches: {len(rerun_batch_counts)}")
print(f"New Source Files: {rerun_total_new}")
print(f"Files Marked for Skipping: {rerun_total_skipped}")
print(f"Skipped Source Bytes: {rerun_skipped_bytes:,}")
print(f"File Skip Rate: {rerun_skip_rate:.2f}%")

print(f"Expected Checks: 12")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0 or total_checks != 12:

    raise RuntimeError(
        "Refreshed ingestion registry validation failed."
    )

print("\nRefreshed Registry Validation: SUCCESS")
print("All existing source files recognized as already ingested.")

REPEATED EXECUTION FILE DISCOVERY
Execution Mode: READ ONLY
[PASS] customers: 2 source files discovered
[PASS] products: 2 source files discovered
[PASS] orders: 2 source files discovered
[PASS] order_items: 2 source files discovered
[PASS] payments: 2 source files discovered

REFRESHING INGESTION REGISTRY
[PASS] customers: 2 ingested files registered
[PASS] products: 2 ingested files registered
[PASS] orders: 2 ingested files registered
[PASS] order_items: 2 ingested files registered
[PASS] payments: 2 ingested files registered

REFRESHED REGISTRY VALIDATION
[PASS] Expected source file count
[PASS] Unique discovered source paths
[PASS] Source file inventory unchanged
[PASS] Expected ingested registry size
[PASS] Unique registered source paths
[PASS] All discovered files registered in Bronze
[PASS] Complete entity-level registry coverage
[PASS] Both source batches discovered
[PASS] Zero new source files
[PASS] All discovered files marked for skipping
[PASS] All files classified as alre

entity,file_path,status,target_table
customers,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,ALREADY_INGESTED,commercelake.bronze.raw_customers
customers,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,ALREADY_INGESTED,commercelake.bronze.raw_customers
order_items,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,ALREADY_INGESTED,commercelake.bronze.raw_order_items
order_items,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,ALREADY_INGESTED,commercelake.bronze.raw_order_items
orders,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,ALREADY_INGESTED,commercelake.bronze.raw_orders
orders,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,ALREADY_INGESTED,commercelake.bronze.raw_orders
payments,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,ALREADY_INGESTED,commercelake.bronze.raw_payments
payments,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,ALREADY_INGESTED,commercelake.bronze.raw_payments
products,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,ALREADY_INGESTED,commercelake.bronze.raw_products
products,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,ALREADY_INGESTED,commercelake.bronze.raw_products



REPEATED EXECUTION FILE CLASSIFICATION


already_ingested,batch_id,entity,file_format,file_name,file_path,file_size_bytes,ingestion_status,planned_action,target_table
true,batch_001,customers,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,171044,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_customers
true,batch_001,order_items,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,391937,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_order_items
true,batch_001,orders,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,1685858,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_orders
true,batch_001,payments,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,2030395,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_payments
true,batch_001,products,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,27458,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_products
true,batch_002,customers,csv,customers.csv,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,34352,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_customers
true,batch_002,order_items,parquet,order_items.parquet,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,84679,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_order_items
true,batch_002,orders,json,orders.json,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,359292,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_orders
true,batch_002,payments,json,payments.json,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,405932,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_payments
true,batch_002,products,csv,products.csv,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,7535,ALREADY_INGESTED,SKIP,commercelake.bronze.raw_products


check_name,status
Expected source file count,PASS
Unique discovered source paths,PASS
Source file inventory unchanged,PASS
Expected ingested registry size,PASS
Unique registered source paths,PASS
All discovered files registered in Bronze,PASS
Complete entity-level registry coverage,PASS
Both source batches discovered,PASS
Zero new source files,PASS
All discovered files marked for skipping,PASS



REFRESHED REGISTRY VALIDATION SUMMARY
Discovered Source Files: 10
Registered Source Files: 10
Discovered Batches: 2
New Source Files: 0
Files Marked for Skipping: 10
Skipped Source Bytes: 5,198,482
File Skip Rate: 100.00%
Expected Checks: 12
Executed Checks: 12
Passed Checks: 12
Failed Checks: 0
Validation Success Rate: 100.00%

Refreshed Registry Validation: SUCCESS
All existing source files recognized as already ingested.


In [0]:
# Cell 23: Generate Repeated Execution Plan

print("=" * 65)
print("REPEATED EXECUTION PLAN GENERATION")
print("=" * 65)


# 1. Create a separate pipeline run context

RERUN_PIPELINE_RUN_ID = str(uuid.uuid4())

RERUN_PIPELINE_START_TIMESTAMP = (
    datetime.now(timezone.utc)
)

RERUN_PIPELINE_STATUS = "PLANNING"

print(f"Original Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Repeated Pipeline Run ID: {RERUN_PIPELINE_RUN_ID}")

print(
    f"Planning Timestamp: "
    f"{RERUN_PIPELINE_START_TIMESTAMP.isoformat()}"
)

print(f"Pipeline Status: {RERUN_PIPELINE_STATUS}")


# 2. Generate execution plan using reusable function

rerun_ingestion_plan = generate_ingestion_plan(
    new_files=rerun_new_source_files,
    pipeline_run_id=RERUN_PIPELINE_RUN_ID
)

print("\n" + "=" * 65)
print("REPEATED INGESTION PLAN")
print("=" * 65)

print(f"Discovered Files: {len(rerun_discovered_files)}")
print(f"Already Ingested: {len(rerun_skipped_source_files)}")
print(f"New Source Files: {len(rerun_new_source_files)}")
print(f"Planned Files: {len(rerun_ingestion_plan)}")

print(f"Execution Plan: {rerun_ingestion_plan}")


# 3. Initialize execution plan validation

rerun_plan_validation = []


def record_rerun_plan_validation(
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    rerun_plan_validation.append({
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("\n" + "=" * 65)
print("REPEATED EXECUTION PLAN VALIDATION")
print("=" * 65)


# Check 1: Independent pipeline run identifier

record_rerun_plan_validation(
    "Unique repeated pipeline run identifier",
    RERUN_PIPELINE_RUN_ID != PIPELINE_RUN_ID
)


# Check 2: Valid UUID format

record_rerun_plan_validation(
    "Valid repeated pipeline UUID",
    str(uuid.UUID(RERUN_PIPELINE_RUN_ID))
    == RERUN_PIPELINE_RUN_ID
)


# Check 3: Valid planning timestamp

record_rerun_plan_validation(
    "Repeated execution timestamp is valid",
    RERUN_PIPELINE_START_TIMESTAMP
    >= IDEMPOTENCY_BASELINE_TIMESTAMP
)


# Check 4: No new source files

record_rerun_plan_validation(
    "No new source files available",
    len(rerun_new_source_files) == 0
)


# Check 5: Empty execution plan

record_rerun_plan_validation(
    "Empty repeated execution plan",
    len(rerun_ingestion_plan) == 0
)


# Check 6: Correct plan size

record_rerun_plan_validation(
    "Execution plan matches new file inventory",
    len(rerun_ingestion_plan)
    == len(rerun_new_source_files)
)


# Check 7: All source files excluded from execution

rerun_planned_paths = {
    file["file_path"]
    for file in rerun_ingestion_plan
}

record_rerun_plan_validation(
    "Previously ingested files excluded from plan",
    rerun_planned_paths.isdisjoint(
        rerun_registered_paths
    )
    and len(rerun_planned_paths) == 0
)


# Check 8: All discovered files accounted for

record_rerun_plan_validation(
    "Complete source file classification",
    len(rerun_ingestion_plan)
    + len(rerun_skipped_source_files)
    == len(rerun_discovered_files)
)


# Check 9: Zero planned source bytes

rerun_planned_bytes = sum(
    file["file_size_bytes"]
    for file in rerun_ingestion_plan
)

record_rerun_plan_validation(
    "Zero source bytes scheduled for ingestion",
    rerun_planned_bytes == 0
)


# Check 10: Original execution plan preserved

record_rerun_plan_validation(
    "Original execution plan preserved",
    len(ingestion_plan) == 5
    and all(
        file["pipeline_run_id"] == PIPELINE_RUN_ID
        and file["batch_id"] == "batch_002"
        for file in ingestion_plan
    )
)


# Check 11: Original execution results preserved

record_rerun_plan_validation(
    "Original successful execution preserved",
    incremental_pipeline_summary["pipeline_run_id"]
    == PIPELINE_RUN_ID
    and incremental_pipeline_summary["pipeline_status"]
    == "SUCCESS"
    and incremental_pipeline_summary["total_inserted_records"]
    == 4219
)


# Check 12: Baseline available for verification

record_rerun_plan_validation(
    "Idempotency baseline available",
    len(idempotency_baseline) == 5
    and sum(
        state["total_records"]
        for state in idempotency_baseline.values()
    ) == 25474
)


# 4. Display validation results

display(
    spark.createDataFrame(
        rerun_plan_validation
    )
)


# 5. Generate validation summary

total_checks = len(rerun_plan_validation)

passed_checks = sum(
    result["status"] == "PASS"
    for result in rerun_plan_validation
)

failed_checks = total_checks - passed_checks

if failed_checks > 0 or total_checks != 12:

    RERUN_PIPELINE_STATUS = "FAILED"

    raise RuntimeError(
        "Repeated execution plan validation failed."
    )


RERUN_PIPELINE_STATUS = "READY"


# 6. Final planning summary

print("\n" + "=" * 65)
print("REPEATED EXECUTION PLAN SUMMARY")
print("=" * 65)

print(f"Original Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Repeated Pipeline Run ID: {RERUN_PIPELINE_RUN_ID}")

print(f"Discovered Files: {len(rerun_discovered_files)}")
print(f"Skipped Files: {len(rerun_skipped_source_files)}")
print(f"Planned Files: {len(rerun_ingestion_plan)}")
print(f"Planned Source Bytes: {rerun_planned_bytes}")
print("Expected New Records: 0")

print(f"Expected Checks: 12")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

print(f"Pipeline Status: {RERUN_PIPELINE_STATUS}")

print("\nRepeated Execution Plan Validation: SUCCESS")
print("Empty execution plan successfully generated.")
print("Pipeline is ready for no-op execution testing.")

REPEATED EXECUTION PLAN GENERATION
Original Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Repeated Pipeline Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Planning Timestamp: 2026-11-26T13:46:31.602555+00:00
Pipeline Status: PLANNING

REPEATED INGESTION PLAN
Discovered Files: 10
Already Ingested: 10
New Source Files: 0
Planned Files: 0
Execution Plan: []

REPEATED EXECUTION PLAN VALIDATION
[PASS] Unique repeated pipeline run identifier
[PASS] Valid repeated pipeline UUID
[PASS] Repeated execution timestamp is valid
[PASS] No new source files available
[PASS] Empty repeated execution plan
[PASS] Execution plan matches new file inventory
[PASS] Previously ingested files excluded from plan
[PASS] Complete source file classification
[PASS] Zero source bytes scheduled for ingestion
[PASS] Original execution plan preserved
[PASS] Original successful execution preserved
[PASS] Idempotency baseline available


check_name,status
Unique repeated pipeline run identifier,PASS
Valid repeated pipeline UUID,PASS
Repeated execution timestamp is valid,PASS
No new source files available,PASS
Empty repeated execution plan,PASS
Execution plan matches new file inventory,PASS
Previously ingested files excluded from plan,PASS
Complete source file classification,PASS
Zero source bytes scheduled for ingestion,PASS
Original execution plan preserved,PASS



REPEATED EXECUTION PLAN SUMMARY
Original Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Repeated Pipeline Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Discovered Files: 10
Skipped Files: 10
Planned Files: 0
Planned Source Bytes: 0
Expected New Records: 0
Expected Checks: 12
Executed Checks: 12
Passed Checks: 12
Failed Checks: 0
Validation Success Rate: 100.00%
Pipeline Status: READY

Repeated Execution Plan Validation: SUCCESS
Empty execution plan successfully generated.
Pipeline is ready for no-op execution testing.


In [0]:
# Cell 24: Execute Repeated Ingestion Pipeline

print("=" * 65)
print("REPEATED INGESTION EXECUTION AUTHORIZATION")
print("=" * 65)


# 1. Validate execution authorization

if RERUN_PIPELINE_STATUS != "READY":

    raise RuntimeError(
        "Repeated pipeline is not ready for execution."
    )

if (
    len(rerun_plan_validation) != 12
    or any(
        result["status"] != "PASS"
        for result in rerun_plan_validation
    )
):

    raise RuntimeError(
        "Repeated execution plan validation failed."
    )

if len(rerun_ingestion_plan) != 0:

    raise RuntimeError(
        "Expected an empty execution plan. "
        "Unexpected files scheduled for ingestion."
    )

if RERUN_PIPELINE_RUN_ID == PIPELINE_RUN_ID:

    raise RuntimeError(
        "Repeated execution must have a separate run ID."
    )


# 2. Verify Bronze state before repeated execution

for entity, config in SOURCE_CONFIG.items():

    current_state = capture_bronze_state(
        target_table=config["target_path"]
    )

    if current_state != idempotency_baseline[entity]:

        raise RuntimeError(
            f"Bronze state changed before rerun: {entity}"
        )

    print(
        f"[PASS] Pre-rerun Bronze state verified: {entity}"
    )


print("\nExecution Authorization: APPROVED")

print(
    f"Repeated Pipeline Run ID: "
    f"{RERUN_PIPELINE_RUN_ID}"
)

print(f"Planned Files: {len(rerun_ingestion_plan)}")
print("Expected Inserted Records: 0")
print("Expected Execution Outcome: NO_NEW_FILES")


# 3. Execute the existing ingestion engine

RERUN_PIPELINE_STATUS = "RUNNING"

try:

    rerun_execution_result = execute_incremental_pipeline(
        ingestion_plan=rerun_ingestion_plan,
        pipeline_run_id=RERUN_PIPELINE_RUN_ID
    )

except Exception:

    RERUN_PIPELINE_STATUS = "FAILED"

    raise


# 4. Extract repeated execution results

rerun_pipeline_summary = (
    rerun_execution_result["pipeline_summary"]
)

rerun_file_results = (
    rerun_execution_result["file_results"]
)

RERUN_PIPELINE_STATUS = (
    rerun_pipeline_summary["pipeline_status"]
)


# 5. Initialize no-op execution validation

rerun_execution_validation = []


def record_rerun_execution_validation(
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    rerun_execution_validation.append({
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("\n" + "=" * 65)
print("NO-OP EXECUTION VALIDATION")
print("=" * 65)


# Check 1: Correct pipeline run ID

record_rerun_execution_validation(
    "Correct repeated pipeline run identifier",
    rerun_pipeline_summary["pipeline_run_id"]
    == RERUN_PIPELINE_RUN_ID
)


# Check 2: Correct pipeline name

record_rerun_execution_validation(
    "Correct pipeline name",
    rerun_pipeline_summary["pipeline_name"]
    == PIPELINE_NAME
)


# Check 3: No new files status

record_rerun_execution_validation(
    "Correct no-op pipeline status",
    RERUN_PIPELINE_STATUS == "NO_NEW_FILES"
)


# Check 4: Zero planned files

record_rerun_execution_validation(
    "Zero planned files",
    rerun_pipeline_summary["planned_files"] == 0
)


# Check 5: Zero processed files

record_rerun_execution_validation(
    "Zero processed files",
    rerun_pipeline_summary["processed_files"] == 0
)


# Check 6: Zero successful file writes

record_rerun_execution_validation(
    "Zero successful file writes",
    rerun_pipeline_summary["successful_files"] == 0
)


# Check 7: Zero runtime skipped files

record_rerun_execution_validation(
    "Zero runtime skipped files",
    rerun_pipeline_summary["skipped_files"] == 0
)


# Check 8: Zero failed files

record_rerun_execution_validation(
    "Zero failed files",
    rerun_pipeline_summary["failed_files"] == 0
)


# Check 9: Zero unprocessed files

record_rerun_execution_validation(
    "Zero unprocessed files",
    rerun_pipeline_summary["unprocessed_files"] == 0
)


# Check 10: Zero uncertain write outcomes

record_rerun_execution_validation(
    "Zero uncertain write outcomes",
    rerun_pipeline_summary["uncertain_write_files"] == 0
)


# Check 11: Zero source records processed

record_rerun_execution_validation(
    "Zero source records processed",
    rerun_pipeline_summary["total_source_records"] == 0
)


# Check 12: Zero inserted records

record_rerun_execution_validation(
    "Zero inserted records",
    rerun_pipeline_summary["total_inserted_records"] == 0
)


# Check 13: Empty file-level execution results

record_rerun_execution_validation(
    "Empty file-level execution results",
    len(rerun_file_results) == 0
)


# Check 14: Valid execution duration

record_rerun_execution_validation(
    "Valid no-op execution duration",
    rerun_pipeline_summary["duration_seconds"] >= 0
)


# Check 15: Original successful execution preserved

record_rerun_execution_validation(
    "Original successful execution preserved",
    incremental_pipeline_summary["pipeline_run_id"]
    == PIPELINE_RUN_ID
    and incremental_pipeline_summary["pipeline_status"]
    == "SUCCESS"
    and incremental_pipeline_summary["total_inserted_records"]
    == 4219
    and len(incremental_file_results) == 5
)


# 6. Display repeated execution summary

display(
    spark.createDataFrame([
        rerun_pipeline_summary
    ])
)

display(
    spark.createDataFrame(
        rerun_execution_validation
    )
)


# 7. Calculate validation metrics

total_checks = len(rerun_execution_validation)

passed_checks = sum(
    result["status"] == "PASS"
    for result in rerun_execution_validation
)

failed_checks = total_checks - passed_checks


# 8. Generate final validation summary

print("\n" + "=" * 65)
print("REPEATED EXECUTION VALIDATION SUMMARY")
print("=" * 65)

print(f"Original Pipeline Run ID: {PIPELINE_RUN_ID}")

print(
    f"Repeated Pipeline Run ID: "
    f"{RERUN_PIPELINE_RUN_ID}"
)

print(
    f"Files Skipped During Planning: "
    f"{len(rerun_skipped_source_files)}"
)

print(
    f"Files Processed During Execution: "
    f"{rerun_pipeline_summary['processed_files']}"
)

print(
    f"Runtime Skipped Files: "
    f"{rerun_pipeline_summary['skipped_files']}"
)

print(
    f"Source Records Processed: "
    f"{rerun_pipeline_summary['total_source_records']}"
)

print(
    f"Inserted Records: "
    f"{rerun_pipeline_summary['total_inserted_records']}"
)

print(
    f"Execution Duration: "
    f"{rerun_pipeline_summary['duration_seconds']} seconds"
)

print(f"Expected Checks: 15")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

print(f"Pipeline Status: {RERUN_PIPELINE_STATUS}")


if failed_checks > 0 or total_checks != 15:

    raise RuntimeError(
        "Repeated execution validation failed."
    )


print("\nRepeated Execution Validation: SUCCESS")
print("No-op ingestion execution successfully verified.")

REPEATED INGESTION EXECUTION AUTHORIZATION
[PASS] Pre-rerun Bronze state verified: customers
[PASS] Pre-rerun Bronze state verified: products
[PASS] Pre-rerun Bronze state verified: orders
[PASS] Pre-rerun Bronze state verified: order_items
[PASS] Pre-rerun Bronze state verified: payments

Execution Authorization: APPROVED
Repeated Pipeline Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Planned Files: 0
Expected Inserted Records: 0
Expected Execution Outcome: NO_NEW_FILES
INCREMENTAL BRONZE PIPELINE EXECUTION
Pipeline Name: commercelake_bronze_incremental_ingestion
Pipeline Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Execution Started: 2026-11-26T13:48:07.104073+00:00
Planned Files: 0
Pipeline Status: RUNNING

PIPELINE EXECUTION SUMMARY
Pipeline Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Planned Files: 0
Processed Files: 0
Successful Files: 0
Skipped Files: 0
Failed Files: 0
Unprocessed Files: 0
Uncertain Write Files: 0
Known Source Records: 0
Verified Inserted Records: 0
Execution Du

duration_seconds,execution_completed,execution_started,failed_files,pipeline_name,pipeline_run_id,pipeline_status,planned_files,processed_files,skipped_files,successful_files,total_inserted_records,total_source_records,uncertain_write_files,unprocessed_files
1.0E-4,2026-11-26T13:48:07.104214+00:00,2026-11-26T13:48:07.104073+00:00,0,commercelake_bronze_incremental_ingestion,9d378a30-1e55-4542-8874-1bea1f834303,NO_NEW_FILES,0,0,0,0,0,0,0,0


check_name,status
Correct repeated pipeline run identifier,PASS
Correct pipeline name,PASS
Correct no-op pipeline status,PASS
Zero planned files,PASS
Zero processed files,PASS
Zero successful file writes,PASS
Zero runtime skipped files,PASS
Zero failed files,PASS
Zero unprocessed files,PASS
Zero uncertain write outcomes,PASS



REPEATED EXECUTION VALIDATION SUMMARY
Original Pipeline Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
Repeated Pipeline Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Files Skipped During Planning: 10
Files Processed During Execution: 0
Runtime Skipped Files: 0
Source Records Processed: 0
Inserted Records: 0
Execution Duration: 0.0001 seconds
Expected Checks: 15
Executed Checks: 15
Passed Checks: 15
Failed Checks: 0
Validation Success Rate: 100.00%
Pipeline Status: NO_NEW_FILES

Repeated Execution Validation: SUCCESS
No-op ingestion execution successfully verified.


In [0]:
# Cell 25: Test Runtime Duplicate Ingestion Guard

print("=" * 65)
print("RUNTIME DUPLICATE INGESTION GUARD TEST")
print("=" * 65)


# 1. Create an independent test execution context

GUARD_TEST_RUN_ID = str(uuid.uuid4())

GUARD_TEST_START_TIMESTAMP = datetime.now(timezone.utc)

print(f"Guard Test Run ID: {GUARD_TEST_RUN_ID}")
print(f"Test Timestamp: {GUARD_TEST_START_TIMESTAMP.isoformat()}")
print("Test Scenario: Simulated stale ingestion plan")
print("Expected Outcome: All files SKIPPED")


# 2. Verify previous no-op execution

if RERUN_PIPELINE_STATUS != "NO_NEW_FILES":

    raise RuntimeError(
        "Repeated execution must succeed before guard testing."
    )

if (
    len(rerun_execution_validation) != 15
    or any(
        result["status"] != "PASS"
        for result in rerun_execution_validation
    )
):

    raise RuntimeError(
        "No-op execution validation is incomplete."
    )


# 3. Verify current Bronze state

for entity, config in SOURCE_CONFIG.items():

    current_state = capture_bronze_state(
        target_table=config["target_path"]
    )

    if current_state != idempotency_baseline[entity]:

        raise RuntimeError(
            f"Bronze state changed before guard test: {entity}"
        )

    print(f"[PASS] Bronze state verified: {entity}")


# 4. Construct a simulated stale execution plan

guard_test_candidates = [
    {
        **file,
        "already_ingested": False,
        "ingestion_status": "NEW_FILE",
        "planned_action": "INGEST"
    }
    for file in rerun_skipped_source_files
]

guard_test_plan = generate_ingestion_plan(
    new_files=guard_test_candidates,
    pipeline_run_id=GUARD_TEST_RUN_ID
)

if len(guard_test_plan) != 10:

    raise RuntimeError(
        "Expected exactly 10 previously ingested files."
    )


# 5. Safety verification against persisted Bronze metadata

for file_info in guard_test_plan:

    if not check_file_already_ingested(
        target_table=file_info["target_table"],
        source_path=file_info["file_path"]
    ):

        raise RuntimeError(
            "Unsafe guard test: file is not registered "
            "as previously ingested."
        )


print("\nStale Plan Files: 10")
print("Previously Ingested Files Confirmed: 10")
print("Guard Test Authorization: APPROVED")


# 6. Initialize validation results

guard_test_results = []
guard_validation_results = []


def record_guard_validation(
    entity: str,
    batch_id: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    guard_validation_results.append({
        "entity": entity,
        "batch_id": batch_id,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


# 7. Submit previously ingested files directly

guard_test_timer = perf_counter()

for file_info in guard_test_plan:

    entity = file_info["entity"]
    batch_id = file_info["batch_id"]

    print("\n" + "-" * 65)
    print(f"TESTING: {entity.upper()} | {batch_id}")
    print("-" * 65)

    result = ingest_source_file(file_info)

    guard_test_results.append(result)


    # Check 1: Runtime guard skipped the file

    record_guard_validation(
        entity,
        batch_id,
        "Previously ingested file skipped",
        result["status"] == "SKIPPED"
    )


    # Check 2: No records inserted

    record_guard_validation(
        entity,
        batch_id,
        "Zero duplicate records inserted",
        result["inserted_records"] == 0
    )


    # Check 3: Source data reading avoided

    record_guard_validation(
        entity,
        batch_id,
        "Source record processing avoided",
        result["source_records"] is None
    )


    # Check 4: Correct source file identity

    record_guard_validation(
        entity,
        batch_id,
        "Correct source file identity",
        result["source_path"] == file_info["file_path"]
    )


    # Check 5: Correct target table

    record_guard_validation(
        entity,
        batch_id,
        "Correct target table",
        result["target_table"] == file_info["target_table"]
    )


    # Check 6: Independent guard test run identifier

    record_guard_validation(
        entity,
        batch_id,
        "Correct guard test run identifier",
        result["pipeline_run_id"] == GUARD_TEST_RUN_ID
    )


    # Stop immediately if the guard behaves unexpectedly

    if (
        result["status"] != "SKIPPED"
        or result["inserted_records"] != 0
    ):

        raise RuntimeError(
            "Runtime duplicate guard failed. "
            "Inspect committed Delta state before continuing."
        )

    print(f"File Status: {result['status']}")
    print(f"Inserted Records: {result['inserted_records']}")


guard_test_duration = round(
    perf_counter() - guard_test_timer,
    4
)


# 8. Consolidated runtime guard validation

print("\n" + "=" * 65)
print("CONSOLIDATED DUPLICATE GUARD VALIDATION")
print("=" * 65)

guard_test_skipped_files = sum(
    result["status"] == "SKIPPED"
    for result in guard_test_results
)

guard_test_inserted_records = sum(
    result["inserted_records"]
    for result in guard_test_results
    if result["inserted_records"] is not None
)

guard_test_failed_files = sum(
    result["status"] == "FAILED"
    for result in guard_test_results
)


# Consolidated check 1

record_guard_validation(
    "pipeline",
    "ALL",
    "All stale-plan files tested",
    len(guard_test_results) == 10
)


# Consolidated check 2

record_guard_validation(
    "pipeline",
    "ALL",
    "All previously ingested files blocked",
    guard_test_skipped_files == 10
)


# Consolidated check 3

record_guard_validation(
    "pipeline",
    "ALL",
    "Zero duplicate records inserted",
    guard_test_inserted_records == 0
)


# Consolidated check 4

record_guard_validation(
    "pipeline",
    "ALL",
    "Zero failed file operations",
    guard_test_failed_files == 0
)


# Consolidated check 5

record_guard_validation(
    "pipeline",
    "ALL",
    "Original ingestion results preserved",
    incremental_pipeline_summary["pipeline_run_id"]
    == PIPELINE_RUN_ID
    and incremental_pipeline_summary["total_inserted_records"]
    == 4219
)


# Consolidated check 6

record_guard_validation(
    "pipeline",
    "ALL",
    "Original no-op execution results preserved",
    rerun_pipeline_summary["pipeline_run_id"]
    == RERUN_PIPELINE_RUN_ID
    and rerun_pipeline_summary["pipeline_status"]
    == "NO_NEW_FILES"
)


# 9. Display runtime guard results
# Exclude source_records because every value is expected
# to be None, which prevents automatic Spark type inference.

guard_results_display = [
    {
        "entity": result["entity"],
        "batch_id": result["batch_id"],
        "source_path": result["source_path"],
        "target_table": result["target_table"],
        "pipeline_run_id": result["pipeline_run_id"],
        "inserted_records": result["inserted_records"],
        "duration_seconds": result["duration_seconds"],
        "status": result["status"]
    }
    for result in guard_test_results
]

display(
    spark.createDataFrame(
        guard_results_display
    ).orderBy("batch_id", "entity")
)

display(
    spark.createDataFrame(
        guard_validation_results
    )
)


# 10. Generate final validation summary

total_checks = len(guard_validation_results)

passed_checks = sum(
    result["status"] == "PASS"
    for result in guard_validation_results
)

failed_checks = total_checks - passed_checks

expected_checks = len(guard_test_plan) * 6 + 6

guard_skip_rate = (
    guard_test_skipped_files
    / len(guard_test_plan) * 100
)

print("\n" + "=" * 65)
print("RUNTIME DUPLICATE GUARD VALIDATION SUMMARY")
print("=" * 65)

print(f"Guard Test Run ID: {GUARD_TEST_RUN_ID}")
print(f"Previously Ingested Files Tested: {len(guard_test_plan)}")
print(f"Files Blocked by Runtime Guard: {guard_test_skipped_files}")
print(f"Failed Files: {guard_test_failed_files}")
print(f"Duplicate Records Inserted: {guard_test_inserted_records}")
print(f"Runtime Guard Skip Rate: {guard_skip_rate:.2f}%")
print(f"Guard Test Duration: {guard_test_duration} seconds")

print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)

if failed_checks > 0 or total_checks != expected_checks:

    raise RuntimeError(
        "Runtime duplicate guard validation failed."
    )

print("\nRuntime Duplicate Guard Validation: SUCCESS")
print("All previously ingested files successfully blocked.")

RUNTIME DUPLICATE INGESTION GUARD TEST
Guard Test Run ID: 76734e8a-afe7-4270-8e8c-6de62ea26d84
Test Timestamp: 2026-11-26T13:53:34.040433+00:00
Test Scenario: Simulated stale ingestion plan
Expected Outcome: All files SKIPPED
[PASS] Bronze state verified: customers
[PASS] Bronze state verified: products
[PASS] Bronze state verified: orders
[PASS] Bronze state verified: order_items
[PASS] Bronze state verified: payments

Stale Plan Files: 10
Previously Ingested Files Confirmed: 10
Guard Test Authorization: APPROVED

-----------------------------------------------------------------
TESTING: CUSTOMERS | batch_001
-----------------------------------------------------------------

PROCESSING FILE 1
Entity: customers
Batch: batch_001
Source: /Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv
Target: commercelake.bronze.raw_customers
Format: csv
[SKIPPED] File already ingested.
[PASS] Previously ingested file skipped
[PASS] Zero duplicate records inserted
[PASS] Sourc

batch_id,duration_seconds,entity,inserted_records,pipeline_run_id,source_path,status,target_table
batch_001,0.6633,customers,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,SKIPPED,commercelake.bronze.raw_customers
batch_001,0.6173,order_items,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,SKIPPED,commercelake.bronze.raw_order_items
batch_001,0.6927,orders,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,SKIPPED,commercelake.bronze.raw_orders
batch_001,0.6523,payments,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,SKIPPED,commercelake.bronze.raw_payments
batch_001,0.6891,products,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,SKIPPED,commercelake.bronze.raw_products
batch_002,0.6763,customers,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,SKIPPED,commercelake.bronze.raw_customers
batch_002,0.6777,order_items,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,SKIPPED,commercelake.bronze.raw_order_items
batch_002,0.6764,orders,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,SKIPPED,commercelake.bronze.raw_orders
batch_002,0.6902,payments,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,SKIPPED,commercelake.bronze.raw_payments
batch_002,0.7497,products,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,SKIPPED,commercelake.bronze.raw_products


batch_id,check_name,entity,status
batch_001,Previously ingested file skipped,customers,PASS
batch_001,Zero duplicate records inserted,customers,PASS
batch_001,Source record processing avoided,customers,PASS
batch_001,Correct source file identity,customers,PASS
batch_001,Correct target table,customers,PASS
batch_001,Correct guard test run identifier,customers,PASS
batch_001,Previously ingested file skipped,products,PASS
batch_001,Zero duplicate records inserted,products,PASS
batch_001,Source record processing avoided,products,PASS
batch_001,Correct source file identity,products,PASS



RUNTIME DUPLICATE GUARD VALIDATION SUMMARY
Guard Test Run ID: 76734e8a-afe7-4270-8e8c-6de62ea26d84
Previously Ingested Files Tested: 10
Files Blocked by Runtime Guard: 10
Failed Files: 0
Duplicate Records Inserted: 0
Runtime Guard Skip Rate: 100.00%
Guard Test Duration: 6.7867 seconds
Expected Checks: 66
Executed Checks: 66
Passed Checks: 66
Failed Checks: 0
Validation Success Rate: 100.00%

Runtime Duplicate Guard Validation: SUCCESS
All previously ingested files successfully blocked.


In [0]:
# Cell 26: Final Idempotency Verification

print("=" * 65)
print("FINAL IDEMPOTENCY VERIFICATION")
print("=" * 65)

print("Execution Mode: READ ONLY")

print(f"Original Run ID: {PIPELINE_RUN_ID}")
print(f"No-Op Run ID: {RERUN_PIPELINE_RUN_ID}")
print(f"Guard Test Run ID: {GUARD_TEST_RUN_ID}")


# 1. Verify prerequisite test results

if RERUN_PIPELINE_STATUS != "NO_NEW_FILES":
    raise RuntimeError(
        "No-op execution was not successful."
    )

if (
    len(guard_validation_results) != 66
    or any(
        result["status"] != "PASS"
        for result in guard_validation_results
    )
):
    raise RuntimeError(
        "Runtime duplicate guard validation incomplete."
    )

if len(idempotency_baseline) != 5:
    raise RuntimeError(
        "Idempotency baseline is incomplete."
    )


# 2. Initialize final validation containers

idempotency_final_state = {}

idempotency_validation_results = []

idempotency_comparison_results = []


def record_idempotency_validation(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    idempotency_validation_results.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


# 3. Capture and compare final Bronze state

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    print("\n" + "-" * 65)
    print(f"VERIFYING ENTITY: {entity.upper()}")
    print("-" * 65)

    baseline = idempotency_baseline[entity]

    current_state = capture_bronze_state(
        target_table=target_table
    )

    idempotency_final_state[entity] = current_state

    bronze_df = spark.table(target_table)


    # Check 1: Total record count unchanged

    record_idempotency_validation(
        entity,
        "Total record count unchanged",
        current_state["total_records"]
        == baseline["total_records"]
    )


    # Check 2: Batch distribution unchanged

    record_idempotency_validation(
        entity,
        "Batch record distribution unchanged",
        current_state["batch_counts"]
        == baseline["batch_counts"]
    )


    # Check 3: Source file count unchanged

    record_idempotency_validation(
        entity,
        "Distinct source file count unchanged",
        current_state["distinct_source_files"]
        == baseline["distinct_source_files"]
    )


    # Check 4: Pipeline run count unchanged

    record_idempotency_validation(
        entity,
        "Distinct pipeline run count unchanged",
        current_state["distinct_pipeline_runs"]
        == baseline["distinct_pipeline_runs"]
    )


    # Check 5: Delta version unchanged

    record_idempotency_validation(
        entity,
        "Delta transaction version unchanged",
        current_state["delta_version"]
        == baseline["delta_version"]
    )


    # Check 6: Physical data file count unchanged

    record_idempotency_validation(
        entity,
        "Physical Delta data file count unchanged",
        current_state["num_files"]
        == baseline["num_files"]
    )


    # Check 7: Storage size unchanged

    record_idempotency_validation(
        entity,
        "Delta data storage size unchanged",
        current_state["size_bytes"]
        == baseline["size_bytes"]
    )


    # Check 8: No records from no-op execution

    rerun_records = (
        bronze_df
        .filter(
            F.col("_pipeline_run_id")
            == RERUN_PIPELINE_RUN_ID
        )
        .count()
    )

    record_idempotency_validation(
        entity,
        "No records inserted by no-op execution",
        rerun_records == 0
    )


    # Check 9: No records from guard test

    guard_records = (
        bronze_df
        .filter(
            F.col("_pipeline_run_id")
            == GUARD_TEST_RUN_ID
        )
        .count()
    )

    record_idempotency_validation(
        entity,
        "No records inserted by runtime guard test",
        guard_records == 0
    )


    # Check 10: No new Delta transactions

    new_transactions = (
        spark.sql(
            f"DESCRIBE HISTORY {target_table}"
        )
        .filter(
            F.col("version")
            > baseline["delta_version"]
        )
        .count()
    )

    record_idempotency_validation(
        entity,
        "No new Delta transactions committed",
        new_transactions == 0
    )


    # Check 11: Exact record-level comparison
    # against the pre-rerun Delta snapshot

    baseline_version = baseline["delta_version"]

    baseline_df = spark.sql(
        f"""
        SELECT *
        FROM {target_table}
        VERSION AS OF {baseline_version}
        """
    )

    current_df = bronze_df.select(
        *baseline_df.columns
    )

    removed_or_modified_records = (
        baseline_df
        .exceptAll(current_df)
        .count()
    )

    added_or_modified_records = (
        current_df
        .exceptAll(baseline_df)
        .count()
    )

    record_differences = (
        removed_or_modified_records
        + added_or_modified_records
    )

    record_idempotency_validation(
        entity,
        "Exact persisted record contents unchanged",
        record_differences == 0
    )


    # Check 12: Complete table state unchanged

    record_idempotency_validation(
        entity,
        "Complete Bronze state unchanged",
        current_state == baseline
    )


    # Store entity-level comparison

    idempotency_comparison_results.append({
        "entity": entity,
        "target_table": target_table,
        "baseline_records": baseline["total_records"],
        "current_records": current_state["total_records"],
        "record_count_difference": (
            current_state["total_records"]
            - baseline["total_records"]
        ),
        "baseline_version": baseline["delta_version"],
        "current_version": current_state["delta_version"],
        "new_delta_transactions": new_transactions,
        "baseline_data_files": baseline["num_files"],
        "current_data_files": current_state["num_files"],
        "baseline_storage_bytes": baseline["size_bytes"],
        "current_storage_bytes": current_state["size_bytes"],
        "storage_difference_bytes": (
            current_state["size_bytes"]
            - baseline["size_bytes"]
        ),
        "rerun_inserted_records": rerun_records,
        "guard_inserted_records": guard_records,
        "record_differences": record_differences
    })

    print(f"Baseline Records: {baseline['total_records']:,}")

    print(
        f"Current Records: "
        f"{current_state['total_records']:,}"
    )

    print(f"Record Differences: {record_differences}")
    print(f"New Delta Transactions: {new_transactions}")

    print(
        f"Delta Version: "
        f"{baseline['delta_version']} -> "
        f"{current_state['delta_version']}"
    )


# 4. Calculate consolidated metrics

final_total_records = sum(
    state["total_records"]
    for state in idempotency_final_state.values()
)

final_total_source_files = sum(
    state["distinct_source_files"]
    for state in idempotency_final_state.values()
)

final_total_delta_files = sum(
    state["num_files"]
    for state in idempotency_final_state.values()
)

final_total_storage_bytes = sum(
    state["size_bytes"]
    for state in idempotency_final_state.values()
)

final_total_record_differences = sum(
    result["record_differences"]
    for result in idempotency_comparison_results
)

final_total_new_transactions = sum(
    result["new_delta_transactions"]
    for result in idempotency_comparison_results
)

final_total_rerun_records = sum(
    result["rerun_inserted_records"]
    for result in idempotency_comparison_results
)

final_total_guard_records = sum(
    result["guard_inserted_records"]
    for result in idempotency_comparison_results
)


# 5. Consolidated pipeline validation

print("\n" + "=" * 65)
print("CONSOLIDATED IDEMPOTENCY VALIDATION")
print("=" * 65)


record_idempotency_validation(
    "pipeline",
    "Complete Bronze table verification coverage",
    len(idempotency_final_state) == 5
)

record_idempotency_validation(
    "pipeline",
    "Total Bronze records unchanged",
    final_total_records == baseline_total_records == 25474
)

record_idempotency_validation(
    "pipeline",
    "Total ingested source files unchanged",
    final_total_source_files
    == baseline_total_source_files == 10
)

record_idempotency_validation(
    "pipeline",
    "Total physical Delta files unchanged",
    final_total_delta_files
    == baseline_total_delta_files
)

record_idempotency_validation(
    "pipeline",
    "Total Delta storage size unchanged",
    final_total_storage_bytes
    == baseline_total_storage_bytes
)

record_idempotency_validation(
    "pipeline",
    "Zero new Delta transactions",
    final_total_new_transactions == 0
)

record_idempotency_validation(
    "pipeline",
    "Zero persisted record differences",
    final_total_record_differences == 0
)

record_idempotency_validation(
    "pipeline",
    "Zero records inserted by repeated execution tests",
    final_total_rerun_records == 0
    and final_total_guard_records == 0
)


# 6. Display comparison and validation results

display(
    spark.createDataFrame(
        idempotency_comparison_results
    ).orderBy("entity")
)

display(
    spark.createDataFrame(
        idempotency_validation_results
    )
)


# 7. Calculate validation results

expected_checks = len(SOURCE_CONFIG) * 12 + 8

total_checks = len(
    idempotency_validation_results
)

passed_checks = sum(
    result["status"] == "PASS"
    for result in idempotency_validation_results
)

failed_checks = total_checks - passed_checks


# 8. Final idempotency summary

print("\n" + "=" * 65)
print("FINAL IDEMPOTENCY VALIDATION SUMMARY")
print("=" * 65)

print(f"Bronze Tables Verified: {len(idempotency_final_state)}")

print(f"Baseline Records: {baseline_total_records:,}")
print(f"Current Records: {final_total_records:,}")

print(
    f"Record Count Difference: "
    f"{final_total_records - baseline_total_records}"
)

print(
    f"Persisted Record Differences: "
    f"{final_total_record_differences}"
)

print(f"Total Source Files: {final_total_source_files}")
print(f"Total Delta Data Files: {final_total_delta_files}")

print(f"Baseline Storage Bytes: {baseline_total_storage_bytes:,}")
print(f"Current Storage Bytes: {final_total_storage_bytes:,}")

print(
    f"Storage Difference Bytes: "
    f"{final_total_storage_bytes - baseline_total_storage_bytes}"
)

print(f"New Delta Transactions: {final_total_new_transactions}")

print(f"No-Op Inserted Records: {final_total_rerun_records}")
print(f"Guard Test Inserted Records: {final_total_guard_records}")

print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)


if failed_checks > 0 or total_checks != expected_checks:

    IDEMPOTENCY_STATUS = "FAILED"

    raise RuntimeError(
        "Final idempotency verification failed."
    )


IDEMPOTENCY_STATUS = "SUCCESS"

print("\nFinal Idempotency Verification: SUCCESS")
print("All Bronze tables remain unchanged.")
print("Planning-level duplicate prevention: VERIFIED")
print("Runtime-level duplicate prevention: VERIFIED")
print("Repeated execution idempotency: VERIFIED")

FINAL IDEMPOTENCY VERIFICATION
Execution Mode: READ ONLY
Original Run ID: 5cd36830-4695-40d1-ab22-17d8db1df9f9
No-Op Run ID: 9d378a30-1e55-4542-8874-1bea1f834303
Guard Test Run ID: 76734e8a-afe7-4270-8e8c-6de62ea26d84

-----------------------------------------------------------------
VERIFYING ENTITY: CUSTOMERS
-----------------------------------------------------------------
[PASS] Total record count unchanged
[PASS] Batch record distribution unchanged
[PASS] Distinct source file count unchanged
[PASS] Distinct pipeline run count unchanged
[PASS] Delta transaction version unchanged
[PASS] Physical Delta data file count unchanged
[PASS] Delta data storage size unchanged
[PASS] No records inserted by no-op execution
[PASS] No records inserted by runtime guard test
[PASS] No new Delta transactions committed
[PASS] Exact persisted record contents unchanged
[PASS] Complete Bronze state unchanged
Baseline Records: 1,200
Current Records: 1,200
Record Differences: 0
New Delta Transactions: 0


baseline_data_files,baseline_records,baseline_storage_bytes,baseline_version,current_data_files,current_records,current_storage_bytes,current_version,entity,guard_inserted_records,new_delta_transactions,record_count_difference,record_differences,rerun_inserted_records,storage_difference_bytes,target_table
2,1200,101167,2,2,1200,101167,2,customers,0,0,0,0,0,0,commercelake.bronze.raw_customers
2,12024,696716,2,2,12024,696716,2,order_items,0,0,0,0,0,0,commercelake.bronze.raw_order_items
2,6000,366231,2,2,6000,366231,2,orders,0,0,0,0,0,0,commercelake.bronze.raw_orders
2,6000,414890,2,2,6000,414890,2,payments,0,0,0,0,0,0,commercelake.bronze.raw_payments
2,250,28360,2,2,250,28360,2,products,0,0,0,0,0,0,commercelake.bronze.raw_products


check_name,entity,status
Total record count unchanged,customers,PASS
Batch record distribution unchanged,customers,PASS
Distinct source file count unchanged,customers,PASS
Distinct pipeline run count unchanged,customers,PASS
Delta transaction version unchanged,customers,PASS
Physical Delta data file count unchanged,customers,PASS
Delta data storage size unchanged,customers,PASS
No records inserted by no-op execution,customers,PASS
No records inserted by runtime guard test,customers,PASS
No new Delta transactions committed,customers,PASS



FINAL IDEMPOTENCY VALIDATION SUMMARY
Bronze Tables Verified: 5
Baseline Records: 25,474
Current Records: 25,474
Record Count Difference: 0
Persisted Record Differences: 0
Total Source Files: 10
Total Delta Data Files: 10
Baseline Storage Bytes: 1,607,364
Current Storage Bytes: 1,607,364
Storage Difference Bytes: 0
New Delta Transactions: 0
No-Op Inserted Records: 0
Guard Test Inserted Records: 0
Expected Checks: 68
Executed Checks: 68
Passed Checks: 68
Failed Checks: 0
Validation Success Rate: 100.00%

Final Idempotency Verification: SUCCESS
All Bronze tables remain unchanged.
Planning-level duplicate prevention: VERIFIED
Runtime-level duplicate prevention: VERIFIED
Repeated execution idempotency: VERIFIED


In [0]:
# Cell 27: Cumulative Bronze Record Validation

print("=" * 65)
print("CUMULATIVE BRONZE RECORD VALIDATION")
print("=" * 65)

print("Execution Mode: READ ONLY")


# 1. Verify prerequisite idempotency testing

if IDEMPOTENCY_STATUS != "SUCCESS":

    raise RuntimeError(
        "Idempotency verification must succeed first."
    )


# 2. Initialize validation containers

cumulative_record_results = []

cumulative_record_validation = []


def record_cumulative_validation(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    cumulative_record_validation.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


# 3. Validate cumulative records for each Bronze table

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    print("\n" + "-" * 65)
    print(f"VALIDATING ENTITY: {entity.upper()}")
    print("-" * 65)

    bronze_df = spark.table(target_table)

    expected_historical = (
        EXPECTED_HISTORICAL_COUNTS[entity]
    )

    expected_incremental = (
        EXPECTED_INCREMENTAL_COUNTS[entity]
    )

    expected_total = (
        expected_historical
        + expected_incremental
    )


    # Calculate actual cumulative record count

    actual_total = bronze_df.count()


    # Calculate batch-level record distribution

    batch_distribution = {
        row["_batch_id"]: row["count"]
        for row in (
            bronze_df
            .groupBy("_batch_id")
            .count()
            .collect()
        )
    }

    actual_historical = batch_distribution.get(
        "batch_001",
        0
    )

    actual_incremental = batch_distribution.get(
        "batch_002",
        0
    )


    # Identify unexpected batch records

    unexpected_batch_records = (
        bronze_df
        .filter(
            F.col("_batch_id").isNull()
            | ~F.col("_batch_id").isin(
                "batch_001",
                "batch_002"
            )
        )
        .count()
    )


    # Check 1: Historical records

    record_cumulative_validation(
        entity,
        "Expected historical record count",
        actual_historical == expected_historical
    )


    # Check 2: Incremental records

    record_cumulative_validation(
        entity,
        "Expected incremental record count",
        actual_incremental == expected_incremental
    )


    # Check 3: Cumulative records

    record_cumulative_validation(
        entity,
        "Expected cumulative record count",
        actual_total == expected_total
    )


    # Check 4: Batch reconciliation

    record_cumulative_validation(
        entity,
        "Historical and incremental records reconcile",
        actual_historical + actual_incremental
        == actual_total
    )


    # Check 5: Expected batch coverage

    record_cumulative_validation(
        entity,
        "Exactly two expected batches present",
        set(batch_distribution) == {
            "batch_001",
            "batch_002"
        }
    )


    # Check 6: No unexpected batch records

    record_cumulative_validation(
        entity,
        "Zero unexpected batch records",
        unexpected_batch_records == 0
    )


    # Check 7: Baseline reconciliation

    record_cumulative_validation(
        entity,
        "Cumulative records match idempotency baseline",
        actual_total
        == idempotency_baseline[entity]["total_records"]
    )


    # Calculate incremental growth

    growth_percentage = (
        actual_incremental
        / actual_historical * 100
        if actual_historical > 0
        else 0
    )


    # Calculate incremental contribution

    incremental_contribution = (
        actual_incremental
        / actual_total * 100
        if actual_total > 0
        else 0
    )


    # Store entity-level metrics

    cumulative_record_results.append({
        "entity": entity,
        "target_table": target_table,
        "historical_records": actual_historical,
        "incremental_records": actual_incremental,
        "expected_total_records": expected_total,
        "actual_total_records": actual_total,
        "record_difference": (
            actual_total - expected_total
        ),
        "growth_percentage": round(
            growth_percentage,
            2
        ),
        "incremental_contribution_percentage": round(
            incremental_contribution,
            2
        ),
        "unexpected_batch_records": (
            unexpected_batch_records
        )
    })


    print(f"Historical Records: {actual_historical:,}")
    print(f"Incremental Records: {actual_incremental:,}")
    print(f"Cumulative Records: {actual_total:,}")
    print(f"Dataset Growth: {growth_percentage:.2f}%")


# 4. Calculate consolidated Bronze metrics

cumulative_historical_records = sum(
    result["historical_records"]
    for result in cumulative_record_results
)

cumulative_incremental_records = sum(
    result["incremental_records"]
    for result in cumulative_record_results
)

cumulative_total_records = sum(
    result["actual_total_records"]
    for result in cumulative_record_results
)

cumulative_expected_records = sum(
    result["expected_total_records"]
    for result in cumulative_record_results
)

cumulative_unexpected_batch_records = sum(
    result["unexpected_batch_records"]
    for result in cumulative_record_results
)

cumulative_growth_percentage = (
    cumulative_incremental_records
    / cumulative_historical_records * 100
)

cumulative_incremental_contribution = (
    cumulative_incremental_records
    / cumulative_total_records * 100
)


# 5. Consolidated pipeline validation

print("\n" + "=" * 65)
print("CONSOLIDATED BRONZE RECORD VALIDATION")
print("=" * 65)


record_cumulative_validation(
    "pipeline",
    "Complete entity validation coverage",
    len(cumulative_record_results) == 5
)

record_cumulative_validation(
    "pipeline",
    "Total historical records reconciled",
    cumulative_historical_records == 21255
)

record_cumulative_validation(
    "pipeline",
    "Total incremental records reconciled",
    cumulative_incremental_records == 4219
)

record_cumulative_validation(
    "pipeline",
    "Total cumulative records reconciled",
    cumulative_total_records
    == cumulative_expected_records
    == 25474
)

record_cumulative_validation(
    "pipeline",
    "Complete historical and incremental reconciliation",
    cumulative_historical_records
    + cumulative_incremental_records
    == cumulative_total_records
)

record_cumulative_validation(
    "pipeline",
    "Zero unexpected batch records",
    cumulative_unexpected_batch_records == 0
)

record_cumulative_validation(
    "pipeline",
    "Incremental records match successful ingestion",
    cumulative_incremental_records
    == incremental_pipeline_summary[
        "total_inserted_records"
    ]
)


# 6. Display cumulative record metrics

display(
    spark.createDataFrame(
        cumulative_record_results
    ).orderBy("entity")
)

display(
    spark.createDataFrame(
        cumulative_record_validation
    )
)


# 7. Calculate validation metrics

expected_checks = len(SOURCE_CONFIG) * 7 + 7

total_checks = len(
    cumulative_record_validation
)

passed_checks = sum(
    result["status"] == "PASS"
    for result in cumulative_record_validation
)

failed_checks = total_checks - passed_checks


# 8. Generate final validation summary

print("\n" + "=" * 65)
print("CUMULATIVE BRONZE RECORD VALIDATION SUMMARY")
print("=" * 65)

print(
    f"Bronze Tables Validated: "
    f"{len(cumulative_record_results)}"
)

print("Total Batches: 2")

print(
    f"Historical Records: "
    f"{cumulative_historical_records:,}"
)

print(
    f"Incremental Records: "
    f"{cumulative_incremental_records:,}"
)

print(
    f"Expected Cumulative Records: "
    f"{cumulative_expected_records:,}"
)

print(
    f"Actual Cumulative Records: "
    f"{cumulative_total_records:,}"
)

print(
    f"Record Difference: "
    f"{cumulative_total_records - cumulative_expected_records}"
)

print(
    f"Dataset Growth: "
    f"{cumulative_growth_percentage:.2f}%"
)

print(
    f"Incremental Contribution: "
    f"{cumulative_incremental_contribution:.2f}%"
)

print(
    f"Unexpected Batch Records: "
    f"{cumulative_unexpected_batch_records}"
)

print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)


if failed_checks > 0 or total_checks != expected_checks:

    CUMULATIVE_RECORD_STATUS = "FAILED"

    raise RuntimeError(
        "Cumulative Bronze record validation failed."
    )


CUMULATIVE_RECORD_STATUS = "SUCCESS"

print("\nCumulative Bronze Record Validation: SUCCESS")
print("All historical and incremental records reconciled.")

CUMULATIVE BRONZE RECORD VALIDATION
Execution Mode: READ ONLY

-----------------------------------------------------------------
VALIDATING ENTITY: CUSTOMERS
-----------------------------------------------------------------
[PASS] Expected historical record count
[PASS] Expected incremental record count
[PASS] Expected cumulative record count
[PASS] Historical and incremental records reconcile
[PASS] Exactly two expected batches present
[PASS] Zero unexpected batch records
[PASS] Cumulative records match idempotency baseline
Historical Records: 1,000
Incremental Records: 200
Cumulative Records: 1,200
Dataset Growth: 20.00%

-----------------------------------------------------------------
VALIDATING ENTITY: PRODUCTS
-----------------------------------------------------------------
[PASS] Expected historical record count
[PASS] Expected incremental record count
[PASS] Expected cumulative record count
[PASS] Historical and incremental records reconcile
[PASS] Exactly two expected batches

actual_total_records,entity,expected_total_records,growth_percentage,historical_records,incremental_contribution_percentage,incremental_records,record_difference,target_table,unexpected_batch_records
1200,customers,1200,20.0,1000,16.67,200,0,commercelake.bronze.raw_customers,0
12024,order_items,12024,19.58,10055,16.38,1969,0,commercelake.bronze.raw_order_items,0
6000,orders,6000,20.0,5000,16.67,1000,0,commercelake.bronze.raw_orders,0
6000,payments,6000,20.0,5000,16.67,1000,0,commercelake.bronze.raw_payments,0
250,products,250,25.0,200,20.0,50,0,commercelake.bronze.raw_products,0


check_name,entity,status
Expected historical record count,customers,PASS
Expected incremental record count,customers,PASS
Expected cumulative record count,customers,PASS
Historical and incremental records reconcile,customers,PASS
Exactly two expected batches present,customers,PASS
Zero unexpected batch records,customers,PASS
Cumulative records match idempotency baseline,customers,PASS
Expected historical record count,products,PASS
Expected incremental record count,products,PASS
Expected cumulative record count,products,PASS



CUMULATIVE BRONZE RECORD VALIDATION SUMMARY
Bronze Tables Validated: 5
Total Batches: 2
Historical Records: 21,255
Incremental Records: 4,219
Expected Cumulative Records: 25,474
Actual Cumulative Records: 25,474
Record Difference: 0
Dataset Growth: 19.85%
Incremental Contribution: 16.56%
Unexpected Batch Records: 0
Expected Checks: 42
Executed Checks: 42
Passed Checks: 42
Failed Checks: 0
Validation Success Rate: 100.00%

Cumulative Bronze Record Validation: SUCCESS
All historical and incremental records reconciled.


In [0]:
# Cell 28: Cumulative Business Key and Source Coverage Validation

print("=" * 65)
print("CUMULATIVE BUSINESS KEY AND SOURCE COVERAGE VALIDATION")
print("=" * 65)

print("Execution Mode: READ ONLY")


# 1. Verify prerequisite validation

if CUMULATIVE_RECORD_STATUS != "SUCCESS":

    raise RuntimeError(
        "Cumulative record validation must succeed first."
    )


# 2. Initialize validation containers

cumulative_key_results = []

cumulative_key_validation = []

cumulative_source_coverage = []


def record_key_validation(
    entity: str,
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    cumulative_key_validation.append({
        "entity": entity,
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


# 3. Validate each Bronze entity

for entity, config in SOURCE_CONFIG.items():

    target_table = config["target_path"]

    primary_key = config["primary_key"]

    expected_total = EXPECTED_CUMULATIVE_COUNTS[entity]

    print("\n" + "-" * 65)
    print(f"VALIDATING ENTITY: {entity.upper()}")
    print("-" * 65)

    print(f"Target Table: {target_table}")
    print(f"Business Key: {primary_key}")

    bronze_df = spark.table(target_table)


    # Check 1: Business key column exists

    record_key_validation(
        entity,
        "Business key column exists",
        primary_key in bronze_df.columns
    )


    # Check 2: No null business keys

    null_keys = (
        bronze_df
        .filter(F.col(primary_key).isNull())
        .count()
    )

    record_key_validation(
        entity,
        "Zero null business keys",
        null_keys == 0
    )


    # Check 3: No blank business keys

    blank_keys = (
        bronze_df
        .filter(
            F.trim(
                F.col(primary_key).cast("string")
            ) == ""
        )
        .count()
    )

    record_key_validation(
        entity,
        "Zero blank business keys",
        blank_keys == 0
    )


    # Check 4: Expected unique business key count

    distinct_keys = (
        bronze_df
        .select(primary_key)
        .distinct()
        .count()
    )

    record_key_validation(
        entity,
        "Expected distinct business key count",
        distinct_keys == expected_total
    )


    # Check 5: Duplicate business key groups

    duplicate_key_groups = (
        bronze_df
        .groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    record_key_validation(
        entity,
        "Zero duplicate business key groups",
        duplicate_key_groups == 0
    )


    # Check 6: Cross-batch business key collisions

    historical_keys = (
        bronze_df
        .filter(F.col("_batch_id") == "batch_001")
        .select(primary_key)
        .distinct()
    )

    incremental_keys = (
        bronze_df
        .filter(F.col("_batch_id") == "batch_002")
        .select(primary_key)
        .distinct()
    )

    cross_batch_collisions = (
        historical_keys
        .join(
            incremental_keys,
            on=primary_key,
            how="inner"
        )
        .count()
    )

    record_key_validation(
        entity,
        "Zero cross-batch business key collisions",
        cross_batch_collisions == 0
    )


    # 4. Validate source file coverage

    expected_files = [
        file
        for file in rerun_discovered_files
        if file["entity"] == entity
    ]

    expected_source_paths = {
        file["file_path"]
        for file in expected_files
    }

    actual_source_paths = get_ingested_file_paths(
        target_table=target_table
    )


    # Check 7: Expected source file count

    record_key_validation(
        entity,
        "Expected distinct source file count",
        len(actual_source_paths) == 2
    )


    # Check 8: Exact source file coverage

    record_key_validation(
        entity,
        "Complete expected source file coverage",
        actual_source_paths == expected_source_paths
    )


    # Build expected file-level record distribution

    expected_file_distribution = {}

    for file in expected_files:

        batch_id = file["batch_id"]

        if batch_id == "batch_001":

            expected_count = (
                EXPECTED_HISTORICAL_COUNTS[entity]
            )

        elif batch_id == "batch_002":

            expected_count = (
                EXPECTED_INCREMENTAL_COUNTS[entity]
            )

        else:

            raise RuntimeError(
                f"Unexpected batch: {batch_id}"
            )

        expected_file_distribution[
            (file["file_path"], batch_id)
        ] = expected_count


    # Read actual file-level record distribution

    actual_file_distribution = {
        (
            row["_source_file"],
            row["_batch_id"]
        ): row["count"]

        for row in (
            bronze_df
            .groupBy("_source_file", "_batch_id")
            .count()
            .collect()
        )
    }


    # Check 9: Correct source-to-batch mapping

    record_key_validation(
        entity,
        "Correct source file and batch mapping",
        set(actual_file_distribution)
        == set(expected_file_distribution)
    )


    # Check 10: Exact file-level record counts

    record_key_validation(
        entity,
        "Exact source file record distribution",
        actual_file_distribution
        == expected_file_distribution
    )


    # Store file-level coverage metrics

    for (
        source_path,
        batch_id
    ), expected_count in expected_file_distribution.items():

        actual_count = actual_file_distribution.get(
            (source_path, batch_id),
            0
        )

        cumulative_source_coverage.append({
            "entity": entity,
            "batch_id": batch_id,
            "source_path": source_path,
            "target_table": target_table,
            "expected_records": expected_count,
            "actual_records": actual_count,
            "record_difference": (
                actual_count - expected_count
            ),
            "status": (
                "PASS"
                if actual_count == expected_count
                else "FAIL"
            )
        })


    # 5. Validate cumulative record hashes

    invalid_hashes = (
        bronze_df
        .filter(
            F.col("_record_hash").isNull()
            | ~F.col("_record_hash").rlike(
                "^[a-f0-9]{64}$"
            )
        )
        .count()
    )


    # Check 11: Valid SHA-256 hashes

    record_key_validation(
        entity,
        "All record hashes have valid SHA-256 format",
        invalid_hashes == 0
    )


    # Recalculate hashes from business columns

    business_columns = sorted([
        column
        for column in bronze_df.columns
        if column not in METADATA_COLUMNS
    ])

    recalculated_hash = F.sha2(
        F.to_json(
            F.struct(*[
                F.col(column)
                for column in business_columns
            ]),
            options={
                "ignoreNullFields": "false"
            }
        ),
        256
    )

    hash_mismatches = (
        bronze_df
        .filter(
            ~F.col("_record_hash").eqNullSafe(
                recalculated_hash
            )
        )
        .count()
    )


    # Check 12: Persisted record hash consistency

    record_key_validation(
        entity,
        "Zero persisted record hash mismatches",
        hash_mismatches == 0
    )


    # 6. Store entity-level results

    cumulative_key_results.append({
        "entity": entity,
        "target_table": target_table,
        "business_key": primary_key,
        "expected_records": expected_total,
        "distinct_business_keys": distinct_keys,
        "null_business_keys": null_keys,
        "blank_business_keys": blank_keys,
        "duplicate_key_groups": duplicate_key_groups,
        "cross_batch_collisions": cross_batch_collisions,
        "expected_source_files": len(expected_source_paths),
        "actual_source_files": len(actual_source_paths),
        "invalid_hashes": invalid_hashes,
        "hash_mismatches": hash_mismatches
    })


    print(f"Distinct Business Keys: {distinct_keys:,}")
    print(f"Null Business Keys: {null_keys}")
    print(f"Blank Business Keys: {blank_keys}")
    print(f"Duplicate Key Groups: {duplicate_key_groups}")
    print(f"Cross-Batch Collisions: {cross_batch_collisions}")
    print(f"Source Files: {len(actual_source_paths)}")
    print(f"Hash Mismatches: {hash_mismatches}")


# 7. Calculate consolidated metrics

total_distinct_business_keys = sum(
    result["distinct_business_keys"]
    for result in cumulative_key_results
)

total_null_business_keys = sum(
    result["null_business_keys"]
    for result in cumulative_key_results
)

total_blank_business_keys = sum(
    result["blank_business_keys"]
    for result in cumulative_key_results
)

total_duplicate_key_groups = sum(
    result["duplicate_key_groups"]
    for result in cumulative_key_results
)

total_cross_batch_collisions = sum(
    result["cross_batch_collisions"]
    for result in cumulative_key_results
)

total_covered_source_files = sum(
    result["actual_source_files"]
    for result in cumulative_key_results
)

total_invalid_hashes = sum(
    result["invalid_hashes"]
    for result in cumulative_key_results
)

total_cumulative_hash_mismatches = sum(
    result["hash_mismatches"]
    for result in cumulative_key_results
)

total_source_record_differences = sum(
    abs(result["record_difference"])
    for result in cumulative_source_coverage
)

total_source_covered_records = sum(
    result["actual_records"]
    for result in cumulative_source_coverage
)


# 8. Consolidated pipeline validation

print("\n" + "=" * 65)
print("CONSOLIDATED BUSINESS KEY VALIDATION")
print("=" * 65)


record_key_validation(
    "pipeline",
    "Complete entity validation coverage",
    len(cumulative_key_results) == 5
)

record_key_validation(
    "pipeline",
    "Expected total distinct business keys",
    total_distinct_business_keys == 25474
)

record_key_validation(
    "pipeline",
    "Zero null or blank business keys",
    total_null_business_keys == 0
    and total_blank_business_keys == 0
)

record_key_validation(
    "pipeline",
    "Zero duplicate business key groups",
    total_duplicate_key_groups == 0
)

record_key_validation(
    "pipeline",
    "Zero cross-batch business key collisions",
    total_cross_batch_collisions == 0
)

record_key_validation(
    "pipeline",
    "Complete source file coverage",
    total_covered_source_files == 10
    and len(cumulative_source_coverage) == 10
)

record_key_validation(
    "pipeline",
    "Complete source-level record reconciliation",
    total_source_covered_records == 25474
    and total_source_record_differences == 0
)

record_key_validation(
    "pipeline",
    "Zero invalid or inconsistent record hashes",
    total_invalid_hashes == 0
    and total_cumulative_hash_mismatches == 0
)


# 9. Display validation results

print("\nBUSINESS KEY VALIDATION RESULTS")

display(
    spark.createDataFrame(
        cumulative_key_results
    ).orderBy("entity")
)

print("\nSOURCE FILE COVERAGE RESULTS")

display(
    spark.createDataFrame(
        cumulative_source_coverage
    ).orderBy("batch_id", "entity")
)

print("\nVALIDATION CHECK RESULTS")

display(
    spark.createDataFrame(
        cumulative_key_validation
    )
)


# 10. Calculate validation metrics

expected_checks = len(SOURCE_CONFIG) * 12 + 8

total_checks = len(cumulative_key_validation)

passed_checks = sum(
    result["status"] == "PASS"
    for result in cumulative_key_validation
)

failed_checks = total_checks - passed_checks


# 11. Generate final validation summary

print("\n" + "=" * 65)
print("CUMULATIVE BUSINESS KEY VALIDATION SUMMARY")
print("=" * 65)

print(f"Bronze Tables Validated: {len(cumulative_key_results)}")
print(f"Distinct Business Keys: {total_distinct_business_keys:,}")
print(f"Null Business Keys: {total_null_business_keys}")
print(f"Blank Business Keys: {total_blank_business_keys}")
print(f"Duplicate Key Groups: {total_duplicate_key_groups}")
print(f"Cross-Batch Collisions: {total_cross_batch_collisions}")
print(f"Source Files Covered: {total_covered_source_files}")
print(f"Source Records Reconciled: {total_source_covered_records:,}")
print(f"Source Record Count Differences: {total_source_record_differences}")
print(f"Invalid Record Hashes: {total_invalid_hashes}")
print(f"Record Hash Mismatches: {total_cumulative_hash_mismatches}")

print(f"Expected Checks: {expected_checks}")
print(f"Executed Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")

print(
    f"Validation Success Rate: "
    f"{passed_checks / total_checks * 100:.2f}%"
)


if failed_checks > 0 or total_checks != expected_checks:

    CUMULATIVE_KEY_STATUS = "FAILED"

    raise RuntimeError(
        "Cumulative business key validation failed."
    )


CUMULATIVE_KEY_STATUS = "SUCCESS"

print("\nCumulative Business Key Validation: SUCCESS")
print("Business keys and source file coverage verified.")

CUMULATIVE BUSINESS KEY AND SOURCE COVERAGE VALIDATION
Execution Mode: READ ONLY

-----------------------------------------------------------------
VALIDATING ENTITY: CUSTOMERS
-----------------------------------------------------------------
Target Table: commercelake.bronze.raw_customers
Business Key: customer_id
[PASS] Business key column exists
[PASS] Zero null business keys
[PASS] Zero blank business keys
[PASS] Expected distinct business key count
[PASS] Zero duplicate business key groups
[PASS] Zero cross-batch business key collisions
[PASS] Expected distinct source file count
[PASS] Complete expected source file coverage
[PASS] Correct source file and batch mapping
[PASS] Exact source file record distribution
[PASS] All record hashes have valid SHA-256 format
[PASS] Zero persisted record hash mismatches
Distinct Business Keys: 1,200
Null Business Keys: 0
Blank Business Keys: 0
Duplicate Key Groups: 0
Cross-Batch Collisions: 0
Source Files: 2
Hash Mismatches: 0

----------------

actual_source_files,blank_business_keys,business_key,cross_batch_collisions,distinct_business_keys,duplicate_key_groups,entity,expected_records,expected_source_files,hash_mismatches,invalid_hashes,null_business_keys,target_table
2,0,customer_id,0,1200,0,customers,1200,2,0,0,0,commercelake.bronze.raw_customers
2,0,order_item_id,0,12024,0,order_items,12024,2,0,0,0,commercelake.bronze.raw_order_items
2,0,order_id,0,6000,0,orders,6000,2,0,0,0,commercelake.bronze.raw_orders
2,0,payment_id,0,6000,0,payments,6000,2,0,0,0,commercelake.bronze.raw_payments
2,0,product_id,0,250,0,products,250,2,0,0,0,commercelake.bronze.raw_products



SOURCE FILE COVERAGE RESULTS


actual_records,batch_id,entity,expected_records,record_difference,source_path,status,target_table
1000,batch_001,customers,1000,0,/Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv,PASS,commercelake.bronze.raw_customers
10055,batch_001,order_items,10055,0,/Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet,PASS,commercelake.bronze.raw_order_items
5000,batch_001,orders,5000,0,/Volumes/commercelake/landing/source_data/orders/batch_001/orders.json,PASS,commercelake.bronze.raw_orders
5000,batch_001,payments,5000,0,/Volumes/commercelake/landing/source_data/payments/batch_001/payments.json,PASS,commercelake.bronze.raw_payments
200,batch_001,products,200,0,/Volumes/commercelake/landing/source_data/products/batch_001/products.csv,PASS,commercelake.bronze.raw_products
200,batch_002,customers,200,0,/Volumes/commercelake/landing/source_data/customers/batch_002/customers.csv,PASS,commercelake.bronze.raw_customers
1969,batch_002,order_items,1969,0,/Volumes/commercelake/landing/source_data/order_items/batch_002/order_items.parquet,PASS,commercelake.bronze.raw_order_items
1000,batch_002,orders,1000,0,/Volumes/commercelake/landing/source_data/orders/batch_002/orders.json,PASS,commercelake.bronze.raw_orders
1000,batch_002,payments,1000,0,/Volumes/commercelake/landing/source_data/payments/batch_002/payments.json,PASS,commercelake.bronze.raw_payments
50,batch_002,products,50,0,/Volumes/commercelake/landing/source_data/products/batch_002/products.csv,PASS,commercelake.bronze.raw_products



VALIDATION CHECK RESULTS


check_name,entity,status
Business key column exists,customers,PASS
Zero null business keys,customers,PASS
Zero blank business keys,customers,PASS
Expected distinct business key count,customers,PASS
Zero duplicate business key groups,customers,PASS
Zero cross-batch business key collisions,customers,PASS
Expected distinct source file count,customers,PASS
Complete expected source file coverage,customers,PASS
Correct source file and batch mapping,customers,PASS
Exact source file record distribution,customers,PASS



CUMULATIVE BUSINESS KEY VALIDATION SUMMARY
Bronze Tables Validated: 5
Distinct Business Keys: 25,474
Null Business Keys: 0
Blank Business Keys: 0
Duplicate Key Groups: 0
Cross-Batch Collisions: 0
Source Files Covered: 10
Source Records Reconciled: 25,474
Source Record Count Differences: 0
Invalid Record Hashes: 0
Record Hash Mismatches: 0
Expected Checks: 68
Executed Checks: 68
Passed Checks: 68
Failed Checks: 0
Validation Success Rate: 100.00%

Cumulative Business Key Validation: SUCCESS
Business keys and source file coverage verified.


In [0]:
# Cell 29: Consolidated Bronze Ingestion Report

print("=" * 70)
print("COMMERCELAKE - CONSOLIDATED BRONZE INGESTION REPORT")
print("=" * 70)

print("Execution Mode: READ ONLY")


# 1. Verify prerequisite execution statuses

required_statuses = {
    "Incremental Ingestion": (
        incremental_pipeline_summary["pipeline_status"],
        "SUCCESS"
    ),
    "Repeated Execution": (
        RERUN_PIPELINE_STATUS,
        "NO_NEW_FILES"
    ),
    "Idempotency Verification": (
        IDEMPOTENCY_STATUS,
        "SUCCESS"
    ),
    "Cumulative Record Validation": (
        CUMULATIVE_RECORD_STATUS,
        "SUCCESS"
    ),
    "Cumulative Business Key Validation": (
        CUMULATIVE_KEY_STATUS,
        "SUCCESS"
    )
}

for stage, (actual, expected) in required_statuses.items():

    if actual != expected:

        raise RuntimeError(
            f"Prerequisite failed: {stage}. "
            f"Expected {expected}, received {actual}"
        )

    print(f"[PASS] {stage}: {actual}")


# 2. Generate consolidated validation stage report

validation_stages = [
    (
        "Source File Discovery",
        discovery_validation_results,
        12
    ),
    (
        "Initial Ingestion Plan",
        plan_validation_results,
        12
    ),
    (
        "Pre-Ingestion Testing",
        pre_ingestion_results,
        71
    ),
    (
        "Incremental Execution Validation",
        file_execution_validation,
        58
    ),
    (
        "Delta Transaction Validation",
        delta_history_validation,
        43
    ),
    (
        "Metadata and Record Integrity",
        metadata_validation_results,
        76
    ),
    (
        "Repeated File Discovery",
        rerun_registry_validation,
        12
    ),
    (
        "Repeated Execution Planning",
        rerun_plan_validation,
        12
    ),
    (
        "No-Op Execution",
        rerun_execution_validation,
        15
    ),
    (
        "Runtime Duplicate Guard",
        guard_validation_results,
        66
    ),
    (
        "Final Idempotency Verification",
        idempotency_validation_results,
        68
    ),
    (
        "Cumulative Record Validation",
        cumulative_record_validation,
        42
    ),
    (
        "Business Key and Source Coverage",
        cumulative_key_validation,
        68
    )
]


consolidated_validation_report = []


for stage_name, results, expected_checks in validation_stages:

    executed_checks = len(results)

    passed_checks = sum(
        result["status"] == "PASS"
        for result in results
    )

    failed_checks = executed_checks - passed_checks

    stage_status = (
        "SUCCESS"
        if executed_checks == expected_checks
        and failed_checks == 0
        else "FAILED"
    )

    consolidated_validation_report.append({
        "validation_stage": stage_name,
        "expected_checks": expected_checks,
        "executed_checks": executed_checks,
        "passed_checks": passed_checks,
        "failed_checks": failed_checks,
        "success_rate": round(
            passed_checks / executed_checks * 100,
            2
        ) if executed_checks else 0.0,
        "status": stage_status
    })


# 3. Calculate consolidated validation metrics

total_expected_validation_checks = sum(
    result["expected_checks"]
    for result in consolidated_validation_report
)

total_executed_validation_checks = sum(
    result["executed_checks"]
    for result in consolidated_validation_report
)

total_passed_validation_checks = sum(
    result["passed_checks"]
    for result in consolidated_validation_report
)

total_failed_validation_checks = sum(
    result["failed_checks"]
    for result in consolidated_validation_report
)

overall_validation_success_rate = (
    total_passed_validation_checks
    / total_executed_validation_checks * 100
)

successful_validation_stages = sum(
    result["status"] == "SUCCESS"
    for result in consolidated_validation_report
)


print("\n" + "=" * 70)
print("CONSOLIDATED VALIDATION STAGE REPORT")
print("=" * 70)

display(
    spark.createDataFrame(
        consolidated_validation_report
    )
)


# 4. Generate entity-level Bronze report

bronze_entity_report = []


for entity, config in SOURCE_CONFIG.items():

    record_metrics = next(
        result
        for result in cumulative_record_results
        if result["entity"] == entity
    )

    key_metrics = next(
        result
        for result in cumulative_key_results
        if result["entity"] == entity
    )

    execution_metrics = next(
        result
        for result in incremental_file_results
        if result["entity"] == entity
    )

    table_state = capture_bronze_state(
        target_table=config["target_path"]
    )

    if table_state != idempotency_final_state[entity]:

        raise RuntimeError(
            f"Bronze state changed after verification: {entity}"
        )

    bronze_entity_report.append({
        "entity": entity,
        "target_table": config["target_path"],
        "historical_records": (
            record_metrics["historical_records"]
        ),
        "incremental_records": (
            record_metrics["incremental_records"]
        ),
        "total_records": table_state["total_records"],
        "growth_percentage": (
            record_metrics["growth_percentage"]
        ),
        "distinct_business_keys": (
            key_metrics["distinct_business_keys"]
        ),
        "duplicate_key_groups": (
            key_metrics["duplicate_key_groups"]
        ),
        "cross_batch_collisions": (
            key_metrics["cross_batch_collisions"]
        ),
        "hash_mismatches": (
            key_metrics["hash_mismatches"]
        ),
        "source_files": (
            table_state["distinct_source_files"]
        ),
        "delta_version": table_state["delta_version"],
        "delta_data_files": table_state["num_files"],
        "delta_storage_bytes": table_state["size_bytes"],
        "incremental_duration_seconds": (
            execution_metrics["duration_seconds"]
        ),
        "status": "SUCCESS"
    })


print("\n" + "=" * 70)
print("BRONZE ENTITY-LEVEL REPORT")
print("=" * 70)

display(
    spark.createDataFrame(
        bronze_entity_report
    ).orderBy("entity")
)


# 5. Calculate ingestion performance metrics

incremental_execution_duration = float(
    incremental_pipeline_summary["duration_seconds"]
)

incremental_records_per_second = (
    cumulative_incremental_records
    / incremental_execution_duration
    if incremental_execution_duration > 0
    else 0
)

noop_execution_duration = float(
    rerun_pipeline_summary["duration_seconds"]
)

total_landing_source_bytes = sum(
    file["file_size_bytes"]
    for file in rerun_discovered_files
)

total_bronze_data_bytes = sum(
    result["delta_storage_bytes"]
    for result in bronze_entity_report
)

storage_reduction_percentage = (
    (
        total_landing_source_bytes
        - total_bronze_data_bytes
    )
    / total_landing_source_bytes * 100
)

source_to_delta_size_ratio = (
    total_landing_source_bytes
    / total_bronze_data_bytes
)


# 6. Build consolidated Bronze ingestion report

BRONZE_REPORT_TIMESTAMP = datetime.now(timezone.utc)

bronze_ingestion_report = {
    "project_name": "CommerceLake",
    "pipeline_name": PIPELINE_NAME,
    "report_timestamp": (
        BRONZE_REPORT_TIMESTAMP.isoformat()
    ),
    "catalog": CATALOG,
    "schema": BRONZE_SCHEMA,
    "incremental_run_id": PIPELINE_RUN_ID,
    "noop_run_id": RERUN_PIPELINE_RUN_ID,
    "guard_test_run_id": GUARD_TEST_RUN_ID,
    "bronze_tables": len(bronze_entity_report),
    "source_batches": 2,
    "source_files": total_covered_source_files,
    "historical_records": cumulative_historical_records,
    "incremental_records": cumulative_incremental_records,
    "total_bronze_records": cumulative_total_records,
    "dataset_growth_percentage": round(
        cumulative_growth_percentage,
        2
    ),
    "incremental_execution_seconds": (
        incremental_execution_duration
    ),
    "incremental_records_per_second": round(
        incremental_records_per_second,
        2
    ),
    "noop_execution_seconds": noop_execution_duration,
    "planning_files_skipped": len(
        rerun_skipped_source_files
    ),
    "runtime_files_blocked": guard_test_skipped_files,
    "runtime_guard_test_seconds": guard_test_duration,
    "duplicate_records_inserted": (
        final_total_rerun_records
        + final_total_guard_records
    ),
    "new_transactions_during_rerun_tests": (
        final_total_new_transactions
    ),
    "persisted_record_differences": (
        final_total_record_differences
    ),
    "distinct_business_keys": (
        total_distinct_business_keys
    ),
    "duplicate_key_groups": total_duplicate_key_groups,
    "cross_batch_collisions": total_cross_batch_collisions,
    "record_hash_mismatches": (
        total_cumulative_hash_mismatches
    ),
    "landing_source_bytes": total_landing_source_bytes,
    "bronze_delta_data_bytes": total_bronze_data_bytes,
    "delta_data_files": final_total_delta_files,
    "storage_reduction_percentage": round(
        storage_reduction_percentage,
        2
    ),
    "source_to_delta_size_ratio": round(
        source_to_delta_size_ratio,
        2
    ),
    "validation_stages": len(
        consolidated_validation_report
    ),
    "successful_validation_stages": (
        successful_validation_stages
    ),
    "expected_validation_checks": (
        total_expected_validation_checks
    ),
    "executed_validation_checks": (
        total_executed_validation_checks
    ),
    "passed_validation_checks": (
        total_passed_validation_checks
    ),
    "failed_validation_checks": (
        total_failed_validation_checks
    ),
    "validation_success_rate": round(
        overall_validation_success_rate,
        2
    )
}


# 7. Verify consolidated report consistency

report_validation_results = []


def record_report_validation(
    check_name: str,
    passed: bool
):

    status = "PASS" if passed else "FAIL"

    report_validation_results.append({
        "check_name": check_name,
        "status": status
    })

    print(f"[{status}] {check_name}")


print("\n" + "=" * 70)
print("CONSOLIDATED REPORT VALIDATION")
print("=" * 70)


record_report_validation(
    "All 13 validation stages successful",
    successful_validation_stages
    == len(validation_stages)
    == 13
)

record_report_validation(
    "Expected validation check count",
    total_expected_validation_checks == 555
)

record_report_validation(
    "All expected validation checks executed",
    total_executed_validation_checks
    == total_expected_validation_checks
)

record_report_validation(
    "Zero failed validation checks",
    total_failed_validation_checks == 0
)

record_report_validation(
    "Complete Bronze entity coverage",
    len(bronze_entity_report) == 5
)

record_report_validation(
    "Total Bronze records reconciled",
    sum(
        result["total_records"]
        for result in bronze_entity_report
    ) == cumulative_total_records == 25474
)

record_report_validation(
    "Incremental execution records reconciled",
    cumulative_incremental_records
    == incremental_pipeline_summary[
        "total_inserted_records"
    ]
    == 4219
)

record_report_validation(
    "Complete source file coverage",
    total_covered_source_files == 10
)

record_report_validation(
    "Zero duplicate records from repeated execution",
    final_total_rerun_records == 0
    and final_total_guard_records == 0
)

record_report_validation(
    "Zero unexpected Delta transactions",
    final_total_new_transactions == 0
)

record_report_validation(
    "Zero persisted record differences",
    final_total_record_differences == 0
)

record_report_validation(
    "Storage metrics reconciled",
    total_bronze_data_bytes
    == final_total_storage_bytes
    == 1607364
)

record_report_validation(
    "All Bronze Delta versions unchanged",
    all(
        result["delta_version"] == 2
        for result in bronze_entity_report
    )
)

record_report_validation(
    "Business keys and record hashes verified",
    total_distinct_business_keys == 25474
    and total_duplicate_key_groups == 0
    and total_cross_batch_collisions == 0
    and total_cumulative_hash_mismatches == 0
)


# 8. Determine consolidated report status

report_checks = len(report_validation_results)

report_passed_checks = sum(
    result["status"] == "PASS"
    for result in report_validation_results
)

report_failed_checks = (
    report_checks - report_passed_checks
)

BRONZE_REPORT_STATUS = (
    "SUCCESS"
    if report_checks == 14
    and report_failed_checks == 0
    else "FAILED"
)

bronze_ingestion_report["report_status"] = (
    BRONZE_REPORT_STATUS
)

bronze_ingestion_report["report_validation_checks"] = (
    report_checks
)

bronze_ingestion_report["report_validation_passed"] = (
    report_passed_checks
)


display(
    spark.createDataFrame(
        report_validation_results
    )
)

display(
    spark.createDataFrame([
        bronze_ingestion_report
    ])
)


# 9. Generate human-readable final report

print("\n" + "=" * 70)
print("COMMERCELAKE - FINAL BRONZE INGESTION SUMMARY")
print("=" * 70)

print(f"Project: CommerceLake")
print(f"Catalog: {CATALOG}")
print(f"Schema: {BRONZE_SCHEMA}")

print(
    f"Report Timestamp: "
    f"{BRONZE_REPORT_TIMESTAMP.isoformat()}"
)

print("\nDATA INGESTION METRICS")
print("-" * 70)

print(f"Bronze Tables: {len(bronze_entity_report)}")
print("Source Batches: 2")
print(f"Source Files: {total_covered_source_files}")
print(f"Historical Records: {cumulative_historical_records:,}")
print(f"Incremental Records: {cumulative_incremental_records:,}")
print(f"Total Bronze Records: {cumulative_total_records:,}")
print(f"Dataset Growth: {cumulative_growth_percentage:.2f}%")

print("\nPERFORMANCE METRICS")
print("-" * 70)

print(
    f"Incremental Execution Duration: "
    f"{incremental_execution_duration:.4f} seconds"
)

print(
    f"Incremental Throughput: "
    f"{incremental_records_per_second:.2f} records/second"
)

print(
    f"No-Op Execution Duration: "
    f"{noop_execution_duration:.4f} seconds"
)

print(
    f"Runtime Guard Test Duration: "
    f"{guard_test_duration:.4f} seconds"
)

print("\nIDEMPOTENCY METRICS")
print("-" * 70)

print(
    f"Files Skipped During Planning: "
    f"{len(rerun_skipped_source_files)}"
)

print(f"Files Blocked at Runtime: {guard_test_skipped_files}")
print("Duplicate Records Inserted: 0")
print(f"New Delta Transactions: {final_total_new_transactions}")
print(f"Persisted Record Differences: {final_total_record_differences}")

print("\nSTORAGE METRICS")
print("-" * 70)

print(f"Landing Source Bytes: {total_landing_source_bytes:,}")
print(f"Bronze Delta Data Bytes: {total_bronze_data_bytes:,}")
print(f"Delta Data Files: {final_total_delta_files}")
print(f"Storage Reduction: {storage_reduction_percentage:.2f}%")
print(f"Source-to-Delta Size Ratio: {source_to_delta_size_ratio:.2f}:1")

print("\nDATA INTEGRITY METRICS")
print("-" * 70)

print(f"Distinct Business Keys: {total_distinct_business_keys:,}")
print(f"Duplicate Key Groups: {total_duplicate_key_groups}")
print(f"Cross-Batch Collisions: {total_cross_batch_collisions}")
print(f"Record Hash Mismatches: {total_cumulative_hash_mismatches}")

print("\nVALIDATION METRICS")
print("-" * 70)

print(f"Validation Stages: {len(validation_stages)}")
print(f"Successful Validation Stages: {successful_validation_stages}")
print(f"Expected Validation Checks: {total_expected_validation_checks}")
print(f"Executed Validation Checks: {total_executed_validation_checks}")
print(f"Passed Validation Checks: {total_passed_validation_checks}")
print(f"Failed Validation Checks: {total_failed_validation_checks}")
print(f"Validation Success Rate: {overall_validation_success_rate:.2f}%")

print(f"Report Validation Checks: {report_checks}")
print(f"Report Validation Passed: {report_passed_checks}")
print(f"Report Validation Failed: {report_failed_checks}")

print(
    f"Total Checks Including Report Validation: "
    f"{total_executed_validation_checks + report_checks}"
)

print("\n" + "=" * 70)

print(f"CONSOLIDATED REPORT STATUS: {BRONZE_REPORT_STATUS}")

print("=" * 70)


if BRONZE_REPORT_STATUS != "SUCCESS":

    raise RuntimeError(
        "Consolidated Bronze ingestion report validation failed."
    )


print("Bronze incremental ingestion successfully verified.")
print("Cumulative Bronze validation completed.")
print("Notebook is ready for export and version control.")

COMMERCELAKE - CONSOLIDATED BRONZE INGESTION REPORT
Execution Mode: READ ONLY
[PASS] Incremental Ingestion: SUCCESS
[PASS] Repeated Execution: NO_NEW_FILES
[PASS] Idempotency Verification: SUCCESS
[PASS] Cumulative Record Validation: SUCCESS
[PASS] Cumulative Business Key Validation: SUCCESS

CONSOLIDATED VALIDATION STAGE REPORT


executed_checks,expected_checks,failed_checks,passed_checks,status,success_rate,validation_stage
12,12,0,12,SUCCESS,100.0,Source File Discovery
12,12,0,12,SUCCESS,100.0,Initial Ingestion Plan
71,71,0,71,SUCCESS,100.0,Pre-Ingestion Testing
58,58,0,58,SUCCESS,100.0,Incremental Execution Validation
43,43,0,43,SUCCESS,100.0,Delta Transaction Validation
76,76,0,76,SUCCESS,100.0,Metadata and Record Integrity
12,12,0,12,SUCCESS,100.0,Repeated File Discovery
12,12,0,12,SUCCESS,100.0,Repeated Execution Planning
15,15,0,15,SUCCESS,100.0,No-Op Execution
66,66,0,66,SUCCESS,100.0,Runtime Duplicate Guard



BRONZE ENTITY-LEVEL REPORT


cross_batch_collisions,delta_data_files,delta_storage_bytes,delta_version,distinct_business_keys,duplicate_key_groups,entity,growth_percentage,hash_mismatches,historical_records,incremental_duration_seconds,incremental_records,source_files,status,target_table,total_records
0,2,101167,2,1200,0,customers,20.0,0,1000,8.1908,200,2,SUCCESS,commercelake.bronze.raw_customers,1200
0,2,696716,2,12024,0,order_items,19.58,0,10055,6.0477,1969,2,SUCCESS,commercelake.bronze.raw_order_items,12024
0,2,366231,2,6000,0,orders,20.0,0,5000,5.7418,1000,2,SUCCESS,commercelake.bronze.raw_orders,6000
0,2,414890,2,6000,0,payments,20.0,0,5000,5.8139,1000,2,SUCCESS,commercelake.bronze.raw_payments,6000
0,2,28360,2,250,0,products,25.0,0,200,6.5008,50,2,SUCCESS,commercelake.bronze.raw_products,250



CONSOLIDATED REPORT VALIDATION
[PASS] All 13 validation stages successful
[PASS] Expected validation check count
[PASS] All expected validation checks executed
[PASS] Zero failed validation checks
[PASS] Complete Bronze entity coverage
[PASS] Total Bronze records reconciled
[PASS] Incremental execution records reconciled
[PASS] Complete source file coverage
[PASS] Zero duplicate records from repeated execution
[PASS] Zero unexpected Delta transactions
[PASS] Zero persisted record differences
[PASS] Storage metrics reconciled
[PASS] All Bronze Delta versions unchanged
[PASS] Business keys and record hashes verified


check_name,status
All 13 validation stages successful,PASS
Expected validation check count,PASS
All expected validation checks executed,PASS
Zero failed validation checks,PASS
Complete Bronze entity coverage,PASS
Total Bronze records reconciled,PASS
Incremental execution records reconciled,PASS
Complete source file coverage,PASS
Zero duplicate records from repeated execution,PASS
Zero unexpected Delta transactions,PASS


bronze_delta_data_bytes,bronze_tables,catalog,cross_batch_collisions,dataset_growth_percentage,delta_data_files,distinct_business_keys,duplicate_key_groups,duplicate_records_inserted,executed_validation_checks,expected_validation_checks,failed_validation_checks,guard_test_run_id,historical_records,incremental_execution_seconds,incremental_records,incremental_records_per_second,incremental_run_id,landing_source_bytes,new_transactions_during_rerun_tests,noop_execution_seconds,noop_run_id,passed_validation_checks,persisted_record_differences,pipeline_name,planning_files_skipped,project_name,record_hash_mismatches,report_status,report_timestamp,report_validation_checks,report_validation_passed,runtime_files_blocked,runtime_guard_test_seconds,schema,source_batches,source_files,source_to_delta_size_ratio,storage_reduction_percentage,successful_validation_stages,total_bronze_records,validation_stages,validation_success_rate
1607364,5,commercelake,0,19.85,10,25474,0,0,555,555,0,76734e8a-afe7-4270-8e8c-6de62ea26d84,21255,32.2961,4219,130.63,5cd36830-4695-40d1-ab22-17d8db1df9f9,5198482,0,1.0E-4,9d378a30-1e55-4542-8874-1bea1f834303,555,0,commercelake_bronze_incremental_ingestion,10,CommerceLake,0,SUCCESS,2026-11-26T14:02:59.949600+00:00,14,14,10,6.7867,bronze,2,10,3.23,69.08,13,25474,13,100.0



COMMERCELAKE - FINAL BRONZE INGESTION SUMMARY
Project: CommerceLake
Catalog: commercelake
Schema: bronze
Report Timestamp: 2026-11-26T14:02:59.949600+00:00

DATA INGESTION METRICS
----------------------------------------------------------------------
Bronze Tables: 5
Source Batches: 2
Source Files: 10
Historical Records: 21,255
Incremental Records: 4,219
Total Bronze Records: 25,474
Dataset Growth: 19.85%

PERFORMANCE METRICS
----------------------------------------------------------------------
Incremental Execution Duration: 32.2961 seconds
Incremental Throughput: 130.63 records/second
No-Op Execution Duration: 0.0001 seconds
Runtime Guard Test Duration: 6.7867 seconds

IDEMPOTENCY METRICS
----------------------------------------------------------------------
Files Skipped During Planning: 10
Files Blocked at Runtime: 10
Duplicate Records Inserted: 0
New Delta Transactions: 0
Persisted Record Differences: 0

STORAGE METRICS
-----------------------------------------------------------